# Wikipedia Thanks Recipient Analysis - PH revision

[J. Nathan Matias](https://natematias.com), June 2024

Key documents:
* [pre-analysis plan: Receiving Thanks on Wikipedia](https://osf.io/c67rg/)
* [data format description](https://docs.google.com/document/d/1plhoDbQryYQ32vZMXu8YmlLSp30QTdup43k6uTePOT4/edit#heading=h.fxaguwxn13cj)
* Tresorit data in `Tresors/CivilServant/projects/wikipedia-integration/gratitude-study`

Decisions:
* With only 786 survey compliers, we are way under the sample needed to test our survey questions. So we will only report summary statistics for survey questions.
* Since we aren't testing survey-based hypotheses, we will reduce the number of comparisons that we adjust for.
* In this code, I don't adjust for language and sub-group in the meta-analysis. That's unnecessary because difference_in_means accounts for the randomization strategy. I misunderstood that at the time I pre-registered.

# Load Data

In [ ]:
library(plyr)
library(AER)
library(tidyverse)
library(magrittr)
library(gmodels)
library(MASS)
library(estimatr)
library(gridExtra)
library(dplyr)
library(ri2)
library(ggplot2)
library(lme4)
library(lmerTest)
library(ggpubr)   # contains ggarrange
library(png)      # to load logo
library(grid)
library(knitr)
library("glmmTMB")
library(performance)
library(DHARMa)
library(ggridges)
library(lubridate)

### TODO: Add Source Sans as a font for ggplot2
library(sysfonts) # to load source sans pro
# https://rdrr.io/github/kjhealy/sourcesans/src/R/sourcesans.r

## Set visual style
catpalette   <- c("#333333", "#E69F00", "#56B4E9", "#7D868C", "#BDBBBB", "#F2F2F2","#F6F2EB")
chartpalette <- c("#E69F00", "#56B4E9", "#7D868C", "#333333", "#F2F2F2","#BDBBBB", "#F6F2EB")

cat.theme <-  theme_bw() +
              theme(plot.title = element_text(size=13, face="bold", color=catpalette[3]),
                    axis.title.x =element_text(size=10, hjust = -0.01, color = catpalette[1]),
                    axis.title.y =element_text(size=10, color = catpalette[1]),
                    panel.background = element_rect(fill=catpalette[6]))

In [ ]:
data.path = Sys.getenv('TRESORDIR', '~/Library/CloudStorage/Tresorit-PsycheHe')
tresor.path = 'wiki-gratitude-study/gratitude-study-copy-2025-10-07/Data Drills/thankee/post_experiment_analysis'
fname = 'grat-thankee-all-pre-post-treatment-vars.csv'
fname2 = 'grat-thankee-all-pre-post-treatment-vars-max-cols.csv'
thank.fname = '2021-04-30-secondary-thanks.csv'
all.thanks.fname <- 'gratitude-second-gen-thanks-analysis-with-reciprocal.csv'
f.path = file.path(data.path, tresor.path, fname)
f.path2 = file.path(data.path, tresor.path, fname2)
all.participants <- read.csv(f.path)
all.participants.max <- read.csv(f.path2)
all.thanks <- read.csv(file.path(data.path, tresor.path, all.thanks.fname))
participant.thanks <- read.csv(file.path(data.path, tresor.path, thank.fname))

## Merge two datasets by ID

In [ ]:
pre.post.labor.hour <- all.participants.max[,c("labor.hours.pre.treatment", "labor.hours.post.treatment", "labor.hours.per.day.pre.treatment", "labor.hours.per.day.post.treatment", "user.id.md5")]
all.participants <- merge(all.participants, pre.post.labor.hour, by = "user.id.md5")

In [ ]:
summary(all.participants$labor.hours.per.day.post.treatment)
summary(all.participants$labor.hours.per.day.pre.treatment)

In [ ]:
start.date = "2019-08-02"
end.date   = "2020-02-11"

end.caption.text = "Details at: citizensandtech.org/research/how-do-wikipedians-thank-each-other/


Citizens & Technology Lab - citizensandtech.org
© Creative Commons International Attribution 4.0"

lab.anewc <- "all newcomers"
lab.newc <- "newcomers"
lab.exp <- "experienced"

In [ ]:
### Load the logo for the blog post
# cat.logo.filename <- "../../assets/CAT-Logo-Horizontal-social-media-preview-color.png"
# logo.img <- readPNG(cat.logo.filename)
# logo.pngob <- rasterGrob(logo.img)

## Print the number of partiicpants in the experiment, before
## any randomization blocks are removed for analysis
nrow(all.participants)

### Convert columns to column names and formats from the experiment plan

In [ ]:
all.participants$two.week.retention  %<>% as.logical
all.participants$complier  %<>% as.logical
all.participants$has.email  %<>% as.logical
all.participants$thanks.not.received.skipped %<>% as.logical
all.participants$thanks.not.received.not.seen  %<>% as.logical
all.participants$thanks.not.received.error  %<>% as.logical
all.participants$thanks.not.received.user.deleted  %<>% as.logical
all.participants$received.multiple.thanks  %<>% as.logical
all.participants$complier.app.any.reason  %<>% as.logical

all.participants$TREAT <- all.participants$randomization.arm %>% as.factor
all.participants$block <- all.participants$randomization.block.id %>% as.factor
all.participants$prev.experience <- all.participants$prev.experience.assignment
all.participants$newcomer <- all.participants$prev.experience=="bin_0"

all.participants$prev.experience.assignment.days <-
    as.numeric(str_replace(all.participants$prev.experience, "bin_", ""))

In [ ]:
summary(all.participants$TREAT)

### Remove blocks that include accounts that were deleted

In [ ]:
blocks.to.omit.deleted <- subset(all.participants, thanks.not.received.user.deleted)$block

#### Remove Blocks that were Multiply Thanked

In [ ]:
length(unique(subset(all.participants,received.multiple.thanks)$block))
blocks.to.omit.multiple.thanks <- unique(subset(all.participants,received.multiple.thanks)$block)

#### Remove blocks that were deleted at the time

In [ ]:
length(unique(c(blocks.to.omit.multiple.thanks,blocks.to.omit.deleted)))

#### Count omitted participants

In [ ]:
nrow(subset(all.participants, block %in% blocks.to.omit.deleted | block %in% blocks.to.omit.multiple.thanks))
nrow(all.participants)

In [ ]:
participants <- subset(all.participants,
                       (block %in% blocks.to.omit.deleted)!=TRUE &
                       (block %in% blocks.to.omit.multiple.thanks)!=TRUE
                      )
nrow(participants)

### Create counts of accounts by language and newcomer
These counts are used in the table of participants.

In [ ]:
# Center labor hours pre treatment
participants$centered.labor.hours.pre.treatment <- scale(participants$labor.hours.pre.treatment, center = TRUE, scale = FALSE)
participants$centered.labor.hours.per.day.pre.treatment <- scale(participants$labor.hours.per.day.pre.treatment, center = TRUE, scale = FALSE)

summary(participants$centered.labor.hours.pre.treatment)
summary(participants$centered.labor.hours.per.day.pre.treatment)

In [ ]:
aggregate(all.participants[c("TREAT")], by=list(newcomer=all.participants$newcomer, lang=all.participants$lang), FUN=length)

In [ ]:
print(paste("Polish newcomer % thanked", prettyNum((nrow(subset(all.participants, newcomer=="TRUE" & lang=="pl" & number.thanks.received>0 & TREAT==1)) / nrow(subset(all.participants, newcomer=="TRUE" & lang=="pl" & TREAT==1)))*100, digits=2),"%"))
print(paste("German newcomer % thanked", prettyNum((nrow(subset(all.participants, newcomer=="TRUE" & lang=="de" & number.thanks.received>0 & TREAT==1)) / nrow(subset(all.participants, newcomer=="TRUE" & lang=="de" & TREAT==1)))*100, digits=2),"%"))
print(paste("Arabic newcomer % thanked", prettyNum((nrow(subset(all.participants, newcomer=="TRUE" & lang=="ar" & number.thanks.received>0 & TREAT==1)) / nrow(subset(all.participants, newcomer=="TRUE" & lang=="ar" & TREAT==1)))*100, digits=2),"%"))

print(paste("Newcomer % thanked", prettyNum((nrow(subset(all.participants, newcomer=="TRUE" & number.thanks.received>0 & TREAT==1)) / nrow(subset(all.participants, newcomer=="TRUE" & TREAT==1)))*100, digits=2),"%"))


print(paste("Polish experienced % thanked", prettyNum((nrow(subset(all.participants, newcomer=="FALSE" & lang=="pl" & number.thanks.received>0 & TREAT==1)) / nrow(subset(all.participants, newcomer=="FALSE" & lang=="pl" & TREAT==1)))*100, digits=2),"%"))
print(paste("Persian experienced % thanked", prettyNum((nrow(subset(all.participants, newcomer=="FALSE" & lang=="fa" & number.thanks.received>0 & TREAT==1)) / nrow(subset(all.participants, newcomer=="FALSE" & lang=="fa" & TREAT==1)))*100, digits=2),"%"))
print(paste("Experienced % thanked", prettyNum((nrow(subset(all.participants, newcomer=="FALSE" & number.thanks.received>0 & TREAT==1)) / nrow(subset(all.participants, newcomer=="FALSE" & TREAT==1)))*100, digits=2),"%"))


In [ ]:
print(paste("Polish newcomer two week retention rate", prettyNum(nrow(subset(all.participants, newcomer=="TRUE" & lang=="pl" & TREAT==0 & two.week.retention))/
                                                                 nrow(subset(all.participants, newcomer=="TRUE" & lang=="pl" & TREAT==0))
                                                                )))

print(paste("German newcomer two week retention rate", prettyNum(nrow(subset(all.participants, newcomer=="TRUE" & lang=="de" & TREAT==0 & two.week.retention))/
                                                                 nrow(subset(all.participants, newcomer=="TRUE" & lang=="de" & TREAT==0))
                                                                )))

print(paste("Arabic newcomer two week retention rate", prettyNum(nrow(subset(all.participants, newcomer=="TRUE" & lang=="ar" & TREAT==0 & two.week.retention))/
                                                                 nrow(subset(all.participants, newcomer=="TRUE" & lang=="ar" & TREAT==0))
                                                                )))

print(paste("Polish experienced two week retention rate", prettyNum(nrow(subset(all.participants, newcomer=="FALSE" & lang=="pl" & TREAT==0 & two.week.retention))/
                                                                 nrow(subset(all.participants, newcomer=="FALSE" & lang=="pl" & TREAT==0))
                                                                )))

print(paste("Persian experienced two week retention rate", prettyNum(nrow(subset(all.participants, newcomer=="FALSE" & lang=="fa" & TREAT==0 & two.week.retention))/
                                                                 nrow(subset(all.participants, newcomer=="FALSE" & lang=="fa" & TREAT==0))
                                                                )))

In [ ]:
print(paste("Polish newcomer labor hours per day (control)", prettyNum(mean(subset(all.participants, newcomer=="TRUE" & lang=="pl" & TREAT==0)$labor.hours.per.day.post.treatment, na.rm=TRUE), digits=2)))

print(paste("German newcomer labor hours per day (control)", prettyNum(mean(subset(all.participants, newcomer=="TRUE" & lang=="de" & TREAT==0)$labor.hours.per.day.post.treatment, na.rm=TRUE), digits=2)))

print(paste("Arabic newcomer labor hours per day (control)", prettyNum(mean(subset(all.participants, newcomer=="TRUE" & lang=="ar" & TREAT==0)$labor.hours.per.day.post.treatment, na.rm=TRUE), digits=2)))

print(paste("Total newcomer labor hours per day (control)", prettyNum(mean(subset(all.participants, newcomer=="TRUE" & TREAT==0)$labor.hours.per.day.post.treatment, na.rm=TRUE), digits=2)))

print(paste("Polish experienced labor hours per day (control)", prettyNum(mean(subset(all.participants, newcomer=="FALSE" & lang=="pl" & TREAT==0)$labor.hours.per.day.post.treatment, na.rm=TRUE), digits=2)))

print(paste("Persian experienced labor hours per day (control)", prettyNum(mean(subset(all.participants, newcomer=="FALSE" & lang=="fa" & TREAT==0)$labor.hours.per.day.post.treatment, na.rm=TRUE), digits=2)))

print(paste("Total experienced labor hours per day (control)", prettyNum(mean(subset(all.participants, newcomer=="FALSE" & TREAT==0)$labor.hours.per.day.post.treatment, na.rm=TRUE), digits=2)))

In [ ]:
nrow(subset(all.participants, newcomer=="TRUE" & lang=="ar" & number.thanks.received>0)) / nrow(subset(all.participants, newcomer=="TRUE" & lang=="ar"))

# Baseline Descriptive Information
Included for anyone who wants to review further details in the study, particularly balance in various variables, but commented out since they are not reported in the paper.

## Study Participants

In [ ]:
# CrossTable(participants$lang, participants$TREAT,
#            prop.c = FALSE, prop.r=FALSE,
#            prop.t=FALSE, prop.chisq = FALSE)

## Previous Experience by Language


In [ ]:
# CrossTable(participants$prev.experience.assignment.days,
#            participants$lang,
#            prop.c = FALSE, prop.r=FALSE,
#            prop.t=FALSE, prop.chisq = FALSE)

In [ ]:
# CrossTable(participants$lang,
#            participants$newcomer,
#            prop.c = FALSE, prop.r=FALSE,
#            prop.t=FALSE, prop.chisq = FALSE)

## Treatment by Previous Experience

In [ ]:
# CrossTable(participants$prev.experience.assignment.days, participants$TREAT,
#            prop.c = FALSE, prop.r=FALSE,
#            prop.t=FALSE, prop.chisq = FALSE)

## Received Thanks by Experience and Language

In [ ]:
# CrossTable(subset(participants, TREAT==1)$newcomer,
#            subset(participants, TREAT==1)$complier.app.any.reason,
#            prop.t=FALSE, prop.chisq=FALSE, prop.c=FALSE)

In [ ]:
# CrossTable(paste(subset(participants, TREAT==1)$lang,
#                  subset(participants, TREAT==1)$newcomer),
#            subset(participants, TREAT==1)$complier.app.any.reason,
#            prop.t=FALSE, prop.chisq=FALSE, prop.c=FALSE)

## Survey Compliance

### Survey Compliance by Language

In [ ]:
sum(participants$complier)/nrow(participants)

In [ ]:
# CrossTable(participants$prev.experience.assignment.days, participants$complier,
#            prop.c = FALSE, prop.r=TRUE,
#            prop.t=FALSE, prop.chisq = FALSE)

# Distribution of Thanks

In [ ]:
# Step 1: Get earliest first.thank.dt per block from all.participants.max
# Exclude invalid thanks: skipped, not seen, error, user deleted, multiple thanks

# First, check which exclusion columns exist in all.participants.max
cat("Columns in all.participants.max:\n")
grep("thanks.not|multiple", colnames(all.participants.max), value = TRUE)

# Get valid participants (no errors/issues with thanks) using user.id.md5
# Convert to logical if needed, or compare as strings
valid.thanks.participants <- all.participants %>%
  filter(
    thanks.not.received.skipped != TRUE & thanks.not.received.skipped != "True" &
    thanks.not.received.not.seen != TRUE & thanks.not.received.not.seen != "True" &
    thanks.not.received.error != TRUE & thanks.not.received.error != "True" &
    thanks.not.received.user.deleted != TRUE & thanks.not.received.user.deleted != "True" &
    received.multiple.thanks != TRUE & received.multiple.thanks != "True"
                                                          # &
# blocks.to.omit.deleted != TRUE & blocks.to.omit.deleted != "True" &
# blocks.to.omit.multiple.thanks != TRUE & blocks.to.omit.multiple.thanks != "True"
  ) %>%
  dplyr::select(user.id.md5) %>%
  distinct()

cat("Valid thanks participants:", nrow(valid.thanks.participants), "\n")

# Filter all.participants.max to only valid participants before computing earliest
first.gen.received.by.block <- all.participants.max %>%
  filter(first.thank.dt != "") %>%
  filter(user.id.md5 %in% valid.thanks.participants$user.id.md5) %>%
  group_by(randomization.block.id) %>%
  summarise(
    earliest.first.gen.thank.in.block.ts = min(first.thank.dt, na.rm = TRUE),
    .groups = "drop"
  )

cat("Unique first-gen thank recipients after filtering:",
    all.participants.max %>%
      filter(first.thank.dt != "") %>%
      filter(user.id.md5 %in% valid.thanks.participants$user.id.md5) %>%
      nrow(), "\n")

# Step 2: Get block.id for each second-gen sender from all.participants.max
second.gen.sent.by.block <- all.thanks %>%
  inner_join(
    all.participants.max %>%
      dplyr::select(user.id.md5, randomization.block.id) %>%
      distinct(user.id.md5, .keep_all = TRUE),
    by = c("second.gen.sender.user.id.md5" = "user.id.md5")
  ) %>%
  group_by(randomization.block.id) %>%
  summarise(
    earliest.second.gen.thank.in.block.ts = min(second.gen.thank.ts, na.rm = TRUE),
    .groups = "drop"
  )

# Step 3: Get earliest behavior.start.dt per block
earliest.behavior.by.block <- all.participants.max %>%
  group_by(randomization.block.id) %>%
  summarise(
    earliest.behavior.start.in.block.dt = min(behavior.start.dt, na.rm = TRUE),
    .groups = "drop"
  )

# Drop columns if they already exist (for re-running)
all.participants.max <- all.participants.max %>%
  dplyr::select(-any_of(c("earliest.first.gen.thank.in.block.ts", "earliest.second.gen.thank.in.block.ts", "earliest.behavior.start.in.block.dt")))

# Step 4: Merge back to all.participants.max
all.participants.max <- all.participants.max %>%
  left_join(first.gen.received.by.block, by = "randomization.block.id") %>%
  left_join(second.gen.sent.by.block, by = "randomization.block.id") %>%
  left_join(earliest.behavior.by.block, by = "randomization.block.id")

In [ ]:
# Find participants with first-gen thanks who are excluded from formal analysis

# All participants with first.thank.dt (received first-gen thanks)
all.first.gen.recipients <- all.participants.max %>%
  filter(first.thank.dt != "") %>%
  dplyr::select(user.id.md5, user.id.md5, first.thank.dt, randomization.block.id, lang)

cat("Total participants with first.thank.dt:", nrow(all.first.gen.recipients), "\n")

# Get exclusion reasons from all.participants
exclusion.info <- all.participants %>%
  dplyr::select(
    user.id.md5,
    thanks.not.received.skipped,
    thanks.not.received.not.seen,
    thanks.not.received.error,
    thanks.not.received.user.deleted,
    received.multiple.thanks
  )

# Join to get exclusion info for first-gen recipients
excluded.participants <- all.first.gen.recipients %>%
  inner_join(exclusion.info, by = "user.id.md5") %>%
  filter(
    thanks.not.received.skipped == TRUE | thanks.not.received.skipped == "True" |
    thanks.not.received.not.seen == TRUE | thanks.not.received.not.seen == "True" |
    thanks.not.received.error == TRUE | thanks.not.received.error == "True" |
    thanks.not.received.user.deleted == TRUE | thanks.not.received.user.deleted == "True" |
    received.multiple.thanks == TRUE | received.multiple.thanks == "True"
  )

cat("Excluded participants:", nrow(excluded.participants), "\n")
cat("Valid participants (should be ~2552):", nrow(all.first.gen.recipients) - nrow(excluded.participants), "\n")

# Summary by exclusion reason
cat("\n--- Exclusion Reasons Summary ---\n")
cat("Skipped:", sum(excluded.participants$thanks.not.received.skipped == TRUE | excluded.participants$thanks.not.received.skipped == "True"), "\n")
cat("Not seen:", sum(excluded.participants$thanks.not.received.not.seen == TRUE | excluded.participants$thanks.not.received.not.seen == "True"), "\n")
cat("Error:", sum(excluded.participants$thanks.not.received.error == TRUE | excluded.participants$thanks.not.received.error == "True"), "\n")
cat("User deleted:", sum(excluded.participants$thanks.not.received.user.deleted == TRUE | excluded.participants$thanks.not.received.user.deleted == "True"), "\n")
cat("Multiple thanks:", sum(excluded.participants$received.multiple.thanks == TRUE | excluded.participants$received.multiple.thanks == "True"), "\n")

# Show excluded participants details with full datetime
cat("\n--- Excluded Participants Details ---\n")
excluded.participants %>%
  dplyr::select(user.id.md5, first.thank.dt, lang, randomization.block.id,
                thanks.not.received.skipped, thanks.not.received.not.seen,
                thanks.not.received.error, thanks.not.received.user.deleted,
                received.multiple.thanks) %>%
  mutate(first.thank.dt = as.character(first.thank.dt)) %>%
  as_tibble() %>%
  print(n = 200, width = Inf)

In [ ]:
# Count participants per language where first gen are not NA
all.participants.max %>%
  filter(!is.na(earliest.first.gen.thank.in.block.ts) &
         !is.na(earliest.behavior.start.in.block.dt)) %>%
  group_by(lang) %>%
  summarise(count = n(), .groups = "drop")

In [ ]:
# Count participants per language where second gen are not NA
all.participants.max %>%
  filter(!is.na(earliest.second.gen.thank.in.block.ts) &
         !is.na(earliest.behavior.start.in.block.dt)) %>%
  group_by(lang) %>%
  summarise(count = n(), .groups = "drop")

In [ ]:
# Count unique first-gen recipients from reciprocity data
nrow(all.thanks['first.gen.recipient.user.id.md5'] %>% distinct())

# Count participants with first.thank.dt from max cols
table(all.participants.max['first.thank.dt'] != "")


# Extract all usernames with first-gen thanks received
first_gen_recipients <- unique(all.thanks$first.gen.recipient.user.id.md5)

# Find their corresponding group id
recipients_with_blocks <- data.frame(user.id.md5 = first_gen_recipients) %>%
  left_join(
    all.participants.max %>% dplyr::select(user.id.md5, randomization.block.id) %>% distinct(user.id.md5, .keep_all = TRUE),
    by = "user.id.md5"
  )

# Report results
cat("Unique first-gen thank recipients:", length(first_gen_recipients), "\n")
cat("Unique groups (blocks) they belong to:", length(unique(recipients_with_blocks$randomization.block.id[!is.na(recipients_with_blocks$randomization.block.id)])), "\n")
cat("Recipients with missing group id:", sum(is.na(recipients_with_blocks$randomization.block.id)), "\n")

# Show any with missing group id
if(sum(is.na(recipients_with_blocks$randomization.block.id)) > 0) {
  cat("\nUsernames with missing group id:\n")
  print(recipients_with_blocks$user.id.md5[is.na(recipients_with_blocks$randomization.block.id)])
}

In [ ]:
# Get valid participants (no errors/issues with thanks)
# Convert to logical if needed, or compare as strings
valid.thanks.participants <- all.participants %>%
  filter(
    thanks.not.received.skipped != TRUE & thanks.not.received.skipped != "True" &
    thanks.not.received.not.seen != TRUE & thanks.not.received.not.seen != "True" &
    thanks.not.received.error != TRUE & thanks.not.received.error != "True" &
    thanks.not.received.user.deleted != TRUE & thanks.not.received.user.deleted != "True" &
    received.multiple.thanks != TRUE & received.multiple.thanks != "True"
  ) %>%
  dplyr::select(user.id.md5) %>%
  distinct()

cat("Valid thanks participants:", nrow(valid.thanks.participants), "\n")

# Filter all.participants.max to only valid participants before computing earliest
first.gen.received.by.block <- all.participants.max %>%
  filter(first.thank.dt != "") %>%
  filter(user.id.md5 %in% valid.thanks.participants$user.id.md5) %>%
  group_by(randomization.block.id) %>%
  summarise(
    earliest.first.gen.thank.in.block.ts = min(first.thank.dt, na.rm = TRUE),
    .groups = "drop"
  )

cat("Unique first-gen thank recipients after filtering:",
    all.participants.max %>%
      filter(first.thank.dt != "") %>%
      filter(user.id.md5 %in% valid.thanks.participants$user.id.md5) %>%
      nrow(), "\n")

# Step 2: Get block.id for each second-gen sender from all.participants.max
second.gen.sent.by.block <- all.thanks %>%
  inner_join(
    all.participants.max %>%
      dplyr::select(user.id.md5, randomization.block.id) %>%
      distinct(user.id.md5, .keep_all = TRUE),
    by = c("second.gen.sender.user.id.md5" = "user.id.md5")
  ) %>%
  group_by(randomization.block.id) %>%
  summarise(
    earliest.second.gen.thank.in.block.ts = min(second.gen.thank.ts, na.rm = TRUE),
    .groups = "drop"
  )

In [ ]:
# Unique count of second-gen thanks sent
all.thanks %>%
  filter(second.gen.thank.ts != "" & !is.na(second.gen.thank.ts)) %>%
  nrow()

# Number of second-gen thanks received
all.thanks %>%
  filter(second.gen.thank.ts != "" & !is.na(second.gen.thank.ts)) %>%
  nrow()

# Unique second-gen thank recipients
all.thanks %>%
  filter(second.gen.thank.ts != "" & !is.na(second.gen.thank.ts)) %>%
  distinct(second.gen.recipient.user.id.md5) %>%
  nrow()

# Count reciprocal second-gen thanks
all.thanks %>%
  filter(second.gen.thank.ts != "" & !is.na(second.gen.thank.ts)) %>%
  filter(second.gen.recipient.user.id.md5 == first.gen.sender.user.id.md5) %>%
  nrow()

## Distribution of First-Gen Thanks Given and Received

In [ ]:
# Prepare data: join all.participants.max with newcomer info from all.participants
first.gen.data <- all.participants.max %>%
  filter(first.thank.dt != "" & !is.na(first.thank.dt)) %>%
  left_join(
    all.participants %>% dplyr::select(user.id.md5, newcomer),
    by = "user.id.md5"
  ) %>%
  mutate(
    thank.date = as.Date(first.thank.dt),
    experience = ifelse(newcomer == TRUE | newcomer == "TRUE", "Newcomer", "Experienced")
  )

# Total first-gen thanks received
cat("=== First-Gen Thanks Received ===\n")
cat("Total:", nrow(first.gen.data), "\n\n")

# By experience level
cat("By Experience Level:\n")
first.gen.data %>%
  group_by(experience) %>%
  summarise(count = n(), .groups = "drop") %>%
  print()

# By language
cat("\nBy Language:\n")
first.gen.data %>%
  group_by(lang) %>%
  summarise(count = n(), .groups = "drop") %>%
  print()

# By language and experience
cat("\nBy Language and Experience:\n")
first.gen.data %>%
  group_by(lang, experience) %>%
  summarise(count = n(), .groups = "drop") %>%
  tidyr::pivot_wider(names_from = experience, values_from = count, values_fill = 0) %>%
  mutate(Total = Newcomer + Experienced) %>%
  print()


## Time Series Plot

In [ ]:
# Daily counts by experience
daily.counts <- first.gen.data %>%
  group_by(thank.date, experience) %>%
  summarise(count = n(), .groups = "drop")

# Publication-quality theme for LaTeX
theme_pub <- theme_minimal() +
  theme(
    text = element_text(size = 11),
    axis.text = element_text(size = 9),
    axis.text.x = element_text(angle = 45, hjust = 1),
    strip.text = element_text(size = 10, face = "bold"),
    legend.position = "bottom",
    legend.text = element_text(size = 9),
    panel.spacing = unit(0.5, "lines")
  )


# Plot: All participants
p1 <- ggplot(first.gen.data %>% group_by(thank.date) %>% summarise(count = n(), .groups = "drop"),
             aes(x = thank.date, y = count)) +
  geom_line(color = "#56B4E9") +
  geom_point(size = 1, alpha = 0.5) +
  labs(title = "First-Gen Thanks Received Over Time (All)",
       x = "Date", y = "Count") +
  theme_pub

# Plot: By experience level
p2 <- ggplot(daily.counts, aes(x = thank.date, y = count, color = experience)) +
  geom_line() +
  geom_point(size = 1, alpha = 0.5) +
  labs(title = "First-Gen Thanks Received Over Time by Experience",
       x = "Date", y = "Count", color = "Experience") +
  scale_color_manual(values = c("Newcomer" = "#E69F00", "Experienced" = "#56B4E9")) +
  theme_pub

# Plot: By language
daily.by.lang <- first.gen.data %>%
  group_by(thank.date, lang) %>%
  summarise(count = n(), .groups = "drop")

p3 <- ggplot(daily.by.lang, aes(x = thank.date, y = count, color = lang)) +
  geom_line() +
  geom_point(size = 1, alpha = 0.5) +
  labs(title = "First-Gen Thanks Received Over Time by Language",
       x = "Date", y = "Count", color = "Language") +
  scale_color_manual(values = palette.colors(4)) +
  theme_pub

# Plot: Faceted by language and experience
daily.by.lang.exp <- first.gen.data %>%
  group_by(thank.date, lang, experience) %>%
  summarise(count = n(), .groups = "drop")

p4 <- ggplot(daily.by.lang.exp, aes(x = thank.date, y = count, color = experience)) +
  geom_line() +
  geom_point(size = 1, alpha = 0.3) +
  facet_wrap(~lang, scales = "free_y") +
  labs(title = "First-Gen Thanks Received by Language and Experience",
       x = "Date", y = "Count", color = "Experience") +
  scale_color_manual(values = c("Newcomer" = "#E69F00", "Experienced" = "#56B4E9")) +
  theme_pub

print(p1)
print(p2)
print(p3)
print(p4)

## Distribution of Second-Gen Thanks Given and Received


In [ ]:
# Prepare data: join all.thanks with participant info from all.participants.max

second.gen.data <- all.thanks %>%
filter(second.gen.thank.ts != "" & !is.na(second.gen.thank.ts)) %>%
inner_join(
all.participants.max %>% dplyr::select(user.id.md5, user.id.md5, randomization.block.id, lang) %>%
distinct(user.id.md5, .keep_all = TRUE),
by = c("second.gen.sender.user.id.md5" = "user.id.md5")
) %>%
left_join(
all.participants %>% dplyr::select(user.id.md5, newcomer),
by = "user.id.md5"
) %>%
mutate(
thank.date = as.Date(second.gen.thank.ts),
experience = ifelse(newcomer == TRUE | newcomer == "TRUE", "Newcomer", "Experienced")
)

# --- Summary Tables ---

# Total second-gen thanks sent
cat("=== Second-Gen Thanks Sent ===\n")
cat("Total:", nrow(second.gen.data), "\n\n")

# By experience level
cat("By Experience Level:\n")
second.gen.data %>%
group_by(experience) %>%
summarise(count = n(), .groups = "drop") %>%
print()

# By language
# Using lang.y instead of lang.x: The second-gen sender is the first-gen recipient who is now sending thanks. So lang.y is the language of the participant who received first-gen thanks and is now sending second-gen thanks.
cat("\nBy Language:\n")
second.gen.data %>%
group_by(lang.y) %>%
summarise(count = n(), .groups = "drop") %>%
print()

# By language and experience
cat("\nBy Language and Experience:\n")
second.gen.data %>%
group_by(lang.y, experience) %>%
summarise(count = n(), .groups = "drop") %>%
tidyr::pivot_wider(names_from = experience, values_from = count, values_fill = 0) %>%
mutate(Total = Newcomer + Experienced) %>%
print()

## Time Series Plot for Second-Gen Thanks

In [ ]:
# Daily counts by experience
daily.counts.2nd <- second.gen.data %>%
group_by(thank.date, experience) %>%
summarise(count = n(), .groups = "drop")

# Plot: All participants
p1.2nd <- ggplot(second.gen.data %>% group_by(thank.date) %>% summarise(count = n(), .groups = "drop"),
aes(x = thank.date, y = count)) +
geom_line(color = "#56B4E9") +
geom_point(size = 1, alpha = 0.5) +
labs(title = "Second-Gen Thanks Sent Over Time (All)",
x = "Date", y = "Count") +
theme_minimal()

# Plot: By experience level
p2.2nd <- ggplot(daily.counts.2nd, aes(x = thank.date, y = count, color = experience)) +
geom_line() +
geom_point(size = 1, alpha = 0.5) +
labs(title = "Second-Gen Thanks Sent Over Time by Experience",
x = "Date", y = "Count", color = "Experience") +
scale_color_manual(values = c("Newcomer" = "#E69F00", "Experienced" = "#56B4E9")) +
theme_minimal()

# Plot: By language
daily.by.lang.2nd <- second.gen.data %>%
group_by(thank.date, lang.y) %>%
summarise(count = n(), .groups = "drop")

p3.2nd <- ggplot(daily.by.lang.2nd, aes(x = thank.date, y = count, color = lang.y)) +
geom_line() +
geom_point(size = 1, alpha = 0.5) +
labs(title = "Second-Gen Thanks Sent Over Time by Language",
x = "Date", y = "Count", color = "Language") +
scale_color_manual(values = palette.colors(4)) +
theme_minimal()

# Plot: Faceted by language and experience
daily.by.lang.exp.2nd <- second.gen.data %>%
group_by(thank.date, lang.y, experience) %>%
summarise(count = n(), .groups = "drop")

p4.2nd <- ggplot(daily.by.lang.exp.2nd, aes(x = thank.date, y = count, color = experience)) +
geom_line() +
geom_point(size = 1, alpha = 0.3) +
facet_wrap(~lang.y, scales = "free_y") +
labs(title = "Second-Gen Thanks Sent by Language and Experience",
x = "Date", y = "Count", color = "Experience") +
scale_color_manual(values = c("Newcomer" = "#E69F00", "Experienced" = "#56B4E9")) +
theme_minimal()

print(p1.2nd)
print(p2.2nd)
print(p3.2nd)
print(p4.2nd)

## Additional plots for first gen thanks

In [ ]:
# 1. CUMULATIVE DISTRIBUTION - Shows growth over time
p.cumulative <- first.gen.data %>%
  arrange(thank.date) %>%
  group_by(lang) %>%
  mutate(cumulative = row_number()) %>%
  ggplot(aes(x = thank.date, y = cumulative, color = lang)) +
  geom_line(linewidth = 1) +
  labs(title = "Cumulative First-Gen Thanks Over Time",
       x = "Date", y = "Cumulative Count") +
  scale_color_manual(values = palette.colors(4)) +
  theme_pub

# 2. HISTOGRAM - Better for showing distribution shape
p.hist <- ggplot(first.gen.data, aes(x = thank.date, fill = lang)) +
  geom_histogram(bins = 30, alpha = 0.7, position = "dodge") +
  labs(title = "Distribution of First-Gen Thanks",
       x = "Date", y = "Count") +
  scale_fill_manual(values = palette.colors(4)) +
  theme_pub

# 3. DENSITY PLOT - Smooth distribution comparison
p.density <- ggplot(first.gen.data, aes(x = thank.date, fill = lang)) +
  geom_density(alpha = 0.5) +
  labs(title = "Density of First-Gen Thanks by Language",
       x = "Date", y = "Density") +
  scale_fill_manual(values = palette.colors(4)) +
  theme_pub

# 4. WEEKLY AGGREGATED BAR - Less noisy than daily
# p.weekly <- first.gen.data %>%
#   mutate(week = floor_date(thank.date, "week")) %>%
#   group_by(week, lang, experience) %>%
#   summarise(count = n(), .groups = "drop") %>%
#   ggplot(aes(x = week, y = count, fill = experience)) +
#   geom_col(position = "stack") +
#   facet_wrap(~lang) +
#   labs(title = "Weekly First-Gen Thanks by Language & Experience",
#        x = "Week", y = "Count") +
#   theme_pub

p.weekly <- first.gen.data %>%
  mutate(week = floor_date(thank.date, "week")) %>%
  group_by(week, lang, experience) %>%
  summarise(count = n(), .groups = "drop") %>%
  ggplot(aes(x = week, y = count, fill = experience)) +
  geom_col(position = "stack") +
  facet_wrap(~lang, ncol = 1) +
  scale_x_date(date_labels = "%b %d", date_breaks = "4 weeks") +
  labs(
      # title = "Weekly First-Gen Thanks by Language & Experience",
       x = "Week (Start Date)", y = "Count of Thanks Received") +
  scale_fill_manual(values = c("Newcomer" = "#E69F00", "Experienced" = "#56B4E9")) +
  theme_pub


# 5. RIDGELINE PLOT - Compact comparison across groups
library(ggridges)
p.ridge <- ggplot(first.gen.data, aes(x = thank.date, y = lang, fill = lang)) +
  geom_density_ridges(alpha = 0.7) +
  labs(title = "Distribution of Thanks by Language",
       x = "Date", y = "") +
  scale_fill_manual(values = palette.colors(4)) +
  theme_pub +
  theme(legend.position = "none")


print(p.cumulative)
print(p.hist)
print(p.density)
print(p.weekly)
print(p.ridge)

ggsave("figs/first.gen.thanks.pdf", p.weekly, width = 6.5, height = 8, dpi = 300)



## Additional plots for second gen thanks

In [ ]:
# 1. CUMULATIVE DISTRIBUTION - Shows growth over time
p.cumulative.2nd <- second.gen.data %>%
  arrange(thank.date) %>%
  group_by(lang.y) %>%
  mutate(cumulative = row_number()) %>%
  ggplot(aes(x = thank.date, y = cumulative, color = lang.y)) +
  geom_line(linewidth = 1) +
  labs(title = "Cumulative Second-Gen Thanks Over Time",
       x = "Date", y = "Cumulative Count", color = "Language") +
  scale_color_manual(values = palette.colors(4)) +
  theme_pub

# 2. HISTOGRAM - Better for showing distribution shape
p.hist.2nd <- ggplot(second.gen.data, aes(x = thank.date, fill = lang.y)) +
  geom_histogram(bins = 30, alpha = 0.7, position = "dodge") +
  labs(title = "Distribution of Second-Gen Thanks",
       x = "Date", y = "Count", fill = "Language") +
  scale_fill_manual(values = palette.colors(4)) +
  theme_pub

# # 3. DENSITY PLOT - Smooth distribution comparison
# p.density.2nd <- ggplot(second.gen.data, aes(x = thank.date, fill = lang.y)) +
#   geom_density(alpha = 0.5) +
#   labs(title = "Density of Second-Gen Thanks by Language",
#        x = "Date", y = "Density", fill = "Language") +
#   theme_pub

# 4. WEEKLY AGGREGATED BAR - Less noisy than daily
# p.weekly.2nd <- second.gen.data %>%
#   mutate(week = floor_date(thank.date, "week")) %>%
#   group_by(week, lang.y, experience) %>%
#   summarise(count = n(), .groups = "drop") %>%
#   ggplot(aes(x = week, y = count, fill = experience)) +
#   geom_col(position = "stack") +
#   facet_wrap(~lang.y) +
#   labs(title = "Weekly Second-Gen Thanks by Language & Experience",
#        x = "Week", y = "Count") +
#   theme_pub

p.weekly.2nd <- second.gen.data %>%
  mutate(week = floor_date(thank.date, "week")) %>%
  group_by(week, lang.y, experience) %>%
  summarise(count = n(), .groups = "drop") %>%
  ggplot(aes(x = week, y = count, fill = experience)) +
  geom_col(position = "stack") +
  facet_wrap(~lang.y, ncol = 1) +
  scale_x_date(date_labels = "%b %d", date_breaks = "4 weeks") +
  labs(
      # title = "Weekly Second-Gen Thanks by Language & Experience",
       x = "Week (Start Date)", y = "Count of Thanks Received") +
  scale_fill_manual(values = c("Newcomer" = "#E69F00", "Experienced" = "#56B4E9")) +
  theme_pub

# # 5. RIDGELINE PLOT - Compact comparison across groups
# p.ridge.2nd <- ggplot(second.gen.data, aes(x = thank.date, y = lang.y, fill = lang.y)) +
#   geom_density_ridges(alpha = 0.7) +
#   labs(title = "Distribution of Second-Gen Thanks by Language",
#        x = "Date", y = "") +
#   theme_pub +
#   theme(legend.position = "none")

print(p.cumulative.2nd)
print(p.hist.2nd)
# print(p.density.2nd)
print(p.weekly.2nd)
# print(p.ridge.2nd)

ggsave("figs/sec.gen.thanks.pdf", p.weekly.2nd, width = 6.5, height = 8, dpi = 300)

## Thanks sent by the controlled group

In [ ]:
# === CONTROL GROUP: Thanks sent by control participants ===
# Use all.participants.max joined with treatment info to get thanks.sent counts
# and all.thanks to find actual thanks activity by control group

# Get control group participants
control.participants <- all.participants.max %>%
  filter(randomization.arm == 0) %>%
  left_join(
    all.participants %>%
      dplyr::select(user.id.md5, newcomer) %>%
      distinct(user.id.md5, .keep_all = TRUE),
    by = "user.id.md5"
  )

cat("=== Control Group Participants ===\n")
cat("Total control participants:", nrow(control.participants), "\n")
cat("Control participants with thanks.sent > 0:", sum(control.participants$thanks.sent > 0, na.rm = TRUE), "\n")

# Find thanks activity by control group in all.thanks
# Check if control participants appear as second-gen senders
control.thanks.in.all.thanks <- all.thanks %>%
  filter(second.gen.thank.ts != "" & !is.na(second.gen.thank.ts)) %>%
  inner_join(
    control.participants %>%
      dplyr::select(user.id.md5, user.id.md5, lang, randomization.block.id) %>%
      distinct(user.id.md5, .keep_all = TRUE),
    by = c("second.gen.sender.user.id.md5" = "user.id.md5")
  ) %>%
  left_join(
    all.participants %>% dplyr::select(user.id.md5, newcomer),
    by = "user.id.md5"
  ) %>%
  mutate(
    thank.date = as.Date(second.gen.thank.ts),
    experience = ifelse(newcomer == TRUE | newcomer == "TRUE", "Newcomer", "Experienced"),
    group = "Control"
  )

cat("\nControl group thanks in all.thanks:", nrow(control.thanks.in.all.thanks), "\n")

# Get treated group second-gen thanks for comparison
treated.participants <- all.participants.max %>%
  filter(randomization.arm == 1) %>%
  left_join(
    all.participants %>%
      dplyr::select(user.id.md5, newcomer) %>%
      distinct(user.id.md5, .keep_all = TRUE),
    by = "user.id.md5"
  )

treated.thanks.in.all.thanks <- all.thanks %>%
  filter(second.gen.thank.ts != "" & !is.na(second.gen.thank.ts)) %>%
  inner_join(
    treated.participants %>%
      dplyr::select(user.id.md5, user.id.md5, lang, randomization.block.id) %>%
      distinct(user.id.md5, .keep_all = TRUE),
    by = c("second.gen.sender.user.id.md5" = "user.id.md5")
  ) %>%
  left_join(
    all.participants %>% dplyr::select(user.id.md5, newcomer),
    by = "user.id.md5"
  ) %>%
  mutate(
    thank.date = as.Date(second.gen.thank.ts),
    experience = ifelse(newcomer == TRUE | newcomer == "TRUE", "Newcomer", "Experienced"),
    group = "Treated"
  )

cat("Treated group thanks in all.thanks:", nrow(treated.thanks.in.all.thanks), "\n")

# Combine for comparison
combined.thanks <- bind_rows(control.thanks.in.all.thanks, treated.thanks.in.all.thanks)

# Summary
cat("\n=== Comparison: Thanks Sent ===\n")
combined.thanks %>%
  group_by(group) %>%
  summarise(
    total_thanks = n(),
    unique_senders = n_distinct(second.gen.sender.user.id.md5),
    .groups = "drop"
  ) %>%
  print()

cat("\nBy group, language, and experience:\n")
combined.thanks %>%
  group_by(group, lang.y, experience) %>%
  summarise(count = n(), .groups = "drop") %>%
  tidyr::pivot_wider(names_from = group, values_from = count, values_fill = 0) %>%
  print()


In [ ]:
# Weekly thanks distribution for control group only, by language and experience
p.weekly.control <- control.thanks.in.all.thanks %>%
  mutate(week = floor_date(thank.date, "week")) %>%
  group_by(week, lang.y, experience) %>%
  summarise(count = n(), .groups = "drop") %>%
  ggplot(aes(x = week, y = count, fill = experience)) +
  geom_col(position = "stack") +
  facet_wrap(~lang.y, ncol = 1) +
  scale_x_date(date_labels = "%b %d", date_breaks = "4 weeks") +
  labs(x = "Week (Start Date)", y = "Count of Thanks Sent") +
  scale_fill_manual(values = c("Newcomer" = "#E69F00", "Experienced" = "#56B4E9")) +
  theme_pub

print(p.weekly.control)

# === WEEKLY COMPARISON PLOT: Treated vs Control Thanks Sent ===

p.weekly.comparison <- combined.thanks %>%
  mutate(week = floor_date(thank.date, "week")) %>%
  group_by(week, lang.y, group) %>%
  summarise(count = n(), .groups = "drop") %>%
  ggplot(aes(x = week, y = count, fill = group)) +
  geom_col(position = "dodge") +
  facet_wrap(~lang.y, ncol = 1) +
  scale_x_date(date_labels = "%b %d", date_breaks = "4 weeks") +
  labs(x = "Week (Start Date)", y = "Count of Thanks Sent", fill = "Group") +
  scale_fill_manual(values = c("Control" = "#E69F00", "Treated" = "#56B4E9")) +
  theme_pub

p.weekly.comparison.exp <- combined.thanks %>%
  mutate(week = floor_date(thank.date, "week")) %>%
  group_by(week, lang.y, group, experience) %>%
  summarise(count = n(), .groups = "drop") %>%
  ggplot(aes(x = week, y = count, fill = group)) +
  geom_col(position = "dodge") +
  facet_wrap(~lang.y + experience, ncol = 1) +
  scale_x_date(date_labels = "%b %d", date_breaks = "4 weeks") +
  labs(x = "Week (Start Date)", y = "Count of Thanks Sent",
       fill = "Group") +
  scale_fill_manual(values = c("Control" = "#E69F00", "Treated" = "#56B4E9")) +
  theme_pub

print(p.weekly.comparison)
print(p.weekly.comparison.exp)

ggsave("figs/control.group.thanks.pdf", p.weekly.control, width = 6.5, height = 8, dpi = 300)
ggsave("figs/treated.vs.control.thanks.pdf", p.weekly.comparison, width = 6.5, height = 8, dpi = 300)


## BALANCE TABLE: Thanks Distribution by Group, Language, Experience

In [ ]:
lang_names <- c(ar = "Arabic", de = "German", fa = "Persian", pl = "Polish")

# Summarize first-gen
first.gen.summary <- first.gen.data %>%
  group_by(lang, experience) %>%
  summarise(count = n(), .groups = "drop") %>%
  rename(first_gen = count)

# Summarize second-gen (treated only)
treated.summary <- treated.thanks.in.all.thanks %>%
  group_by(lang.y, experience) %>%
  summarise(count = n(), .groups = "drop") %>%
  rename(lang = lang.y, second_gen_treated = count)

# Summarize control
control.summary <- control.thanks.in.all.thanks %>%
  group_by(lang.y, experience) %>%
  summarise(count = n(), .groups = "drop") %>%
  rename(lang = lang.y, control = count)

# Merge all
thanks.table <- first.gen.summary %>%
  full_join(treated.summary, by = c("lang", "experience")) %>%
  full_join(control.summary, by = c("lang", "experience")) %>%
  replace(is.na(.), 0) %>%
  arrange(lang, desc(experience))

# Totals by language
lang.totals <- thanks.table %>%
  group_by(lang) %>%
  summarise(first_gen = sum(first_gen),
            second_gen_treated = sum(second_gen_treated),
            control = sum(control), .groups = "drop")

# Totals by experience
exp.totals <- thanks.table %>%
  group_by(experience) %>%
  summarise(first_gen = sum(first_gen),
            second_gen_treated = sum(second_gen_treated),
            control = sum(control), .groups = "drop")

# Grand totals
grand.total <- thanks.table %>%
  summarise(first_gen = sum(first_gen),
            second_gen_treated = sum(second_gen_treated),
            control = sum(control))

# --- Build LaTeX ---
fmt_n_pct <- function(n, total) {
  sprintf("%s (%.1f\\%%)", formatC(n, format = "d", big.mark = ","), n / total * 100)
}

latex <- c()
latex <- c(latex, "\\begin{table*}[t]")
latex <- c(latex, "\\centering")
latex <- c(latex, "\\caption{Distribution of thanks by group, Wikipedia language edition, and participant experience level. First-generation thanks are those sent by the research team to treatment group participants as part of the experimental intervention. Second-generation thanks (treated) are organic thanks subsequently sent by treated participants to other editors. Control group thanks are organic thanks sent by control group participants who did not receive the intervention. Percentages are column-wise within each group.}")
latex <- c(latex, "\\label{table:thanks-distribution}")
latex <- c(latex, "\\begin{tabular}{lrrr}")
latex <- c(latex, "\\toprule")
latex <- c(latex, sprintf(" & \\textbf{First-Gen} & \\textbf{Second-Gen} & \\textbf{Control} \\\\"))
latex <- c(latex, sprintf(" & \\textbf{(Received)} & \\textbf{(Treated)} & \\textbf{(Sent)} \\\\"))
latex <- c(latex, sprintf(" & (N = %s) & (N = %s) & (N = %s) \\\\",
  formatC(grand.total$first_gen, big.mark = ","),
  formatC(grand.total$second_gen_treated, big.mark = ","),
  formatC(grand.total$control, big.mark = ",")))
latex <- c(latex, "\\midrule")

# By Experience
latex <- c(latex, "\\multicolumn{4}{l}{\\textit{Experience level}} \\\\")
for (i in 1:nrow(exp.totals)) {
  latex <- c(latex, sprintf("\\quad %s & %s & %s & %s \\\\",
    exp.totals$experience[i],
    fmt_n_pct(exp.totals$first_gen[i], grand.total$first_gen),
    fmt_n_pct(exp.totals$second_gen_treated[i], grand.total$second_gen_treated),
    fmt_n_pct(exp.totals$control[i], grand.total$control)))
}

# By Language
latex <- c(latex, "\\addlinespace")
latex <- c(latex, "\\multicolumn{4}{l}{\\textit{Language}} \\\\")
for (l in c("ar", "de", "fa", "pl")) {
  row <- lang.totals %>% filter(lang == l)
  latex <- c(latex, sprintf("\\quad %s & %s & %s & %s \\\\",
    lang_names[l],
    fmt_n_pct(row$first_gen, grand.total$first_gen),
    fmt_n_pct(row$second_gen_treated, grand.total$second_gen_treated),
    fmt_n_pct(row$control, grand.total$control)))
}

# By Language x Experience
latex <- c(latex, "\\addlinespace")
latex <- c(latex, "\\multicolumn{4}{l}{\\textit{Language $\\times$ Experience}} \\\\")
for (l in c("ar", "de", "fa", "pl")) {
  rows <- thanks.table %>% filter(lang == l)
  for (j in 1:nrow(rows)) {
    latex <- c(latex, sprintf("\\quad %s -- %s & %s & %s & %s \\\\",
      lang_names[l], rows$experience[j],
      fmt_n_pct(rows$first_gen[j], grand.total$first_gen),
      fmt_n_pct(rows$second_gen_treated[j], grand.total$second_gen_treated),
      fmt_n_pct(rows$control[j], grand.total$control)))
  }
}

latex <- c(latex, "\\bottomrule")
latex <- c(latex, "\\end{tabular}")
latex <- c(latex, "\\end{table*}")

# Write to file
writeLines(latex, "tables/thanks_distribution_table.tex")
cat("Written to tables/thanks_distribution_table.tex\n\n")
cat(paste(latex, collapse = "\n"))

# Analysis

## Create Study Sub-Samples

In [ ]:
newcomers.df <- subset(participants, newcomer==TRUE)
experienced.df <- subset(participants, newcomer==FALSE)
newcomer.compliers.df <- subset(newcomers.df,complier==TRUE)

In [ ]:
# CrossTable(newcomers.df$lang, newcomers.df$TREAT,
#            prop.c = FALSE, prop.r=FALSE,
#            prop.t=FALSE, prop.chisq = FALSE)

In [ ]:
# CrossTable(experienced.df$lang, experienced.df$TREAT,
#            prop.c = FALSE, prop.r=FALSE,
#            prop.t=FALSE, prop.chisq = FALSE)

In [ ]:
# CrossTable(newcomers.df$TREAT, newcomers.df$complier,
#            prop.c = FALSE, prop.r=TRUE,
#            prop.t=FALSE, prop.chisq = FALSE)

# Per-Language Analysis Method

In [ ]:
df.estimates <- function(df, lang="all", subgroup="all"){

    df.compliers <- subset(df, complier)

    thanks.model.formula <- "thanks.sent ~ TREAT"
    if(lang=="all"){
        thanks.model.formula <- paste(thanks.model.formula, " + lang")
    }

    if(subgroup=="all"){
        thanks.model.formula <- paste(thanks.model.formula, " + newcomer")
    }

    thanks.model         <- glm.nb(thanks.model.formula, data=df)
    m.thanks.sent        <- coef(summary(thanks.model))
    m.thanks.sent        <- data.frame(cbind(m.thanks.sent, confint(thanks.model)))
    names(m.thanks.sent)[4] <- "pvalue"
    names(m.thanks.sent)[5] <- "CI.Lower"
    names(m.thanks.sent)[6] <- "CI.Upper"
    m.thanks.sent$DF       <- thanks.model$df.residual
    m.thanks.sent$model    <- "thanks sent"
    m.thanks.sent$t.value  <- NA
    m.thanks.sent$n.size   <- nrow(df)
    m.thanks.sent$n.thanked.count <- nrow(subset(df, complier.app.any.reason))

    m.retention            <- data.frame(coef(summary(difference_in_means(two.week.retention ~ TREAT, blocks=block, ci=TRUE, data=df))))
    m.retention$model      <- "retention"
    m.retention$z.value    <- NA
    m.retention$n.size     <- nrow(df)
    m.retention$n.thanked.count <- nrow(subset(df, complier.app.any.reason))


# Fit labor hours per day model for this specific subset
if(lang == "all" & subgroup == "all") {
    # For overall analysis, use the full model with random effects
    labor.per.day.formula <- "labor.hours.per.day.post.treatment ~ TREAT*centered.labor.hours.per.day.pre.treatment + (1|block)"
} else {
    # For subgroup analyses, use simpler model without random effects
    labor.per.day.formula <- "labor.hours.per.day.post.treatment ~ TREAT*centered.labor.hours.per.day.pre.treatment"
}

# Check if the variables exist
if(!"centered.labor.hours.per.day.pre.treatment" %in% names(df)) {
    warning("centered.labor.hours.per.day.pre.treatment not found in data, skipping labor hours model")
    m.labor.hours <- NULL
} else {
    labor.per.day.model <- glmmTMB(as.formula(labor.per.day.formula),
                                   data=df,
                                   family=tweedie(link="log"))

    m.labor.hours <- data.frame(coef(summary(labor.per.day.model))$cond)
    m.labor.hours <- m.labor.hours[rownames(m.labor.hours) == "TREAT1", , drop = FALSE]

    # Add confidence intervals
    labor.hours.ci <- confint(labor.per.day.model)
    m.labor.hours$CI.Lower <- labor.hours.ci["TREAT1", "2.5 %"]
    m.labor.hours$CI.Upper <- labor.hours.ci["TREAT1", "97.5 %"]

    m.labor.hours$model <- "labor hours per day"
    m.labor.hours$n.size <- nobs(labor.per.day.model)
    m.labor.hours$n.thanked.count <- nrow(subset(df, complier.app.any.reason))
}

    m.manip.check          <- data.frame(coef(summary(difference_in_means(remembered.thanks<=3 ~ TREAT, ci=TRUE, data=df.compliers))))
    m.manip.check$model    <- "manipulation check"
    m.manip.check$z.value  <- NA
    m.manip.check$n.size   <- nrow(df.compliers)
    m.manip.check$n.thanked.count <-  nrow(subset(df.compliers, complier.app.any.reason))

    names(m.retention)[4]           <- "pvalue"
    names(m.manip.check)[4]         <- "pvalue"
    names(m.labor.hours)[4]         <- "pvalue"

    m.thanks.sent$estimator        <- "glm.nb"
    m.retention$estimator          <- "difference_in_means"
    m.labor.hours$estimator        <- "glmmTMB (Tweedie)"
    m.manip.check$estimator        <- "difference_in_means"

    result.df <- dplyr::bind_rows(m.thanks.sent, m.retention, m.labor.hours, m.manip.check)

    result.df <- result.df[str_detect(row.names(result.df), "TREAT"),]

    result.df[row.names(result.df)!="(Intercept)",]

    result.df$lang     <- lang
    result.df$subgroup <- subgroup

    result.df
}

### Aggregate and Adjust Results for All Participants

In [ ]:
all.results <- df.estimates(participants)
# all.results <- subset(all.results, model!="labor hours diff")

# all.newcomer.results <- df.estimates(subset(participants, newcomer), subgroup="newcomer")
# all.results <- dplyr::bind_rows(all.results, subset(all.newcomer.results, model=="labor hours diff"))

## Adjust for multiple comparisons
# adjustment needs to be done at the level of each
# dataframe, since the tests are different for
# different subgroups and languages.
# When doing adjustment for multiple comparisons
# we omit the manipulation check, since it's not
# one of the confirmatory tests

## Note: store the original p values in a list so they can be
## used for later adjustments
all.results.pvalues <- all.results[all.results$model!="manipulation check",]$pvalue


all.results[all.results$model!="manipulation check",]$pvalue <- p.adjust(all.results[all.results$model!="manipulation check",]$pvalue, method="holm")

In [ ]:
all.results[c("model","Estimate", "pvalue", "CI.Lower", "CI.Upper", "n.size", "estimator", "subgroup")]
print(all.results[,c("model", "estimator", "subgroup", "n.size", "Estimate", "Std..Error", "pvalue")])

In [ ]:
all.results[all.results$model!="manipulation check",]

## Diagnostics

In [ ]:
# Post-drop Thanks Sent model + DHARMa diagnostics
thanks.model <- glm.nb(thanks.sent ~ TREAT + lang + newcomer, data = participants)
cat("=== Thanks Sent (glm.nb) — Post-Drop (N =", nrow(participants), ") ===\n")
summary(thanks.model)

sim.thanks <- simulateResiduals(thanks.model, n=1000)
plot(sim.thanks, title="Thanks Sent (Negative Binomial) — Post-Drop")
testDispersion(sim.thanks)
testZeroInflation(sim.thanks)

# Save for SI
png("figs/si_diagnostic_thanks_sent.png", width=1600, height=700, res=150)
plot(sim.thanks, title="Thanks Sent (Negative Binomial)")
dev.off()

In [ ]:
# Pre-registered DiM model on labor hours diff — showing poor fit
# (Motivates switch to Tweedie, cited in Methods)
dim.labor <- lm(labor.hours.per.day.diff ~ TREAT, data = participants)

par(mfrow=c(1,2))
qqnorm(residuals(dim.labor),
       main="Pre-registered DiM: Labor Hours Diff\nNormal Q-Q Plot",
       pch=20, cex=0.3, col=rgb(0,0,0,0.3))
qqline(residuals(dim.labor), col="red", lwd=2)

plot(fitted(dim.labor), residuals(dim.labor),
     main="Pre-registered DiM: Labor Hours Diff\nResiduals vs Fitted",
     xlab="Fitted values", ylab="Residuals",
     pch=20, cex=0.3, col=rgb(0,0,0,0.3))
abline(h=0, col="red", lwd=2)
par(mfrow=c(1,1))

# Save for SI
png("figs/si_diagnostic_dim_labor_hours.png", width=1600, height=700, res=150)
par(mfrow=c(1,2))
qqnorm(residuals(dim.labor),
       main="Pre-registered DiM: Labor Hours Diff\nNormal Q-Q Plot",
       pch=20, cex=0.3, col=rgb(0,0,0,0.3))
qqline(residuals(dim.labor), col="red", lwd=2)
plot(fitted(dim.labor), residuals(dim.labor),
     main="Pre-registered DiM: Labor Hours Diff\nResiduals vs Fitted",
     xlab="Fitted values", ylab="Residuals",
     pch=20, cex=0.3, col=rgb(0,0,0,0.3))
abline(h=0, col="red", lwd=2)
par(mfrow=c(1,1))
dev.off()

## Tweedie Models For Labor Hour Measures That Handle Overdispersion of Zeros

### Models for Total Labor Hours

In [ ]:
# Total Labor Hours Baseline Model
summary(lm.labor.tweedie.baseline <- glmmTMB(labor.hours.post.treatment ~ TREAT + (1|block),
                            data = participants,
                            family = tweedie(link = "log")))

# Total Labor Hours With Pre Treatment Covariate
summary(lm.labor.tweedie <- glmmTMB(labor.hours.post.treatment ~ TREAT + centered.labor.hours.pre.treatment + (1|block),
                            data = participants,
                            family = tweedie(link = "log")))

# Total Labor Hours With Interaction
summary(lm.labor.tweedie.int <- glmmTMB(labor.hours.post.treatment ~ TREAT*centered.labor.hours.pre.treatment + (1|block),
                            data = participants,
                            family = tweedie(link = "log")))

### Models for Labor Hours Per Day

In [ ]:
# Labor Hours Per Day Baseline Model
summary(lm.labor.per.day.tweedie.baseline <- glmmTMB(labor.hours.per.day.post.treatment ~ TREAT + (1|block),
                            data = participants,
                            family = tweedie(link = "log")))

# Labor Hours Per Day With Pre Treatment Covariate
summary(lm.labor.per.day.tweedie <- glmmTMB(labor.hours.per.day.post.treatment ~ TREAT + centered.labor.hours.per.day.pre.treatment + (1|block),
                            data = participants,
                            family = tweedie(link = "log")))

# Labor Hours Per Day With Interaction
summary(lm.labor.per.day.tweedie.int <- glmmTMB(labor.hours.per.day.post.treatment ~ TREAT*centered.labor.hours.per.day.pre.treatment + (1|block),
                            data = participants,
                            family = tweedie(link = "log")))

In [ ]:
# Simulate residuals for Total Labor Hours
simRes_tweedie_baseline <- simulateResiduals(lm.labor.per.day.tweedie.baseline)
simRes_tweedie <- simulateResiduals(lm.labor.per.day.tweedie)
simRes_tweedie_int <- simulateResiduals(lm.labor.per.day.tweedie.int)

# Diagnostic plots for Total Labor Hours
plot(simRes_tweedie_baseline, main="Hours Per Day - Baseline Model")
plot(simRes_tweedie, main="Hours Per Day - With Covariate")
plot(simRes_tweedie_int, main="Hours Per Day - With Interaction")

# Test assumptions for Total Labor Hours
# Baseline
testDispersion(simRes_tweedie_baseline)
testOutliers(simRes_tweedie_baseline)
testZeroInflation(simRes_tweedie_baseline)

# Covariate
testDispersion(simRes_tweedie)
testOutliers(simRes_tweedie)
testZeroInflation(simRes_tweedie)

# Interaction
testDispersion(simRes_tweedie_int)
testOutliers(simRes_tweedie_int)
testZeroInflation(simRes_tweedie_int)

# Compare Total Labor Hours models
compare_performance(lm.labor.per.day.tweedie.baseline,
                    lm.labor.per.day.tweedie,
                    lm.labor.per.day.tweedie.int,
                    rank=TRUE)
print(compare_performance)

# Summary interpretation
best_model_idx <- which.min(c(AIC(lm.labor.per.day.tweedie.baseline),
                               AIC(lm.labor.per.day.tweedie),
                               AIC(lm.labor.per.day.tweedie.int)))
best_model_name <- c("Baseline", "With Covariate", "With Interaction")[best_model_idx]

### Models for Labor Hours Per Day

In [ ]:
# Simulate residuals for Labor Hours Per Day
simRes_tweedie_per_day_baseline <- simulateResiduals(lm.labor.per.day.tweedie.baseline)
simRes_tweedie_per_day <- simulateResiduals(lm.labor.per.day.tweedie)
simRes_tweedie_per_day_int <- simulateResiduals(lm.labor.per.day.tweedie.int)

# Diagnostic plots for Labor Hours Per Day
plot(simRes_tweedie_per_day_baseline, main="Per Day - Baseline Model")
plot(simRes_tweedie_per_day, main="Per Day - With Covariate")
plot(simRes_tweedie_per_day_int, main="Per Day - With Interaction")

# Test assumptions for Labor Hours Per Day
# Baseline
testDispersion(simRes_tweedie_per_day_baseline)
testOutliers(simRes_tweedie_per_day_baseline)
testZeroInflation(simRes_tweedie_per_day_baseline)

# Covariate
testDispersion(simRes_tweedie_per_day)
testOutliers(simRes_tweedie_per_day)
testZeroInflation(simRes_tweedie_per_day)

# Interaction
testDispersion(simRes_tweedie_per_day_int)
testOutliers(simRes_tweedie_per_day_int)
testZeroInflation(simRes_tweedie_per_day_int)

# Model Comparison
comp_perday <- compare_performance(lm.labor.per.day.tweedie.baseline,
                                    lm.labor.per.day.tweedie,
                                    lm.labor.per.day.tweedie.int,
                                    rank=TRUE)
print(comp_perday)

# Summary interpretation
best_model_pd_idx <- which.min(c(AIC(lm.labor.per.day.tweedie.baseline),
                                  AIC(lm.labor.per.day.tweedie),
                                  AIC(lm.labor.per.day.tweedie.int)))
best_model_pd_name <- c("Baseline", "With Covariate", "With Interaction")[best_model_pd_idx]

## Separate Tweedie Models for Newcomers and Experienced Contributors

In [ ]:
# Split analysis by newcomer status
# Re-center covariates within each group
newcomers.df$centered.labor.hours.pre.treatment <-
scale(newcomers.df$labor.hours.pre.treatment, center = TRUE, scale = FALSE)

newcomers.df$centered.labor.hours.per.day.pre.treatment <-
scale(newcomers.df$labor.hours.per.day.pre.treatment, center = TRUE, scale = FALSE)

experienced.df$centered.labor.hours.pre.treatment <-
scale(experienced.df$labor.hours.pre.treatment, center = TRUE, scale = FALSE)

experienced.df$centered.labor.hours.per.day.pre.treatment <-
scale(experienced.df$labor.hours.per.day.pre.treatment, center = TRUE, scale = FALSE)

# Check sample size
cat(sprintf("Newcomers: %d\n", nrow(newcomers.df)))
cat(sprintf("Experienced: %d\n", nrow(experienced.df)))

### Newcomer Models

#### Models for Total Labor Hours

In [ ]:
# Total Labor Hours - Newcomers
summary(lm.labor.tweedie.baseline.newcomer <- glmmTMB(
    labor.hours.post.treatment ~ TREAT + (1 | block),
data = newcomers.df,
family = tweedie(link="log")))

summary(lm.labor.tweedie.newcomer <- glmmTMB(
    labor.hours.post.treatment ~ TREAT + centered.labor.hours.pre.treatment + (1 | block),
data = newcomers.df,
family = tweedie(link="log")))

summary(lm.labor.tweedie.int.newcomer <- glmmTMB(
    labor.hours.post.treatment ~ TREAT * centered.labor.hours.pre.treatment + (1 | block),
data = newcomers.df,
family = tweedie(link="log")))

# Model comparison - Newcomers
compare_performance(lm.labor.tweedie.baseline.newcomer,
                    lm.labor.tweedie.newcomer,
                    lm.labor.tweedie.int.newcomer,
                    rank=TRUE)


#### Models for Labor Hours Per Day


In [ ]:
# Labor Hours Per Day - Newcomers
summary(lm.labor.per.day.tweedie.baseline.newcomer <- glmmTMB(
    labor.hours.per.day.post.treatment ~ TREAT + (1 | block),
data = newcomers.df,
family = tweedie(link="log")))

summary(lm.labor.per.day.tweedie.newcomer <- glmmTMB(
    labor.hours.per.day.post.treatment ~ TREAT + centered.labor.hours.per.day.pre.treatment + (1 | block),
data = newcomers.df,
family = tweedie(link="log")))

summary(lm.labor.per.day.tweedie.int.newcomer <- glmmTMB(
    labor.hours.per.day.post.treatment ~ TREAT * centered.labor.hours.per.day.pre.treatment + (1 | block),
data = newcomers.df,
family = tweedie(link="log")))

# Model Comparisons - Newcomers
compare_performance(lm.labor.per.day.tweedie.baseline.newcomer,
                    lm.labor.per.day.tweedie.newcomer,
                    lm.labor.per.day.tweedie.int.newcomer,
                    rank=TRUE)


### Experienced Models

#### Models for Total Labor Hours

In [ ]:
# Total Labor Hours - Experienced
summary(lm.labor.tweedie.baseline.experienced <- glmmTMB(
    labor.hours.post.treatment ~ TREAT + (1 | block),
data = experienced.df,
family = tweedie(link="log")))

summary(lm.labor.tweedie.experienced <- glmmTMB(
    labor.hours.post.treatment ~ TREAT + centered.labor.hours.pre.treatment + (1 | block),
data = experienced.df,
family = tweedie(link="log")))

summary(lm.labor.tweedie.int.experienced <- glmmTMB(
    labor.hours.post.treatment ~ TREAT * centered.labor.hours.pre.treatment + (1 | block),
data = experienced.df,
family = tweedie(link="log")))

# Model comparison - Experienced
compare_performance(lm.labor.tweedie.baseline.experienced,
                    lm.labor.tweedie.experienced,
                    lm.labor.tweedie.int.experienced,
                    rank=TRUE)

#### Models for Labor Hours Per Day


In [ ]:
# Labor Hours Per Day - Experienced
summary(lm.labor.per.day.tweedie.baseline.experienced <- glmmTMB(
    labor.hours.per.day.post.treatment ~ TREAT + (1 | block),
data = experienced.df,
family = tweedie(link="log")))

summary(lm.labor.per.day.tweedie.experienced <- glmmTMB(
    labor.hours.per.day.post.treatment ~ TREAT + centered.labor.hours.per.day.pre.treatment + (1 | block),
data = experienced.df,
family = tweedie(link="log")))

summary(lm.labor.per.day.tweedie.int.experienced <- glmmTMB(
    labor.hours.per.day.post.treatment ~ TREAT * centered.labor.hours.per.day.pre.treatment + (1 | block),
data = experienced.df,
family = tweedie(link="log")))

# Model comparison - Experienced
compare_performance(lm.labor.per.day.tweedie.baseline.experienced,
                    lm.labor.per.day.tweedie.experienced,
                    lm.labor.per.day.tweedie.int.experienced,
                    rank=TRUE)

## Check Tweedie Models Diagnostics for Newcomer Subgroup

### Models for Total Labor Hours

In [ ]:
# Simulate residuals for Newcomers - Total Labor Hours
simRes_tweedie_baseline_newcomer <- simulateResiduals(lm.labor.tweedie.baseline.newcomer)
simRes_tweedie_newcomer <- simulateResiduals(lm.labor.tweedie.newcomer)
simRes_tweedie_int_newcomer <- simulateResiduals(lm.labor.tweedie.int.newcomer)

# Diagnostic plots for Total Labor Hours - Newcomers
plot(simRes_tweedie_baseline_newcomer, main="Newcomers: Total Hours - Baseline")
plot(simRes_tweedie_newcomer, main="Newcomers: Total Hours - Covariate")
plot(simRes_tweedie_int_newcomer, main="Newcomers: Total Hours - Interaction")

# Test assumptions - Baseline
disp_baseline_newc <- testDispersion(simRes_tweedie_baseline_newcomer)
out_baseline_newc <- testOutliers(simRes_tweedie_baseline_newcomer)
zero_baseline_newc <- testZeroInflation(simRes_tweedie_baseline_newcomer)

cat(sprintf("  Dispersion: dispersion = %.3f, p = %.4f %s\n",
            disp_baseline_newc$statistic,
            disp_baseline_newc$p.value,
            ifelse(disp_baseline_newc$p.value < 0.05, "[PROBLEM]", "[OK]")))
cat(sprintf("  Outliers: p = %.4f %s\n",
            out_baseline_newc$p.value,
            ifelse(out_baseline_newc$p.value < 0.05, "[PROBLEM]", "[OK]")))
cat(sprintf("  Zero-Inflation: p = %.4f %s\n\n",
            zero_baseline_newc$p.value,
            ifelse(zero_baseline_newc$p.value < 0.05, "[PROBLEM]", "[OK]")))

# Test assumptions - Covariate
disp_cov_newc <- testDispersion(simRes_tweedie_newcomer)
out_cov_newc <- testOutliers(simRes_tweedie_newcomer)
zero_cov_newc <- testZeroInflation(simRes_tweedie_newcomer)

cat(sprintf("  Dispersion: dispersion = %.3f, p = %.3f\n",
            disp_cov_newc$statistic,
            disp_cov_newc$p.value))
cat(sprintf("  Outliers: p = %.3f\n",
            out_cov_newc$p.value))
cat(sprintf("  Zero-Inflation: p = %.3f\n\n",
            zero_cov_newc$p.value))

# Test assumptions - Interaction
disp_int_newc <- testDispersion(simRes_tweedie_int_newcomer)
out_int_newc <- testOutliers(simRes_tweedie_int_newcomer)
zero_int_newc <- testZeroInflation(simRes_tweedie_int_newcomer)

cat(sprintf("  Dispersion: dispersion = %.3f, p = %.4f %s\n",
            disp_int_newc$statistic,
            disp_int_newc$p.value,
            ifelse(disp_int_newc$p.value < 0.05, "[PROBLEM]", "[OK]")))
cat(sprintf("  Outliers: p = %.4f %s\n",
            out_int_newc$p.value,
            ifelse(out_int_newc$p.value < 0.05, "[PROBLEM]", "[OK]")))
cat(sprintf("  Zero-Inflation: p = %.4f %s\n\n",
            zero_int_newc$p.value,
            ifelse(zero_int_newc$p.value < 0.05, "[PROBLEM]", "[OK]")))


### Models for Labor Hours Per Day

In [ ]:
# Simulate residuals for Newcomers - Labor Hours Per Day
simRes_tweedie_per_day_baseline_newcomer <- simulateResiduals(lm.labor.per.day.tweedie.baseline.newcomer)
simRes_tweedie_per_day_newcomer <- simulateResiduals(lm.labor.per.day.tweedie.newcomer)
simRes_tweedie_per_day_int_newcomer <- simulateResiduals(lm.labor.per.day.tweedie.int.newcomer)

# Diagnostic plots for Labor Hours Per Day - Newcomers
plot(simRes_tweedie_per_day_baseline_newcomer, main="Newcomers: Per Day - Baseline")
plot(simRes_tweedie_per_day_newcomer, main="Newcomers: Per Day - Covariate")
plot(simRes_tweedie_per_day_int_newcomer, main="Newcomers: Per Day - Interaction")

# Test assumptions - Baseline
disp_pd_baseline_newc <- testDispersion(simRes_tweedie_per_day_baseline_newcomer)
out_pd_baseline_newc <- testOutliers(simRes_tweedie_per_day_baseline_newcomer)
zero_pd_baseline_newc <- testZeroInflation(simRes_tweedie_per_day_baseline_newcomer)

cat(sprintf("  Dispersion: dispersion = %.3f, p = %.4f %s\n",
            disp_pd_baseline_newc$statistic,
            disp_pd_baseline_newc$p.value,
            ifelse(disp_pd_baseline_newc$p.value < 0.05, "[PROBLEM]", "[OK]")))
cat(sprintf("  Outliers: p = %.4f %s\n",
            out_pd_baseline_newc$p.value,
            ifelse(out_pd_baseline_newc$p.value < 0.05, "[PROBLEM]", "[OK]")))
cat(sprintf("  Zero-Inflation: p = %.4f %s\n\n",
            zero_pd_baseline_newc$p.value,
            ifelse(zero_pd_baseline_newc$p.value < 0.05, "[PROBLEM]", "[OK]")))

# Test assumptions - Covariate
disp_pd_cov_newc <- testDispersion(simRes_tweedie_per_day_newcomer)
out_pd_cov_newc <- testOutliers(simRes_tweedie_per_day_newcomer)
zero_pd_cov_newc <- testZeroInflation(simRes_tweedie_per_day_newcomer)

cat(sprintf("  Dispersion: dispersion = %.3f, p = %.4f %s\n",
            disp_pd_cov_newc$statistic,
            disp_pd_cov_newc$p.value,
            ifelse(disp_pd_cov_newc$p.value < 0.05, "[PROBLEM]", "[OK]")))
cat(sprintf("  Outliers: p = %.4f %s\n",
            out_pd_cov_newc$p.value,
            ifelse(out_pd_cov_newc$p.value < 0.05, "[PROBLEM]", "[OK]")))
cat(sprintf("  Zero-Inflation: p = %.4f %s\n\n",
            zero_pd_cov_newc$p.value,
            ifelse(zero_pd_cov_newc$p.value < 0.05, "[PROBLEM]", "[OK]")))

# Test assumptions - Interaction
disp_pd_int_newc <- testDispersion(simRes_tweedie_per_day_int_newcomer)
out_pd_int_newc <- testOutliers(simRes_tweedie_per_day_int_newcomer)
zero_pd_int_newc <- testZeroInflation(simRes_tweedie_per_day_int_newcomer)

cat(sprintf("  Dispersion: dispersion = %.3f, p = %.4f %s\n",
            disp_pd_int_newc$statistic,
            disp_pd_int_newc$p.value,
            ifelse(disp_pd_int_newc$p.value < 0.05, "[PROBLEM]", "[OK]")))
cat(sprintf("  Outliers: p = %.4f %s\n",
            out_pd_int_newc$p.value,
            ifelse(out_pd_int_newc$p.value < 0.05, "[PROBLEM]", "[OK]")))
cat(sprintf("  Zero-Inflation: p = %.4f %s\n\n",
            zero_pd_int_newc$p.value,
            ifelse(zero_pd_int_newc$p.value < 0.05, "[PROBLEM]", "[OK]")))

## Check Tweedie Models Diagnostics for Experienced Subgroup

### Models for Total Labor Hours

In [ ]:
# Simulate residuals for Experienced - Total Labor Hours
simRes_tweedie_baseline_exp <- simulateResiduals(lm.labor.tweedie.baseline.experienced)
simRes_tweedie_exp <- simulateResiduals(lm.labor.tweedie.experienced)
simRes_tweedie_int_exp <- simulateResiduals(lm.labor.tweedie.int.experienced)

# Diagnostic plots for Total Labor Hours - Experienced
plot(simRes_tweedie_baseline_exp, main="Experienced: Total Hours - Baseline")
plot(simRes_tweedie_exp, main="Experienced: Total Hours - With Covariate")
plot(simRes_tweedie_int_exp, main="Experienced: Total Hours - Interaction")

# Test assumptions - Baseline
cat("\nModel 1: Baseline\n")
disp_baseline_exp <- testDispersion(simRes_tweedie_baseline_exp)
out_baseline_exp <- testOutliers(simRes_tweedie_baseline_exp)
zero_baseline_exp <- testZeroInflation(simRes_tweedie_baseline_exp)

cat(sprintf("  Dispersion: dispersion = %.3f, p = %.4f %s\n",
            disp_baseline_exp$statistic,
            disp_baseline_exp$p.value,
            ifelse(disp_baseline_exp$p.value < 0.05, "[PROBLEM]", "[OK]")))
cat(sprintf("  Outliers: p = %.4f %s\n",
            out_baseline_exp$p.value,
            ifelse(out_baseline_exp$p.value < 0.05, "[PROBLEM]", "[OK]")))
cat(sprintf("  Zero-Inflation: p = %.4f %s\n\n",
            zero_baseline_exp$p.value,
            ifelse(zero_baseline_exp$p.value < 0.05, "[PROBLEM]", "[OK]")))

# Test assumptions - Covariate
disp_cov_exp <- testDispersion(simRes_tweedie_exp)
out_cov_exp <- testOutliers(simRes_tweedie_exp)
zero_cov_exp <- testZeroInflation(simRes_tweedie_exp)

cat(sprintf("  Dispersion: dispersion = %.3f, p = %.4f %s\n",
            disp_cov_exp$statistic,
            disp_cov_exp$p.value,
            ifelse(disp_cov_exp$p.value < 0.05, "[PROBLEM]", "[OK]")))
cat(sprintf("  Outliers: p = %.4f %s\n",
            out_cov_exp$p.value,
            ifelse(out_cov_exp$p.value < 0.05, "[PROBLEM]", "[OK]")))
cat(sprintf("  Zero-Inflation: p = %.4f %s\n\n",
            zero_cov_exp$p.value,
            ifelse(zero_cov_exp$p.value < 0.05, "[PROBLEM]", "[OK]")))

# Test assumptions - Interaction
disp_int_exp <- testDispersion(simRes_tweedie_int_exp)
out_int_exp <- testOutliers(simRes_tweedie_int_exp)
zero_int_exp <- testZeroInflation(simRes_tweedie_int_exp)

cat(sprintf("  Dispersion: dispersion = %.3f, p = %.4f %s\n",
            disp_int_exp$statistic,
            disp_int_exp$p.value,
            ifelse(disp_int_exp$p.value < 0.05, "[PROBLEM]", "[OK]")))
cat(sprintf("  Outliers: p = %.4f %s\n",
            out_int_exp$p.value,
            ifelse(out_int_exp$p.value < 0.05, "[PROBLEM]", "[OK]")))
cat(sprintf("  Zero-Inflation: p = %.4f %s\n\n",
            zero_int_exp$p.value,
            ifelse(zero_int_exp$p.value < 0.05, "[PROBLEM]", "[OK]")))

### Models for Labor Hours Per Day

In [ ]:
# Simulate residuals for Experienced - Labor Hours Per Day
simRes_tweedie_per_day_baseline_exp <- simulateResiduals(lm.labor.per.day.tweedie.baseline.experienced)
simRes_tweedie_per_day_exp <- simulateResiduals(lm.labor.per.day.tweedie.experienced)
simRes_tweedie_per_day_int_exp <- simulateResiduals(lm.labor.per.day.tweedie.int.experienced)

# Diagnostic plots for Labor Hours Per Day - Experienced
plot(simRes_tweedie_per_day_baseline_exp, main="Experienced: Per Day - Baseline")
plot(simRes_tweedie_per_day_exp, main="Experienced: Per Day - With Covariate")
plot(simRes_tweedie_per_day_int_exp, main="Experienced: Per Day - Interaction")

# Test assumptions - Baseline
disp_pd_baseline_exp <- testDispersion(simRes_tweedie_per_day_baseline_exp)
out_pd_baseline_exp <- testOutliers(simRes_tweedie_per_day_baseline_exp)
zero_pd_baseline_exp <- testZeroInflation(simRes_tweedie_per_day_baseline_exp)

cat(sprintf("  Dispersion: dispersion = %.3f, p = %.4f %s\n",
            disp_pd_baseline_exp$statistic,
            disp_pd_baseline_exp$p.value,
            ifelse(disp_pd_baseline_exp$p.value < 0.05, "[PROBLEM]", "[OK]")))
cat(sprintf("  Outliers: p = %.4f %s\n",
            out_pd_baseline_exp$p.value,
            ifelse(out_pd_baseline_exp$p.value < 0.05, "[PROBLEM]", "[OK]")))
cat(sprintf("  Zero-Inflation: p = %.4f %s\n\n",
            zero_pd_baseline_exp$p.value,
            ifelse(zero_pd_baseline_exp$p.value < 0.05, "[PROBLEM]", "[OK]")))

# Test assumptions - Covariate
disp_pd_cov_exp <- testDispersion(simRes_tweedie_per_day_exp)
out_pd_cov_exp <- testOutliers(simRes_tweedie_per_day_exp)
zero_pd_cov_exp <- testZeroInflation(simRes_tweedie_per_day_exp)

cat(sprintf("  Dispersion: dispersion = %.3f, p = %.4f %s\n",
            disp_pd_cov_exp$statistic,
            disp_pd_cov_exp$p.value,
            ifelse(disp_pd_cov_exp$p.value < 0.05, "[PROBLEM]", "[OK]")))
cat(sprintf("  Outliers: p = %.4f %s\n",
            out_pd_cov_exp$p.value,
            ifelse(out_pd_cov_exp$p.value < 0.05, "[PROBLEM]", "[OK]")))
cat(sprintf("  Zero-Inflation: p = %.4f %s\n\n",
            zero_pd_cov_exp$p.value,
            ifelse(zero_pd_cov_exp$p.value < 0.05, "[PROBLEM]", "[OK]")))

# Test assumptions - Interaction
disp_pd_int_exp <- testDispersion(simRes_tweedie_per_day_int_exp)
out_pd_int_exp <- testOutliers(simRes_tweedie_per_day_int_exp)
zero_pd_int_exp <- testZeroInflation(simRes_tweedie_per_day_int_exp)

cat(sprintf("  Dispersion: dispersion = %.3f, p = %.4f %s\n",
            disp_pd_int_exp$statistic,
            disp_pd_int_exp$p.value,
            ifelse(disp_pd_int_exp$p.value < 0.05, "[PROBLEM]", "[OK]")))
cat(sprintf("  Outliers: p = %.4f %s\n",
            out_pd_int_exp$p.value,
            ifelse(out_pd_int_exp$p.value < 0.05, "[PROBLEM]", "[OK]")))
cat(sprintf("  Zero-Inflation: p = %.4f %s\n\n",
            zero_pd_int_exp$p.value,
            ifelse(zero_pd_int_exp$p.value < 0.05, "[PROBLEM]", "[OK]")))

### Diagnostic Comparison: Newcomers vs. Experienced Subgroups

In [ ]:
cat("TOTAL LABOR HOURS - INTERACTION MODEL:\n")
cat(strrep("-", 80), "\n")
cat(sprintf("%-20s | %15s | %15s\n", "Test", "Newcomers", "Experienced"))
cat(strrep("-", 80), "\n")
cat(sprintf("%-20s | disp=%.2f p=%.3f | disp=%.2f p=%.3f\n",
            "Dispersion",
            disp_int_newc$statistic, disp_int_newc$p.value,
            disp_int_exp$statistic, disp_int_exp$p.value))
cat(sprintf("%-20s | p=%.3f %8s | p=%.3f %8s\n",
            "Outliers",
            out_int_newc$p.value, ifelse(out_int_newc$p.value < 0.05, "[FAIL]", "[OK]"),
            out_int_exp$p.value, ifelse(out_int_exp$p.value < 0.05, "[FAIL]", "[OK]")))
cat(sprintf("%-20s | p=%.3f %8s | p=%.3f %8s\n",
            "Zero-Inflation",
            zero_int_newc$p.value, ifelse(zero_int_newc$p.value < 0.05, "[FAIL]", "[OK]"),
            zero_int_exp$p.value, ifelse(zero_int_exp$p.value < 0.05, "[FAIL]", "[OK]")))

cat("\n\nLABOR HOURS PER DAY - INTERACTION MODEL:\n")
cat(strrep("-", 80), "\n")
cat(sprintf("%-20s | %15s | %15s\n", "Test", "Newcomers", "Experienced"))
cat(strrep("-", 80), "\n")
cat(sprintf("%-20s | disp=%.2f p=%.3f | disp=%.2f p=%.3f\n",
            "Dispersion",
            disp_pd_int_newc$statistic, disp_pd_int_newc$p.value,
            disp_pd_int_exp$statistic, disp_pd_int_exp$p.value))
cat(sprintf("%-20s | p=%.3f %8s | p=%.3f %8s\n",
            "Outliers",
            out_pd_int_newc$p.value, ifelse(out_pd_int_newc$p.value < 0.05, "[FAIL]", "[OK]"),
            out_pd_int_exp$p.value, ifelse(out_pd_int_exp$p.value < 0.05, "[FAIL]", "[OK]")))
cat(sprintf("%-20s | p=%.3f %8s | p=%.3f %8s\n",
            "Zero-Inflation",
            zero_pd_int_newc$p.value, ifelse(zero_pd_int_newc$p.value < 0.05, "[FAIL]", "[OK]"),
            zero_pd_int_exp$p.value, ifelse(zero_pd_int_exp$p.value < 0.05, "[FAIL]", "[OK]")))

cat("\n\nSUMMARY:\n")
all_ok_newc <- all(c(disp_int_newc$p.value, out_int_newc$p.value, zero_int_newc$p.value,
                     disp_pd_int_newc$p.value, out_pd_int_newc$p.value, zero_pd_int_newc$p.value) > 0.05)
all_ok_exp <- all(c(disp_int_exp$p.value, out_int_exp$p.value, zero_int_exp$p.value,
                    disp_pd_int_exp$p.value, out_pd_int_exp$p.value, zero_pd_int_exp$p.value) > 0.05)

cat(sprintf("Newcomers: %s\n",
            ifelse(all_ok_newc, "All diagnostic tests passed ✓", "Some diagnostic issues detected")))
cat(sprintf("Experienced: %s\n",
            ifelse(all_ok_exp, "All diagnostic tests passed ✓", "Some diagnostic issues detected")))

In [ ]:
# Extract coefficients from the three models
extract_model_coefs <- function(model, model_name, subgroup) {
  coef_summary <- coef(summary(model))$cond  # Extract conditional model coefficients

  result <- data.frame(
    model = model_name,
    estimator = "glmmTMB (Tweedie)",
    subgroup = subgroup,
    n.size = nobs(model),
    predictor = rownames(coef_summary),
    Estimate = coef_summary[, "Estimate"],
    Std..Error = coef_summary[, "Std. Error"],
    pvalue = coef_summary[, "Pr(>|z|)"],
    row.names = NULL
  )

  return(result)
}

# Combine results from all three models
labor_per_day_results <- rbind(
  extract_model_coefs(lm.labor.per.day.tweedie.int,
                      "labor hours per day",
                      "all"),
  extract_model_coefs(lm.labor.per.day.tweedie.int.newcomer,
                      "labor hours per day",
                      "newcomer"),
  extract_model_coefs(lm.labor.per.day.tweedie.int.experienced,
                      "labor hours per day",
                      "experienced")
)

# Print the table
print(labor_per_day_results[, c("model", "estimator", "subgroup", "n.size",
                                 "predictor", "Estimate", "Std..Error", "pvalue")])

In [ ]:
save(all.results, labor_per_day_results, file = "ITT_table.RData", version = 2)

### Result Comparisons of Newcomer and Experienced Models

In [ ]:
# Model comparison
# Total Labor Hours - Newcomers
newcomer_total_comp <- compare_performance(lm.labor.tweedie.baseline.newcomer,
                                            lm.labor.tweedie.newcomer,
                                            lm.labor.tweedie.int.newcomer,
                                            rank=TRUE)
print(newcomer_total_comp)

# Total Labor Hours - Experienced
exp_total_comp <- compare_performance(lm.labor.tweedie.baseline.experienced,
                                       lm.labor.tweedie.experienced,
                                       lm.labor.tweedie.int.experienced,
                                       rank=TRUE)
print(exp_total_comp)

# Labor Hours Per Day - Newcomers
newcomer_perday_comp <- compare_performance(lm.labor.per.day.tweedie.baseline.newcomer,
                                             lm.labor.per.day.tweedie.newcomer,
                                             lm.labor.per.day.tweedie.int.newcomer,
                                             rank=TRUE)
print(newcomer_perday_comp)

# Labor Hours Per Day - Experienced
exp_perday_comp <- compare_performance(lm.labor.per.day.tweedie.baseline.experienced,
                                        lm.labor.per.day.tweedie.experienced,
                                        lm.labor.per.day.tweedie.int.experienced,
                                        rank=TRUE)
print(exp_perday_comp)


# ============================================================================
# PART 7: CROSS-GROUP SUMMARY TABLE
# ============================================================================

cat("\n\n", strrep("=", 80), "\n", sep="")
cat("CROSS-GROUP COMPARISON SUMMARY\n")
cat(strrep("=", 80), "\n\n")

cat("TOTAL LABOR HOURS - BEST MODELS:\n")
cat(strrep("-", 80), "\n")
cat(sprintf("%-20s | %-35s | %10s | %10s\n",
            "Group", "Best Model", "AIC Weight", "Diagnostics"))
cat(strrep("-", 80), "\n")

# Newcomers - use interaction model diagnostics (best by AIC)
diag_newc <- ifelse(all(c(disp_int_newc$p.value, out_int_newc$p.value, zero_int_newc$p.value) > 0.05),
                    "PASS", "ISSUES")
cat(sprintf("%-20s | %-35s | %10.4f | %10s\n",
            "Newcomers",
            newcomer_total_comp$Name[1],
            newcomer_total_comp$AIC_wt[1],
            diag_newc))

# Experienced - use interaction model diagnostics (best by AIC)
diag_exp <- ifelse(all(c(disp_int_exp$p.value, out_int_exp$p.value, zero_int_exp$p.value) > 0.05),
                   "PASS", "ISSUES")
cat(sprintf("%-20s | %-35s | %10.4f | %10s\n",
            "Experienced",
            exp_total_comp$Name[1],
            exp_total_comp$AIC_wt[1],
            diag_exp))

cat("\n\nLABOR HOURS PER DAY - BEST MODELS:\n")
cat(strrep("-", 80), "\n")
cat(sprintf("%-20s | %-35s | %10s | %10s\n",
            "Group", "Best Model", "AIC Weight", "Diagnostics"))
cat(strrep("-", 80), "\n")

# Newcomers per day - use interaction model diagnostics
diag_newc_pd <- ifelse(all(c(disp_pd_int_newc$p.value, out_pd_int_newc$p.value, zero_pd_int_newc$p.value) > 0.05),
                       "PASS", "ISSUES")
cat(sprintf("%-20s | %-35s | %10.4f | %10s\n",
            "Newcomers",
            newcomer_perday_comp$Name[1],
            newcomer_perday_comp$AIC_wt[1],
            diag_newc_pd))

# Experienced per day - use interaction model diagnostics
diag_exp_pd <- ifelse(all(c(disp_pd_int_exp$p.value, out_pd_int_exp$p.value, zero_pd_int_exp$p.value) > 0.05),
                      "PASS", "ISSUES")
cat(sprintf("%-20s | %-35s | %10.4f | %10s\n",
            "Experienced",
            exp_perday_comp$Name[1],
            exp_perday_comp$AIC_wt[1],
            diag_exp_pd))


## Final Results Table: Labor Hours Tweedie Models with Interactions


# Generate Sub-Group Analyses w/ Language

In [ ]:
# Extract results from all 6 Tweedie interaction models
# Function to extract model results for TREAT and interaction coefficients
extract_tweedie_results <- function(model, model_name, subgroup){
    # Get summary
    model_summary <- summary(model)

    # Extract coefficients
    coef_df <- as.data.frame(coef(model_summary)$cond)

    # Get TREAT row (main effect)
    treat_row <- coef_df["TREAT1",]

    # Get interaction row - need to identify which interaction term
    # For total hours: TREAT1:centered.labor.hours.pre.treatment
    # For per day: TREAT1:centered.labor.hours.per.day.pre.treatment
    interaction_names <- rownames(coef_df)
    interaction_row_name <- interaction_names[grep("TREAT1:centered", interaction_names)]
    interaction_row <- coef_df[interaction_row_name,]

    # Create result rows - one for main effect, one for interaction
    result_treat <- data.frame(
        model_name=model_name,
        estimator="glmmTMB (Tweedie)",
        subgroup=subgroup,
        n_size=nobs(model),
        predictor="TREAT",
        estimate=treat_row[["Estimate"]],
        std_error=treat_row[["Std. Error"]],
        p_value=treat_row[["Pr(>|z|)"]],
        stringsAsFactors=FALSE
    )

    result_interaction <- data.frame(
        model_name=model_name,
        estimator="glmmTMB (Tweedie)",
        subgroup=subgroup,
        n_size=nobs(model),
        predictor="TREAT × Pre-Treatment Hours",
        estimate=interaction_row[["Estimate"]],
        std_error=interaction_row[["Std. Error"]],
        p_value=interaction_row[["Pr(>|z|)"]],
        stringsAsFactors=FALSE
    )

    # Return both rows combined
    return(rbind(result_treat, result_interaction))
}

# Extract results for all 6 models
cat("Extracting results from Tweedie interaction models...\n\n")

# 1. Total Labor Hours - All Participants
result_1 <- extract_tweedie_results(
    lm.labor.tweedie.int,
    "Total Labor Hours (Interaction)",
    "all"
)

# 2. Total Labor Hours - Newcomers
result_2 <- extract_tweedie_results(
    lm.labor.tweedie.int.newcomer,
    "Total Labor Hours (Interaction)",
    "newcomer"
)

# 3. Total Labor Hours - Experienced
result_3 <- extract_tweedie_results(
    lm.labor.tweedie.int.experienced,
    "Total Labor Hours (Interaction)",
    "experienced"
)

# 4. Labor Hours Per Day - All Participants
result_4 <- extract_tweedie_results(
    lm.labor.per.day.tweedie.int,
    "Labor Hours Per Day (Interaction)",
    "all"
)

# 5. Labor Hours Per Day - Newcomers
result_5 <- extract_tweedie_results(
    lm.labor.per.day.tweedie.int.newcomer,
    "Labor Hours Per Day (Interaction)",
    "newcomer"
)

# 6. Labor Hours Per Day - Experienced
result_6 <- extract_tweedie_results(
    lm.labor.per.day.tweedie.int.experienced,
    "Labor Hours Per Day (Interaction)",
    "experienced"
)

# Combine all results
final_results_table <- rbind(
    result_1,
    result_2,
    result_3,
    result_4,
    result_5,
    result_6
)

# Format the table
final_results_table$estimate <- sprintf("%.4f", final_results_table$estimate)
final_results_table$std_error <- sprintf("%.4f", final_results_table$std_error)
final_results_table$p_value <- sprintf("%.4e", as.numeric(final_results_table$p_value))

# Rename columns
colnames(final_results_table) <- c(
    "Model Name",
    "Estimator",
    "Subgroup",
    "N",
    "Predictor",
    "Estimate",
    "Std. Error",
    "p-value"
)
# Display the table
cat(rep("=", 120), "\n", sep="")
cat("FINAL RESULTS TABLE: Treatment Effects on Labor Hours (Tweedie Models with Interactions)\n")
cat(rep("=", 120), "\n\n", sep="")

print(final_results_table, row.names = FALSE)

cat("\n", rep("=", 120), "\n\n", sep="")

# Also create a publication-ready version with markdown formatting
cat("\n### Table for Manuscript (Markdown Format):\n\n")
cat("| Model Name | Estimator | Subgroup | N | Predictor | Estimate | Std. Error | p-value |\n")
cat("|------------|-----------|----------|---|-----------|----------|------------|---------|\n")
for (i in 1:nrow(final_results_table)) {
    # For every second row (interaction term), blank out the first 4 columns
    if (i %% 2 == 0) {
        cat(sprintf("| %s | %s | %s | %s | %s | %s | %s | %s |\n",
                    "", "", "", "",
                    final_results_table[i, "Predictor"],
                    final_results_table[i, "Estimate"],
                    final_results_table[i, "Std. Error"],
                    final_results_table[i, "p-value"]))
    } else {
        cat(sprintf("| %s | %s | %s | %d | %s | %s | %s | %s |\n",
                    final_results_table[i, "Model Name"],
                    final_results_table[i, "Estimator"],
                    final_results_table[i, "Subgroup"],
                    final_results_table[i, "N"],
                    final_results_table[i, "Predictor"],
                    final_results_table[i, "Estimate"],
                    final_results_table[i, "Std. Error"],
                    final_results_table[i, "p-value"]))
    }
}

# Export to CSV for manuscript
write.csv(final_results_table,
          "labor_hours_tweedie_interaction_results.csv",
          row.names = FALSE)
cat("\n\n✓ Results exported to: labor_hours_tweedie_interaction_results.csv\n")

# Create LaTeX version for papers with multirow
cat("\n\n### LaTeX Table Format:\n\n")
cat("% Note: Requires \\usepackage{multirow} in preamble\n")
cat("\\begin{table}[htbp]\n")
cat("\\centering\n")
cat("\\caption{Treatment Effects on Labor Hours: Tweedie Models with Pre-Treatment Interactions}\n")
cat("\\label{tab:labor_hours_results}\n")
cat("\\begin{tabular}{lllrlrrr}\n")
cat("\\hline\n")
cat("Model & Estimator & Subgroup & N & Predictor & Estimate & SE & \\textit{p} \\\\\n")
cat("\\hline\n")
for (i in 1:nrow(final_results_table)) {
    # For odd rows (first of each pair), use multirow
    if (i %% 2 == 1) {
        cat(sprintf("\\multirow{2}{*}{%s} & \\multirow{2}{*}{%s} & \\multirow{2}{*}{%s} & \\multirow{2}{*}{%s} & %s & %s & %s & %s \\\\\n",
                    final_results_table[i, "Model Name"],
                    gsub("_", "\\\\_", final_results_table[i, "Estimator"]),
                    final_results_table[i, "Subgroup"],
                    final_results_table[i, "N"],
                    gsub("×", "$\\\\times$", final_results_table[i, "Predictor"]),
                    final_results_table[i, "Estimate"],
                    final_results_table[i, "Std. Error"],
                    final_results_table[i, "p-value"]))
    } else {
        # For even rows (second of each pair), just show predictor and stats
        cat(sprintf(" & & & & %s & %s & %s & %s \\\\\n",
                    gsub("×", "$\\\\times$", final_results_table[i, "Predictor"]),
                    final_results_table[i, "Estimate"],
                    final_results_table[i, "Std. Error"],
                    final_results_table[i, "p-value"]))
        # Add hline after each model (every 2 rows)
        if (i < nrow(final_results_table)) {
            cat("\\cline{5-8}\n")
        }
    }
}
cat("\\hline\n")
cat("\\end{tabular}\n")
cat("\\end{table}\n")

In [ ]:
ar.newcomer.results <- df.estimates(subset(participants, lang=="ar" & newcomer), lang="ar", subgroup="newcomer")
de.newcomer.results <- df.estimates(subset(participants, lang=="de" & newcomer), lang="de", subgroup="newcomer")
pl.newcomer.results <- df.estimates(subset(participants, lang=="pl" & newcomer), lang="pl", subgroup="newcomer")



In [ ]:
fa.experienced.results <- df.estimates(subset(participants, lang=="fa" & newcomer!=TRUE), lang="fa", subgroup="experienced")


pl.experienced.results <- df.estimates(subset(participants, lang=="pl" & newcomer!=TRUE), lang="pl", subgroup="experienced")


In [ ]:
ar.newcomer.results[ar.newcomer.results$model!="manipulation check",]$pvalue <-
        p.adjust(ar.newcomer.results[ar.newcomer.results$model!="manipulation check",]$pvalue, method="holm")

de.newcomer.results[de.newcomer.results$model!="manipulation check",]$pvalue <-
        p.adjust(de.newcomer.results[de.newcomer.results$model!="manipulation check",]$pvalue, method="holm")

pl.newcomer.results[pl.newcomer.results$model!="manipulation check",]$pvalue <-
        p.adjust(pl.newcomer.results[pl.newcomer.results$model!="manipulation check",]$pvalue, method="holm")

fa.experienced.results[fa.experienced.results$model!="manipulation check",]$pvalue <-
        p.adjust(fa.experienced.results[fa.experienced.results$model!="manipulation check",]$pvalue, method="holm")

pl.experienced.results[pl.experienced.results$model!="manipulation check",]$pvalue <-
        p.adjust(pl.experienced.results[pl.experienced.results$model!="manipulation check",]$pvalue, method="holm")

In [ ]:

# Create pooled subgroup results for newcomers and experienced contributors
all.newcomer.results <- df.estimates(subset(participants, newcomer), subgroup="newcomer")
all.newcomer.results[all.newcomer.results$model!="manipulation check",]$pvalue <-
        p.adjust(all.newcomer.results[all.newcomer.results$model!="manipulation check",]$pvalue, method="holm")

all.experienced.results <- df.estimates(subset(participants, newcomer!=TRUE), subgroup="experienced")
all.experienced.results[all.experienced.results$model!="manipulation check",]$pvalue <-
        p.adjust(all.experienced.results[all.experienced.results$model!="manipulation check",]$pvalue, method="holm")


all.lang.results <- rbind(ar.newcomer.results, de.newcomer.results,
                          pl.newcomer.results, fa.experienced.results,
                          pl.experienced.results, all.newcomer.results,
                          all.experienced.results, all.results)

all.lang.results

### Adjust Sub-Group Analyses for Multiple Comparisons

In [ ]:
all.results[all.results$model!="manipulation check",]$pvalue <- p.adjust(all.results[all.results$model!="manipulation check",]$pvalue, method="holm")

In [ ]:
subset(all.lang.results, model=="labor hours per day")

# Plot Results

### Plot Effect on Labor Hours

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 5, repr.plot.res = 100)

lab.newc <- "Newcomers"
lab.exp <- "Experienced"

ymax = 3.5
ymin = 0.5
scale.by = 0.5

df1 <- subset(all.lang.results, model=="labor hours per day" & lang == "all" & subgroup == "all")
df2 <- subset(all.lang.results, model=="labor hours per day" & lang!="all" & subgroup == "newcomer")
df3 <- subset(all.lang.results, model=="labor hours per day" & lang != "all" & subgroup == "experienced")

labor.hours.participants.count <- prettyNum(df1$n.size, big.mark=",")
labor.hours.participants.count.new.ar <- prettyNum(subset(df2, lang=="ar")$n.size, big.mark=',')
labor.hours.participants.count.new.de <- prettyNum(subset(df2, lang=="de")$n.size, big.mark=',')
labor.hours.participants.count.new.pl <-prettyNum(subset(df2, lang=="pl")$n.size, big.mark=',')
retention.plot.count.part.exp.fa <- prettyNum(subset(df3, lang=="fa")$n.size, big.mark=',')
retention.plot.count.part.exp.pl <-prettyNum(subset(df3, lang=="pl")$n.size, big.mark=',')
labor.hours.participants.assigned.perc <-round(df1$n.thanked.count/(df1$n.size/2)*100)
labor.hours.participants.assigned.total <-prettyNum(df1$n.size/2, big.mark=",")
labor.hours.participants.assigned.dimest <- prettyNum(df1$pvalue, digits=2)
labor.hours.ylab <- "Treatment Effect on Daily Labor Hours \n(hours ratio)"


labor.hours.plot.caption <- str_interp("This study did not detect an effect of organized thanks on changes in the amount of time that
newcomers contribute to Wikipedia on average.")


all.plot <- ggplot(df1, aes(lang, exp(Estimate))) +
        geom_hline(yintercept = 1, linetype="dashed", color="#999999") +
        geom_errorbar(aes(ymax=exp(CI.Upper), ymin=exp(CI.Lower)),
                      linewidth=1, color=chartpalette[1], width=0.1) +
        geom_point(color=chartpalette[1], size=2.5) +
        ylab(labor.hours.ylab) +
        scale_y_continuous(limits=c(ymin, ymax), breaks=seq(ymin, ymax, by=scale.by)) +
        cat.theme +
        theme(axis.title.x=element_blank(),
              axis.text.x = element_text(size=12, face="bold", color=chartpalette[1]))

newcomer.plot <- ggplot(df2, aes(lang, exp(Estimate))) +
        geom_hline(yintercept = 1, linetype="dashed", color="#999999") +
        geom_errorbar(aes(ymax=exp(CI.Upper), ymin=exp(CI.Lower)),
                      linewidth=1, color=chartpalette[4], width=0.1) +
        geom_point(color=chartpalette[4], size=2.5) +
        annotate(geom="text", x=1.2,y=3.5, label=lab.newc,
                 color=chartpalette[4], fontface=2, size=4, hjust=1) +
        scale_y_continuous(limits=c(ymin, ymax), breaks=seq(ymin, ymax, by=scale.by)) +
        cat.theme +
        theme(axis.text.y = element_blank(),
              axis.title  = element_blank(),
              axis.title.x = element_blank(),
              axis.title.y = element_blank(),
              axis.text.x = element_text(size=12, face="bold", color=chartpalette[4]))


experienced.plot <- ggplot(df3, aes(lang, exp(Estimate))) +
        geom_hline(yintercept = 1, linetype="dashed", color="#999999") +
        geom_errorbar(aes(ymax=exp(CI.Upper), ymin=exp(CI.Lower)),
                      linewidth=1, color=chartpalette[3], width=0.1) +
        geom_point(color=chartpalette[3], size=2.5) +
        annotate(geom="text", x=2.5, y=3.5, label=lab.exp,
         color=chartpalette[3], fontface=2, size=4, hjust=1) +
        scale_y_continuous( limits=c(ymin, ymax), breaks=seq(ymin, ymax, by=scale.by)) +
        cat.theme +
        theme(axis.text.y = element_blank(),
              axis.title  = element_blank(),
              axis.title.x = element_blank(),
              axis.title.y = element_blank(),
              axis.text.x = element_text(size=12, face="bold", color=chartpalette[4]))

labor.hours.plot <- ggarrange(all.plot, newcomer.plot, experienced.plot, ncol=3, nrow=1, widths=c(1.2, 4, 2.5))

ggsave(file.path('figs', 'labor.hours.plot.itt.png'),
       width=6, height=2 , units='in', device="png",
       plot=labor.hours.plot)

labor.hours.plot

### Plot Effect on Two-Week Retention

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 5, repr.plot.res = 100)

ymax = 9
ymin = -3
scale.by = 3

lab.newc <- "Newcomers"
lab.exp <- "Experienced"

df1 <- subset(all.lang.results, model=="retention" & lang == "all" & subgroup == "all")
df2 <- subset(all.lang.results, model=="retention" & lang!="all" & subgroup == "newcomer")
# df3 <- subset(all.newcomer.results, model=="retention")
df4 <- subset(all.lang.results, model=="retention" & lang != "all" & subgroup == "experienced")


retention.plot.est <- prettyNum(df1$Estimate*100, digits=1, format="fg")
retention.plot.count.part <-prettyNum(df1$n.size, big.mark=",")
retention.plot.count.part.new.ar <- prettyNum(subset(df2, lang=="ar")$n.size, big.mark=',')
retention.plot.count.part.new.de <- prettyNum(subset(df2, lang=="de")$n.size, big.mark=',')
retention.plot.count.part.new.pl <- prettyNum(subset(df2, lang=="pl")$n.size, big.mark=',')
retention.plot.count.part.exp.fa <- prettyNum(subset(df4, lang=="fa")$n.size, big.mark=',')
retention.plot.count.part.exp.pl <-prettyNum(subset(df4, lang=="pl")$n.size, big.mark=',')
retention.plot.participants.assigned.perc <-round(df1$n.thanked.count/(df1$n.size/2)*100)
retention.plot.participants.assigned.total <-prettyNum(df1$n.size/2, big.mark=",")
retention.plot.participants.assigned.dimest <- prettyNum(df1$pvalue, digits=2)

retention.plot.ylab <- "Treatment Effect on Two-Week Retention \n(percentage points)"
retention.plot.caption <- str_interp(
    "Organized thanking increases two-week retention of Wikipedia contributors by ${retention.plot.est} percentage
points on average among experienced and newcomer accounts")

all.plot <- ggplot(df1, aes(lang, Estimate*100)) +
        geom_hline(yintercept = 0, linetype="dashed", color="#999999") +
            geom_errorbar(aes(ymax=CI.Upper*100, ymin=CI.Lower*100),
                          linewidth=1, color=chartpalette[1], width=0.1) +
            geom_point(color=chartpalette[1], size=2.5) +
            ylab(retention.plot.ylab) +
            scale_y_continuous(limits=c(ymin, ymax), breaks=seq(ymin, ymax, by=scale.by)) +
            cat.theme +
            theme(axis.title.x=element_blank(),
                  axis.text.x = element_text(size=12, face="bold", color=chartpalette[1]))



lang.plot <- ggplot(df2, aes(lang, Estimate*100)) +
        geom_hline(yintercept = 0, linetype="dashed", color="#999999") +
        geom_errorbar(aes(ymax=CI.Upper*100, ymin=CI.Lower*100),
                      linewidth=1, color=chartpalette[4], width=0.1) +
        geom_point(color=chartpalette[4], size=2.5) +
        annotate(geom="text", x=1.2,y=9, label=lab.newc,
                 color=chartpalette[4], fontface=2, size=4, hjust=1) +
        scale_y_continuous(limits=c(ymin, ymax), breaks=seq(ymin, ymax, by=scale.by)) +
        cat.theme +
        theme(axis.text.y = element_blank(),
              axis.title  = element_blank(),
              axis.title.x = element_blank(),
              axis.title.y = element_blank(),
              axis.text.x = element_text(size=12, face="bold", color=chartpalette[4]))


experienced.plot <- ggplot(df4, aes(lang, Estimate*100)) +
        geom_hline(yintercept = 0, linetype="dashed", color="#999999") +
        geom_errorbar(aes(ymax=CI.Upper*100, ymin=CI.Lower*100),
                      linewidth=1, color=chartpalette[3], width=0.1) +
        geom_point(color=chartpalette[3], size=2.5) +
        annotate(geom="text", x=2.5,y=9, label=lab.exp,
                 color=chartpalette[3], fontface=2, size=4, hjust=1) +
        scale_y_continuous( limits=c(ymin, ymax), breaks=seq(ymin, ymax, by=scale.by)) +
        cat.theme +
        theme(axis.text.y = element_blank(),
              axis.title  = element_blank(),
              axis.title.x = element_blank(),
              axis.title.y = element_blank(),
              axis.text.x = element_text(size=12, face="bold", color=chartpalette[4]))

retention.plot<- ggarrange(all.plot, lang.plot, experienced.plot, ncol=3, nrow=1, widths=c(1.2,4, 2.5))

ggsave(file.path('figs', 'retention.plot.itt.png'),
       width=6, height=2 , device="png", units='in',
       plot=retention.plot)

retention.plot

### Plot effect on rate of thanks

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 5, repr.plot.res = 100)

ymax = 6
ymin = 0
scale.by = 1
lab.newc <- "Newcomers"
lab.exp <- "Experienced"

df1 <- subset(all.lang.results, model=="thanks sent" & lang == "all" & subgroup == "all")
df2 <- subset(all.lang.results, model=="thanks sent" & lang!="all" & subgroup == "newcomer")
df3 <- subset(all.lang.results, model=="thanks sent" & lang != "all" & subgroup == "experienced")

thanks.sent.est <- prettyNum(exp(df1$Estimate), digits=2)
thanks.sent.ci.lower <- prettyNum(exp(df1$CI.Lower), digits=2, format="fg")
thanks.sent.ci.upper <- prettyNum(exp(df1$CI.Upper), digits=2, format="fg")
thanks.sent.part <- prettyNum(df1$n.size, big.mark=",")
thanks.sent.new.ar <- prettyNum(subset(df2, lang=="ar")$n.size, big.mark=',')
thanks.sent.new.de <- prettyNum(subset(df2, lang=="de")$n.size, big.mark=',')
thanks.sent.new.pl <- prettyNum(subset(df2, lang=="pl")$n.size, big.mark=',')
thanks.sent.exp.fa <- prettyNum(subset(df3, lang=="fa")$n.size, big.mark=',')
thanks.sent.exp.pl <- prettyNum(subset(df3, lang=="pl")$n.size, big.mark=',')
thanks.sent.assigned.perc <-round(df1$n.thanked.count/(df1$n.size/2)*100)
thanks.sent.assigned.total <-prettyNum(df1$n.size/2, big.mark=",")
thanks.sent.assigned.dimest <- prettyNum(df1$pvalue, digits=2)



thanks.sent.ylab <- "Treatment Effect on Thanks Sent\n(rate ratio)"
thanks.sent.plot.caption <- str_interp("Organizing to thank Wikipedia volunteers caused them to thank others and increased the thanks they sent
by ${thanks.sent.est} times on average")


all.plot <- ggplot(df1, aes(lang, exp(Estimate))) +
        geom_hline(yintercept = 1, linetype="dashed", color="#999999") +
            geom_errorbar(aes(ymax=exp(CI.Upper), ymin=exp(CI.Lower)),
                          size=1, color=chartpalette[1], width=0.25) +
            geom_point(color=chartpalette[1], size=2.5) +
            ylab(thanks.sent.ylab) +
            scale_y_continuous(limits=c(ymin, ymax), breaks=seq(ymin, ymax, by=scale.by)) +
            cat.theme +
            theme(axis.title.x=element_blank(),
                  axis.text.x = element_text(size=12, face="bold", color=chartpalette[1]))



newcomer.plot <- ggplot(df2, aes(lang, exp(Estimate))) +
        geom_hline(yintercept = 1, linetype="dashed", color="#999999") +
        geom_errorbar(aes(ymax=exp(CI.Upper), ymin=exp(CI.Lower)),
                      size=1, color=chartpalette[4], width=0.1) +
        geom_point(color=chartpalette[4], size=2.5) +
        annotate(geom="text", x=0.5,y=6, label=lab.newc,
                 color=chartpalette[4], fontface=2, size=4, hjust=0) +
        scale_y_continuous(limits=c(ymin, ymax), breaks=seq(ymin, ymax, by=scale.by)) +
        cat.theme +
        theme(axis.text.y = element_blank(),
              axis.title  = element_blank(),
              axis.title.x = element_blank(),
              axis.title.y = element_blank(),
              axis.text.x = element_text(size=12, face="bold", color=chartpalette[4]))



experienced.plot <- ggplot(df3, aes(lang, exp(Estimate))) +
        geom_hline(yintercept = 1, linetype="dashed", color="#999999") +
        geom_errorbar(aes(ymax=exp(CI.Upper), ymin=exp(CI.Lower)),
                      size=1, color=chartpalette[3], width=0.1) +
        geom_point(color=chartpalette[3], size=2.5) +
        annotate(geom="text", x=1.65,y=6, label=lab.exp,
                 color=chartpalette[3], fontface=2, size=4, hjust=0) +
        scale_y_continuous(limits=c(ymin, ymax), breaks=seq(ymin, ymax, by=scale.by)) +
        cat.theme +
        theme(axis.text.y = element_blank(),
              axis.title=element_blank(),
              axis.title.x = element_blank(),
              axis.title.y = element_blank(),
              axis.text.x = element_text(size=12, face="bold", color=chartpalette[3]))

thanks.sent.plot <- ggarrange(all.plot, newcomer.plot, experienced.plot, ncol=3, nrow=1, widths=c(1.2,4, 2.5))

ggsave(file.path('figs', 'thanks.sent.plot.itt.png'),
       width=6, height=6, device="png", units='in',
       plot=thanks.sent.plot)

thanks.sent.plot

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 12, repr.plot.res = 100)

combined.plot <- ggarrange(retention.plot, thanks.sent.plot, labor.hours.plot, ncol=1, nrow=3, heights=c(1,1,1))

ggsave(file.path('figs', 'combined.plot.itt.png'),
       width=8, height=12, device="png", units='in',
       plot=combined.plot)
combined.plot

### Plot distribution of the number of thanks

In [ ]:
library(scales)

# Faceted Histograms (Best for comparison)

# Thanks Received - Faceted
p1 <- ggplot(participants, aes(x = number.thanks.received, fill = TREAT)) +
  geom_histogram(bins = 50, alpha = 0.8) +
  scale_y_log10(labels = comma) +
  scale_fill_manual(values = c("0" = "#E69F00", "1" = "#56B4E9"),
                    labels = c("Control", "Treatment")) +
  facet_wrap(~TREAT, ncol = 1,
             labeller = labeller(TREAT = c("0" = "Control", "1" = "Treatment"))) +
  labs(
    title = "Distribution of Thanks Received by Treatment Group",
    subtitle = "Log count scale for better visibility",
    x = "Number of Thanks Received",
    y = "Count (log scale)",
    fill = "Group"
  ) +
  theme_minimal(base_size = 14) +
  theme(
    legend.position = "none",
    strip.text = element_text(size = 14, face = "bold"),
    strip.background = element_rect(fill = "grey90", color = "grey50"),
    plot.title = element_text(size = 16, face = "bold"),
    plot.subtitle = element_text(size = 12, color = "grey40")
  )

# Thanks Sent - Faceted
p2 <- ggplot(participants, aes(x = thanks.sent, fill = TREAT)) +
  geom_histogram(bins = 50, alpha = 0.8) +
  scale_y_log10(labels = comma) +
  scale_fill_manual(values = c("0" = "#E69F00", "1" = "#56B4E9"),
                    labels = c("Control", "Treatment")) +
  facet_wrap(~TREAT, ncol = 1,
             labeller = labeller(TREAT = c("0" = "Control", "1" = "Treatment"))) +
  labs(
    title = "Distribution of Thanks Sent by Treatment Group",
    subtitle = "Log count scale for better visibility",
    x = "Number of Thanks Sent",
    y = "Count (log scale)",
    fill = "Group"
  ) +
  theme_minimal(base_size = 14) +
  theme(
    legend.position = "none",
    strip.text = element_text(size = 14, face = "bold"),
    strip.background = element_rect(fill = "grey90", color = "grey50"),
    plot.title = element_text(size = 16, face = "bold"),
    plot.subtitle = element_text(size = 12, color = "grey40")
  )

# Display faceted plots
print(p1)
print(p2)


# ============================================================================
# Option 2: Overlaid with Semi-Transparent Bars (Alternative view)
# ============================================================================

p3 <- ggplot(participants, aes(x = number.thanks.received, fill = TREAT)) +
  geom_histogram(bins = 50, alpha = 0.6, position = "identity") +
  scale_y_log10(labels = comma, breaks = c(1, 10, 100, 1000, 10000)) +
  scale_fill_manual(values = c("0" = "#E69F00", "1" = "#56B4E9"),
                    labels = c("Control", "Treatment")) +
  labs(
    title = "Distribution of Thanks Received (Overlaid)",
    subtitle = "Log count scale, semi-transparent for overlap visualization",
    x = "Number of Thanks Received",
    y = "Count (log scale)",
    fill = "Group"
  ) +
  theme_minimal(base_size = 14) +
  theme(
    legend.position = "bottom",
    legend.title = element_text(face = "bold"),
    plot.title = element_text(size = 16, face = "bold"),
    plot.subtitle = element_text(size = 12, color = "grey40"),
    panel.grid.minor = element_blank()
  )

p4 <- ggplot(participants, aes(x = thanks.sent, fill = TREAT)) +
  geom_histogram(bins = 50, alpha = 0.6, position = "identity") +
  scale_y_log10(labels = comma, breaks = c(1, 10, 100, 1000, 10000)) +
  scale_fill_manual(values = c("0" = "#E69F00", "1" = "#56B4E9"),
                    labels = c("Control", "Treatment")) +
  labs(
    title = "Distribution of Thanks Sent (Overlaid)",
    subtitle = "Log count scale, semi-transparent for overlap visualization",
    x = "Number of Thanks Sent",
    y = "Count (log scale)",
    fill = "Group"
  ) +
  theme_minimal(base_size = 14) +
  theme(
    legend.position = "bottom",
    legend.title = element_text(face = "bold"),
    plot.title = element_text(size = 16, face = "bold"),
    plot.subtitle = element_text(size = 12, color = "grey40"),
    panel.grid.minor = element_blank()
  )

# Display overlaid plots
print(p3)
print(p4)


# ============================================================================
# Option 3: Side-by-side with Better Binning
# ============================================================================
# Alternative: Frequency polygon (lines instead of bars)
p5 <- ggplot(participants, aes(x = number.thanks.received, color = TREAT)) +
  geom_freqpoly(bins = 40, linewidth = 1.2) +
  scale_y_log10(labels = comma,
                breaks = c(1, 10, 100, 1000, 10000)) +
  scale_color_manual(values = c("0" = "#E69F00", "1" = "#56B4E9"),
                     labels = c("Control", "Treatment")) +
  labs(
    title = "Distribution of Thanks Received",
    subtitle = "Log count scale - frequency polygon",
    x = "Number of Thanks Received",
    y = "Count (log scale)",
    color = "Group"
  ) +
  theme_minimal(base_size = 14) +
  theme(
    legend.position = "bottom",
    legend.title = element_text(face = "bold"),
    legend.text = element_text(size = 12),
    plot.title = element_text(size = 16, face = "bold"),
    plot.subtitle = element_text(size = 11, color = "grey40"),
    panel.grid.minor = element_blank()
  )

p6 <- ggplot(participants, aes(x = thanks.sent, color = TREAT)) +
  geom_freqpoly(bins = 40, linewidth = 1.2) +
  scale_y_log10(labels = comma,
                breaks = c(1, 10, 100, 1000, 10000)) +
  scale_color_manual(values = c("0" = "#E69F00", "1" = "#56B4E9"),
                     labels = c("Control", "Treatment")) +
  labs(
    title = "Distribution of Thanks Sent",
    subtitle = "Log count scale - frequency polygon",
    x = "Number of Thanks Sent",
    y = "Count (log scale)",
    color = "Group"
  ) +
  theme_minimal(base_size = 14) +
  theme(
    legend.position = "bottom",
    legend.title = element_text(face = "bold"),
    legend.text = element_text(size = 12),
    plot.title = element_text(size = 16, face = "bold"),
    plot.subtitle = element_text(size = 11, color = "grey40"),
    panel.grid.minor = element_blank()
  )
# Display side-by-side plots
print(p5)
print(p6)


combined <- grid.arrange(p1, p2, ncol = 2)


In [ ]:
# Create a clean newcomer label
participants$newcomer_label <- ifelse(participants$newcomer == "TRUE",
                                      "Newcomers",
                                      "Experienced")

# Summary statistics
cat("=== Summary: Thanks Received by Newcomer Status ===\n")
participants %>%
  group_by(newcomer_label) %>%
  summarise(
    n = n(),
    n_received_any = sum(number.thanks.received > 0),
    prop_received_any = mean(number.thanks.received > 0),
    mean = mean(number.thanks.received, na.rm = TRUE),
    median = median(number.thanks.received, na.rm = TRUE),
    sd = sd(number.thanks.received, na.rm = TRUE),
    min = min(number.thanks.received, na.rm = TRUE),
    max = max(number.thanks.received, na.rm = TRUE)
  )

cat("\n=== Summary: Thanks Sent by Newcomer Status ===\n")
participants %>%
  group_by(newcomer_label) %>%
  summarise(
    n = n(),
    n_sent_any = sum(thanks.sent > 0),
    prop_sent_any = mean(thanks.sent > 0),
    mean = mean(thanks.sent, na.rm = TRUE),
    median = median(thanks.sent, na.rm = TRUE),
    sd = sd(thanks.sent, na.rm = TRUE),
    min = min(thanks.sent, na.rm = TRUE),
    max = max(thanks.sent, na.rm = TRUE)
  )

# ============================================================================
# Faceted Histograms (Best for comparison)
# ============================================================================

# Thanks Received - Faceted by Newcomer Status
p1 <- ggplot(participants, aes(x = number.thanks.received, fill = newcomer_label)) +
  geom_histogram(bins = 50, alpha = 0.85, color = "white", linewidth = 0.2) +
  scale_y_log10(labels = comma) +
  scale_fill_manual(values = c("Newcomers" = "#E69F00", "Experienced" = "#56B4E9")) +
  facet_wrap(~newcomer_label, ncol = 1) +
  labs(
    title = "Distribution of Thanks Received by Editor Experience",
    subtitle = "Log count scale - Faceted for clarity",
    x = "Number of Thanks Received",
    y = "Count (log scale)"
  ) +
  theme_minimal(base_size = 14) +
  theme(
    legend.position = "none",
    strip.text = element_text(size = 14, face = "bold"),
    strip.background = element_rect(fill = "grey90", color = "grey50"),
    plot.title = element_text(size = 16, face = "bold"),
    plot.subtitle = element_text(size = 12, color = "grey40"),
    panel.grid.minor = element_blank()
  )

# Thanks Sent - Faceted by Newcomer Status
p2 <- ggplot(participants, aes(x = thanks.sent, fill = newcomer_label)) +
  geom_histogram(bins = 50, alpha = 0.85, color = "white", linewidth = 0.2) +
  scale_y_log10(labels = comma) +
  scale_fill_manual(values = c("Newcomers" = "#E69F00", "Experienced" = "#56B4E9")) +
  facet_wrap(~newcomer_label, ncol = 1) +
  labs(
    title = "Distribution of Thanks Sent by Editor Experience",
    subtitle = "Log count scale - Faceted for clarity",
    x = "Number of Thanks Sent",
    y = "Count (log scale)"
  ) +
  theme_minimal(base_size = 14) +
  theme(
    legend.position = "none",
    strip.text = element_text(size = 14, face = "bold"),
    strip.background = element_rect(fill = "grey90", color = "grey50"),
    plot.title = element_text(size = 16, face = "bold"),
    plot.subtitle = element_text(size = 12, color = "grey40"),
    panel.grid.minor = element_blank()
  )

# Display plots
print(p1)
print(p2)

# ============================================================================
# Frequency Polygons (Alternative visualization)
# ============================================================================

p3 <- ggplot(participants, aes(x = number.thanks.received, color = newcomer_label)) +
  geom_freqpoly(bins = 40, linewidth = 1.2) +
  scale_y_log10(labels = comma, breaks = c(1, 10, 100, 1000, 10000)) +
  scale_color_manual(values = c("Newcomers" = "#E69F00", "Experienced" = "#56B4E9")) +
  labs(
    title = "Distribution of Thanks Received by Editor Experience",
    subtitle = "Log count scale - Frequency polygon",
    x = "Number of Thanks Received",
    y = "Count (log scale)",
    color = "Experience"
  ) +
  theme_minimal(base_size = 14) +
  theme(
    legend.position = "bottom",
    legend.title = element_text(face = "bold"),
    legend.text = element_text(size = 12),
    plot.title = element_text(size = 16, face = "bold"),
    plot.subtitle = element_text(size = 11, color = "grey40"),
    panel.grid.minor = element_blank()
  )

p4 <- ggplot(participants, aes(x = thanks.sent, color = newcomer_label)) +
  geom_freqpoly(bins = 40, linewidth = 1.2) +
  scale_y_log10(labels = comma, breaks = c(1, 10, 100, 1000, 10000)) +
  scale_color_manual(values = c("Newcomers" = "#E69F00", "Experienced" = "#56B4E9")) +
  labs(
    title = "Distribution of Thanks Sent by Editor Experience",
    subtitle = "Log count scale - Frequency polygon",
    x = "Number of Thanks Sent",
    y = "Count (log scale)",
    color = "Experience"
  ) +
  theme_minimal(base_size = 14) +
  theme(
    legend.position = "bottom",
    legend.title = element_text(face = "bold"),
    legend.text = element_text(size = 12),
    plot.title = element_text(size = 16, face = "bold"),
    plot.subtitle = element_text(size = 11, color = "grey40"),
    panel.grid.minor = element_blank()
  )

print(p3)
print(p4)

# ============================================================================
# Overlapping Histograms (Alternative visualization)
# ============================================================================

p_overlap_received <- ggplot(participants, aes(x = number.thanks.received, fill = newcomer_label)) +
  geom_histogram(bins = 50, alpha = 0.5, position = "identity", color = "white", linewidth = 0.2) +
  scale_y_log10(labels = comma) +
  scale_fill_manual(values = c("Newcomers" = "#E69F00", "Experienced" = "#56B4E9")) +
  labs(
    title = "Distribution of Thanks Received by Editor Experience",
    subtitle = "Log count scale - Overlapping for direct comparison",
    x = "Number of Thanks Received",
    y = "Count (log scale)",
    fill = "Experience"
  ) +
  theme_minimal(base_size = 14) +
  theme(
    legend.position = "bottom",
    legend.title = element_text(face = "bold"),
    legend.text = element_text(size = 12),
    plot.title = element_text(size = 16, face = "bold"),
    plot.subtitle = element_text(size = 12, color = "grey40"),
    panel.grid.minor = element_blank()
  )

p_overlap_sent <- ggplot(participants, aes(x = thanks.sent, fill = newcomer_label)) +
  geom_histogram(bins = 50, alpha = 0.5, position = "identity", color = "white", linewidth = 0.2) +
  scale_y_log10(labels = comma) +
  scale_fill_manual(values = c("Newcomers" = "#E69F00", "Experienced" = "#56B4E9")) +
  labs(
    title = "Distribution of Thanks Sent by Editor Experience",
    subtitle = "Log count scale - Overlapping for direct comparison",
    x = "Number of Thanks Sent",
    y = "Count (log scale)",
    fill = "Experience"
  ) +
  theme_minimal(base_size = 14) +
  theme(
    legend.position = "bottom",
    legend.title = element_text(face = "bold"),
    legend.text = element_text(size = 12),
    plot.title = element_text(size = 16, face = "bold"),
    plot.subtitle = element_text(size = 12, color = "grey40"),
    panel.grid.minor = element_blank()
  )

print(p_overlap_received)
print(p_overlap_sent)

# ============================================================================
# Box plots for comparison
# ============================================================================

p5 <- ggplot(participants, aes(x = newcomer_label, y = number.thanks.received,
                                fill = newcomer_label)) +
  geom_boxplot(alpha = 0.7) +
  scale_fill_manual(values = c("Newcomers" = "#E69F00", "Experienced" = "#56B4E9")) +
  labs(
    title = "Thanks Received by Editor Experience",
    x = "Experience Level",
    y = "Number of Thanks Received"
  ) +
  theme_minimal(base_size = 14) +
  theme(
    legend.position = "none",
    plot.title = element_text(size = 16, face = "bold")
  )

p6 <- ggplot(participants, aes(x = newcomer_label, y = thanks.sent,
                                fill = newcomer_label)) +
  geom_boxplot(alpha = 0.7) +
  scale_fill_manual(values = c("Newcomers" = "#E69F00", "Experienced" = "#56B4E9")) +
  labs(
    title = "Thanks Sent by Editor Experience",
    x = "Experience Level",
    y = "Number of Thanks Sent"
  ) +
  theme_minimal(base_size = 14) +
  theme(
    legend.position = "none",
    plot.title = element_text(size = 16, face = "bold")
  )

print(p5)
print(p6)

# Combined view
grid.arrange(p1, p2, ncol = 2)

In [ ]:
summary(participants$number.thanks.received)

In [ ]:
# Prepare the table
table_data <- all.results %>%
  select(
    Outcome = model,
    Estimator = estimator,
    Group = subgroup,
    N = n.size,
    ATE = Estimate,
    `Std Error` = Std..Error,
    `P value` = pvalue
  ) %>%
  mutate(
    ATE = sprintf("%.4f", ATE),
    `Std Error` = sprintf("%.4f", `Std Error`),
    `P value` = sprintf("%.4f", `P value`),
    N = as.integer(N)
  )

# Generate LaTeX
cat(kable(table_data, format = "latex", booktabs = TRUE))

# Complier Average Causal Effect
We use two stage least squares regression to estimate the complier average causal effect.

In [ ]:
# ## this method generates complier average treatment effect estimates
# ## for a given subgroup of the study. It assumes that rates of actual
# ## thanking by volunteers will vary by language and newcomer status
#
# df.cace.estimates <- function(df, lang="all", subgroup="all"){
#
#
#     treat.prob.formula <- "complier.app.any.reason ~ TREAT"
#     if(lang=="all"){
#         treat.prob.formula <- paste(treat.prob.formula, " + lang")
#     }
#
#     if(subgroup=="all"){
#         treat.prob.formula <- paste(treat.prob.formula, " + newcomer")
#     }
#
#     treat.prob.m <- lm(treat.prob.formula, data=df)
#
#     df$iv <- predict(treat.prob.m, df)
#
#     df.compliers <- subset(df, complier)
#
#
#     thanks.model         <- glm.nb(thanks.sent ~ iv, data=df)
#     m.thanks.sent        <- coef(summary(thanks.model))
#     m.thanks.sent        <- data.frame(cbind(m.thanks.sent, confint(thanks.model)))
#     names(m.thanks.sent)[4] <- "pvalue"
#     names(m.thanks.sent)[5] <- "CI.Lower"
#     names(m.thanks.sent)[6] <- "CI.Upper"
#     m.thanks.sent$DF       <- thanks.model$df.residual
#     m.thanks.sent$model    <- "thanks sent"
#     m.thanks.sent$t.value  <- NA
#     m.thanks.sent$n.size   <- nrow(df)
#     m.thanks.sent$n.thanked.count <- nrow(subset(df, complier.app.any.reason))
#
#     retention.model        <- lm(two.week.retention ~ iv, data=df)
#     m.retention            <- data.frame(coef(summary(retention.model)))
#     m.retention            <- cbind(m.retention, confint(retention.model))
#     names(m.retention)[4]  <- "pvalue"
#     names(m.retention)[5]  <- "CI.Lower"
#     names(m.retention)[6]  <- "CI.Upper"
#     m.retention$DF         <- retention.model$df
#     m.retention$model      <- "retention"
#     m.retention$z.value    <- NA
#     m.retention$n.size     <- nrow(df)
#     m.retention$n.thanked.count <- nrow(subset(df, complier.app.any.reason))
#
#     labor.hour.model       <- lm(labor.hours.per.day.diff ~ iv, data=df)
#     m.labor.hours          <- data.frame(coef(summary(labor.hour.model)))
#     m.labor.hours          <- cbind(m.labor.hours, confint(labor.hour.model))
#     names(m.labor.hours)[4] <- "pvalue"
#     names(m.labor.hours)[5] <- "CI.Lower"
#     names(m.labor.hours)[6] <- "CI.Upper"
#     m.labor.hours$DF       <- labor.hour.model$df
#     m.labor.hours$model    <- "labor hours per day"
#     m.labor.hours$z.value  <- NA
#     m.labor.hours$n.size   <- nrow(df)
#     m.labor.hours$n.thanked.count <- nrow(subset(df, complier.app.any.reason))
#
#     manip.check.model      <- lm(remembered.thanks<=3 ~ iv, data=df.compliers)
#     m.manip.check          <- data.frame(coef(summary(manip.check.model)))
#     m.manip.check          <- cbind(m.manip.check, confint(manip.check.model))
#     names(m.manip.check)[4] <- "pvalue"
#     names(m.manip.check)[5] <- "CI.Lower"
#     names(m.manip.check)[6] <- "CI.Upper"
#     m.manip.check$DF       <- manip.check.model$df
#     m.manip.check$model    <- "manipulation check"
#     m.manip.check$z.value  <- NA
#     m.manip.check$n.size   <- nrow(df.compliers)
#     m.manip.check$n.thanked.count <-  nrow(subset(df.compliers, complier.app.any.reason))
#
#     m.thanks.sent$estimator        <- "glm.nb"
#     m.retention$estimator          <- "2sls"
#     m.labor.hours$estimator        <- "2sls"
#     m.manip.check$estimator        <- "2sls"
#
#     result.df <- rbind(m.retention, m.labor.hours, m.thanks.sent,
#                        m.manip.check)
#
#     #result.df <- result.df[row.names(result.df)!="(Intercept)",]
#
#     result.df$lang     <- lang
#     result.df$subgroup <- subgroup
#
#     result.df
# }

#### Estimate CACE Results

In [ ]:
# ar.newcomer.cace.results <- df.cace.estimates(subset(participants, lang=="ar" & newcomer), lang="ar", subgroup="newcomer")
# de.newcomer.cace.results <- df.cace.estimates(subset(participants, lang=="de" & newcomer), lang="de", subgroup="newcomer")
# pl.newcomer.cace.results <- df.cace.estimates(subset(participants, lang=="pl" & newcomer), lang="pl", subgroup="newcomer")
#
# fa.experienced.cace.results <- df.cace.estimates(subset(participants, lang=="fa" & newcomer!=TRUE), lang="fa", subgroup="experienced")
# fa.experienced.cace.results <- subset(fa.experienced.cace.results,model!="labor hours per day")
#
# pl.experienced.cace.results <- df.cace.estimates(subset(participants, lang=="pl" & newcomer!=TRUE), lang="pl", subgroup="experienced")
# pl.experienced.cace.results <- subset(pl.experienced.cace.results,model!="labor hours per day")
#
# all.newcomer.cace.df <- df.cace.estimates(subset(participants, newcomer), subgroup="newcomer")
# all.experienced.cace.df <- df.cace.estimates(subset(participants, newcomer!=TRUE), subgroup="experienced")
# all.experienced.cace.df <- subset(all.experienced.cace.df,model!="labor hours per day")
#
# ## create a dataframe of the main results table
# all.cace.df <- df.cace.estimates(participants)
#
# ## create a dataframe of all results, for plotting charts
# all.cace.models.df <- rbind(
#     ar.newcomer.cace.results,
#     de.newcomer.cace.results,
#     pl.newcomer.cace.results,
#     fa.experienced.cace.results,
#     pl.experienced.cace.results,
#     all.newcomer.cace.df,
#     all.experienced.cace.df,
#     all.cace.df
#     )

In [ ]:
## In case readers want to view subgroups of CACE models

In [ ]:
# subset(all.cace.models.df, model =="retention" & 
#                           str_detect(row.names(all.cace.models.df), "Intercept")!=TRUE)

In [ ]:
# subset(all.cace.models.df, model =="labor hours per day" & 
#                           str_detect(row.names(all.cace.models.df), "Intercept")!=TRUE)

In [ ]:
# subset(all.cace.models.df, model =="thanks sent" & 
#                           str_detect(row.names(all.cace.models.df), "Intercept")!=TRUE)

## Adjust p Values for CACE
Here, we adjust the CACE p values for all of the pre-registered analyses, as well as the CACE analyses.

In [ ]:
# ## append all p values from the paper into a single list
# ## then adjust the p-values
# ## then return the first three, so they can be added to the adjustments
#
# all.cace.iv.df <-  cbind(factor = rownames(all.cace.df), all.cace.df)
# all.cace.iv.df <- subset(all.cace.iv.df, substring(all.cace.iv.df$factor, 0,2)=="iv")
#
# all.cace.iv.df[all.cace.iv.df$model!="manipulation check",]$pvalue <-
#         p.adjust(c(all.cace.iv.df[all.cace.iv.df$model!="manipulation check",]$pvalue,all.results.pvalues))[1:3]
# all.cace.iv.df

# Intent to Treat Effect


## Plot ITT

In [ ]:
dotsize = 1.5
errorsize = 1
all.lang.results <-  cbind(factor = rownames(all.lang.results), all.lang.results)

In [ ]:
all.lang.results

### Plot IV Labor Hours

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 5, repr.plot.res = 100)

# For overall newcomer ITT effect (pooled across languages)
lh1 <- subset(all.newcomer.results, model=="labor hours per day" & lang == "all")

# For language-specific newcomer effects
lh2 <- subset(all.lang.results, model=="labor hours per day" & subgroup=="newcomer" & lang!="all")

In [ ]:
lh1

In [ ]:
labor.hours.iv.participants.count <- prettyNum(lh1$n.size, big.mark=",")
labor.hours.iv.participants.count.new.ar <- prettyNum(subset(lh2, lang=="ar")$n.size, big.mark=',')
labor.hours.iv.participants.count.new.de <- prettyNum(subset(lh2, lang=="de")$n.size, big.mark=',')
labor.hours.iv.participants.count.new.pl <-prettyNum(subset(lh2, lang=="pl")$n.size, big.mark=',')
labor.hours.iv.participants.assigned.perc <-round(lh1$n.thanked.count/(lh1$n.size/2)*100)
labor.hours.iv.participants.assigned.total <-prettyNum(lh1$n.size/2, big.mark=",")
labor.hours.iv.participants.assigned.dimest <- prettyNum(lh1$pvalue, digits=2)
labor.hours.iv.ylab <- "Labor Hours Per Day"


labor.hours.iv.plot.caption <- str_interp("This study did not detect an effect of organized thanks on changes in the amount of time that
newcomers contribute to Wikipedia on average.")

In [ ]:
lh.all.plot <- ggplot(lh1, aes(lang, Estimate)) +
        geom_hline(yintercept = 0, linetype="dashed", color="#999999") +
        geom_errorbar(aes(ymax=CI.Upper, ymin=CI.Lower),
                      size=errorsize, color=chartpalette[1], width=0.1) +
        geom_point(color=chartpalette[1], size=dotsize) +
        ylab(labor.hours.iv.ylab) +
        cat.theme +
        theme(axis.title.x=element_blank(),
              axis.text.x = element_text(size=12, face="bold", color=chartpalette[1])) +
        ylim(-0.06, 0.1)

lh.all.plot

In [ ]:
lh2

In [ ]:
lh.lang.plot <- ggplot(lh2, aes(lang, Estimate)) +
        geom_hline(yintercept = 0, linetype="dashed", color="#999999") +
        geom_errorbar(aes(ymax=CI.Upper, ymin=CI.Lower),
                      size=errorsize, color=chartpalette[3], width=0.1) +
        geom_point(color=chartpalette[3], size=dotsize) +
        cat.theme +
        theme(axis.text.y = element_blank(),
              axis.title  = element_blank(),
              axis.title.x=element_blank(),
              axis.text.x = element_text(size=12, face="bold", color=chartpalette[3])) +
         ylab("") +
         xlab("") +
        ylim(-0.06, 0.1)

lh.lang.plot

In [ ]:
labor.hours.iv.plot <- ggarrange(lh.all.plot, lh.lang.plot, ncol=2, nrow=1, widths=c(1.5,3.2))

ggsave(file.path('figs', 'labor.hours.iv.plot.pdf'),
       width=6, height=2 , units='in', device="pdf",
       plot=labor.hours.iv.plot)

labor.hours.iv.plot

## keep for compatibility with previous versions of ggplot2 
# labor.hours.iv.plot <- annotate_figure(labor.hours.iv.plot,
#                         bottom=text_grob(labor.hours.iv.plot.caption, 
#                                          hjust=0, x=0, size=10, 
#                                          color=chartpalette[4])) +
#                     ggsave(file.path('figs', 'labor.hours.iv.plot.png'),
#                           width=6, height=3.375 , units='in')
#labor.hours.iv.plot

### Plot IV Two Week Retention

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 5, repr.plot.res = 100)

ymax = 0.5
ymin = -0.2

In [ ]:
#all.lang.results
twr1 <- subset(all.lang.results, model=="retention" & lang == "all" & subgroup == "all")
twr2 <- subset(all.lang.results, model=="retention" & lang!="all" & subgroup == "newcomer")
# twr3 <- subset(all.newcomer.results, model=="retention")
twr4 <- subset(all.lang.results, model=="retention" & lang != "all" & subgroup == "experienced")

In [ ]:
retention.iv.plot.est <- prettyNum(twr1$Estimate*100, digits=1, format="fg")
retention.iv.plot.count.part <-prettyNum(twr1$n.size, big.mark=",")
retention.iv.plot.count.part.new.ar <- prettyNum(subset(twr2, lang=="ar")$n.size, big.mark=',')
retention.iv.plot.count.part.new.de <- prettyNum(subset(twr2, lang=="de")$n.size, big.mark=',')
retention.iv.plot.count.part.new.pl <- prettyNum(subset(twr2, lang=="pl")$n.size, big.mark=',')
retention.iv.plot.count.part.exp.fa <- prettyNum(subset(twr4, lang=="fa")$n.size, big.mark=',')
retention.iv.plot.count.part.exp.pl <-prettyNum(subset(twr4, lang=="pl")$n.size, big.mark=',')
retention.iv.plot.participants.assigned.perc <-round(twr1$n.thanked.count/(twr1$n.size/2)*100)
retention.iv.plot.participants.assigned.total <-prettyNum(twr1$n.size/2, big.mark=",")
retention.iv.plot.participants.assigned.dimest <- prettyNum(twr1$pvalue, digits=2)

retention.iv.plot.ylab <- "Effect on Two\nWeek Retention"
retention.iv.plot.caption <- str_interp(
    "Organized thanking increases two-week retention of Wikipedia contributors by ${retention.iv.plot.est} percentage
points on average among experienced and newcomer accounts")

In [ ]:
twr1

In [ ]:
twr.all.plot <- ggplot(twr1, aes(lang, Estimate)) +
        geom_hline(yintercept = 0, linetype="dashed", color="#999999") +
            geom_errorbar(aes(ymax=CI.Upper, ymin=CI.Lower),
                          size=errorsize, color=chartpalette[1], width=0.2) +
            geom_point(color=chartpalette[1], size=dotsize) +
            ylab(retention.iv.plot.ylab) +
            scale_y_continuous(limits=c(ymin, ymax), breaks=seq(ymin, ymax, by=0.1)) +
            cat.theme +
            theme(axis.title.x=element_blank(),
                  axis.text.x = element_text(size=12, face="bold", color=chartpalette[1]))

twr.all.plot

twr.lang.plot <- ggplot(twr2, aes(lang, Estimate)) +
        geom_hline(yintercept = 0, linetype="dashed", color="#999999") +
        geom_errorbar(aes(ymax=CI.Upper, ymin=CI.Lower),
                      size=errorsize, color=chartpalette[4], width=0.1) +
        geom_point(color=chartpalette[4], size=dotsize) +
        annotate(geom="text", x=3.5,y=0.49, label=lab.newc,
                 color=chartpalette[4], fontface=2, size=4, hjust=1) +
        scale_y_continuous(limits=c(ymin, ymax), breaks=seq(ymin, ymax, by=0.1)) +
        cat.theme +
        theme(axis.text.y = element_blank(),
              axis.title.y = element_blank(),
              axis.title  = element_blank(),
              axis.title.x = element_blank(),
              axis.text.x = element_text(size=12, face="bold", color=chartpalette[4]))
twr.lang.plot

In [ ]:
twr4

In [ ]:
twr.experienced.plot <- ggplot(twr4, aes(lang, Estimate)) +
        geom_hline(yintercept = 0, linetype="dashed", color="#999999") +
        geom_errorbar(aes(ymax=CI.Upper, ymin=CI.Lower),
                      size=errorsize, color=chartpalette[3], width=0.1) +
        geom_point(color=chartpalette[3], size=dotsize) +
        annotate(geom="text", x=2.5,y=0.49, label=lab.exp,
                 color=chartpalette[3], fontface=2, size=4, hjust=1) +
        scale_y_continuous( limits=c(ymin, ymax), breaks=seq(ymin, ymax, by=0.1)) +
        cat.theme +
        theme(axis.text.y = element_blank(),
              axis.title=element_blank(),
              axis.title.x = element_blank(),
              axis.title.y = element_blank(),
              axis.text.x = element_text(size=12, face="bold", color=chartpalette[3]))

#twr.experienced.plot
retention.iv.plot <- ggarrange(twr.all.plot, twr.experienced.plot, twr.lang.plot, ncol=3, nrow=1, widths=c(2,3, 4))

ggsave(file.path('figs', 'retention.iv.plot.pdf'),
       width=6, height=2 , device="pdf", units='in',
       plot=retention.iv.plot)

## keep for compatibility with previous versions of ggplot2
# retention.iv.plot <- annotate_figure(retention.plot,
#                         bottom=text_grob(retention.iv.plot.caption, 
#                                          hjust=0, x=0, size=10, 
#                                          color=chartpalette[4]))+
#                     ggsave(file.path('figs', 'retention.iv.plot.png'),
#                           width=6, height=3.375 , units='in')
retention.iv.plot

### Plot IV Effect on Rate of Thanks

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 5, repr.plot.res = 100)

ymax = 15
ymin = -10
scale.by = 5
rt1 <- subset(all.lang.results, model=="thanks sent" & lang == "all" & subgroup == "all")
rt2 <- subset(all.lang.results, model=="thanks sent" & lang!="all" & subgroup == "newcomer")
rt3 <- subset(all.lang.results, model=="thanks sent" & lang != "all" & subgroup == "experienced")

thanks.sent.iv.est <- prettyNum(exp(rt1$Estimate), digits=2)
thanks.sent.iv.ci.lower <- prettyNum(exp(rt1$CI.Lower), digits=2, format="fg")
thanks.sent.iv.ci.upper <- prettyNum(exp(rt1$CI.Upper), digits=2, format="fg")
thanks.sent.iv.part <- prettyNum(rt1$n.size, big.mark=",")
thanks.sent.iv.new.ar <- prettyNum(subset(rt2, lang=="ar")$n.size, big.mark=',')
thanks.sent.iv.new.de <- prettyNum(subset(rt2, lang=="de")$n.size, big.mark=',')
thanks.sent.iv.new.pl <- prettyNum(subset(rt2, lang=="pl")$n.size, big.mark=',')
thanks.sent.iv.exp.fa <- prettyNum(subset(rt3, lang=="fa")$n.size, big.mark=',')
thanks.sent.iv.exp.pl <- prettyNum(subset(rt3, lang=="pl")$n.size, big.mark=',')
thanks.sent.iv.assigned.perc <-round(rt1$n.thanked.count/(rt1$n.size/2)*100)
thanks.sent.iv.assigned.total <-prettyNum(rt1$n.size/2, big.mark=",")
thanks.sent.iv.assigned.dimest <- prettyNum(rt1$pvalue, digits=2)


thanks.sent.iv.ylab <- "Negative binomial estimate\nof thanks sent"
thanks.sent.iv.plot.caption <- str_interp("Organizing to thank Wikipedia volunteers caused them to thank others and increased the thanks they sent
by ${thanks.sent.iv.est} times on average")
#thanks.sent.iv.plot.caption
#rt3

In [ ]:
rt.all.plot <- ggplot(rt1, aes(lang, Estimate)) +
        geom_hline(yintercept = 0, linetype="dashed", color="#999999") +
            geom_errorbar(aes(ymax=CI.Upper, ymin=CI.Lower),
                          size=1, color=chartpalette[1], width=0.5) +
            geom_point(color=chartpalette[1], size=dotsize) +
            ylab(thanks.sent.iv.ylab) +
            scale_y_continuous(limits=c(ymin, ymax), breaks=seq(ymin, ymax, by=scale.by)) +
            cat.theme +
            theme(axis.title.x=element_blank(),
                  axis.text.x = element_text(size=12, face="bold", color=chartpalette[1]))



rt.newcomer.plot <- ggplot(rt2, aes(lang, Estimate)) +
        geom_hline(yintercept = 0, linetype="dashed", color="#999999") +
        geom_errorbar(aes(ymax=CI.Upper, ymin=CI.Lower),
                      size=1, color=chartpalette[4], width=0.1) +
        geom_point(color=chartpalette[4], size=dotsize) +
        annotate(geom="text", x=2.6,y=ymax-0.25, label=lab.newc,
                 color=chartpalette[4], fontface=2, size=4, hjust=0) +
        scale_y_continuous(limits=c(ymin, ymax), breaks=seq(ymin, ymax, by=scale.by)) +
        cat.theme +
        theme(axis.text.y = element_blank(),
              axis.title  = element_blank(),
              axis.title.x = element_blank(),
              axis.title.y = element_blank(),
              axis.text.x = element_text(size=12, face="bold", color=chartpalette[4]))


rt.experienced.plot <- ggplot(rt3, aes(lang, Estimate)) +
        geom_hline(yintercept = 0, linetype="dashed", color="#999999") +
        geom_errorbar(aes(ymax=CI.Upper, ymin=CI.Lower),
                      size=1, color=chartpalette[3], width=0.1) +
        geom_point(color=chartpalette[3], size=dotsize) +
        annotate(geom="text", x=1.4,y=ymax-0.25, label=lab.exp,
                 color=chartpalette[3], fontface=2, size=4, hjust=0) +
        scale_y_continuous( limits=c(ymin, ymax), breaks=seq(ymin, ymax, by=scale.by)) +
        cat.theme +
        theme(axis.text.y = element_blank(),
              axis.title=element_blank(),
              axis.title.x = element_blank(),
              axis.title.y = element_blank(),
              axis.text.x = element_text(size=12, face="bold", color=chartpalette[3]))

# rt.newcomer.plot
# rt.all.plot
#rt.experienced.plot

In [ ]:
thanks.sent.iv.plot <- ggarrange(rt.all.plot,  rt.experienced.plot, rt.newcomer.plot, ncol=3, nrow=1, widths=c(1.2,2.5, 4))

ggsave(file.path('figs', 'thanks.sent.iv.plot.pdf'),
    width=6, height=2, device="pdf", units='in',
    plot = thanks.sent.iv.plot)

# thanks.sent.iv.plot <- annotate_figure(thanks.sent.iv.plot,
#                         bottom=text_grob(thanks.sent.iv.plot.caption, 
#                                          hjust=0, x=0, size=10, 
#                                          color=chartpalette[4]))+
#                     ggsave(file.path('figs', 'thanks.sent.iv.plot.png'),
#                           width=6, height=4 , units='in')
thanks.sent.iv.plot

# Analyze Participant Thanks

In [ ]:
colnames(participant.thanks)

In [ ]:
print(paste("according to the thanks dataset, ", nrow(subset(participant.thanks, identifiable.thanks.sent>0 & condition=="1")), "treatment group accounts sent thanks"))

print(paste("according to the earlier participant dataset, ", nrow(subset(participants, thanks.sent>0 & TREAT==1)), "treatment group accounts sent thanks"))

In [ ]:
print(paste("according to the thanks dataset", sum(subset(participant.thanks, condition=='1')$identifiable.thanks.sent), "thanks were sent by treatment group accounts"))

print(paste("according to the participant dataset", sum(subset(participants, TREAT==1)$thanks.sent), "thanks were sent by treatment group accounts"))

In [ ]:
colnames(participant.thanks)

In [ ]:
#participant.thanks
participant.thanks$any.reciprocal.thanks <- as.integer(participant.thanks$reciprocal.thanks.sent>0)
participant.thanks$any.nonreciprocal.thanks <- as.integer(participant.thanks$nonreciprocal.thanks.sent>0)

In [ ]:
treat.reciprocal.thanks.df <- aggregate( . ~ lang,
                                        data=participant.thanks[c('reciprocal.thanks.sent',
                                                                  'nonreciprocal.thanks.sent',
                                                                  'any.reciprocal.thanks',
                                                                  'any.nonreciprocal.thanks',
                                                                  'lang')],
                                        FUN=sum)
treat.reciprocal.thanks.df

In [ ]:
## percentage of thanks sent to people who did not thank them
prettyNum(sum(treat.reciprocal.thanks.df$nonreciprocal.thanks.sent)/ (sum(treat.reciprocal.thanks.df$nonreciprocal.thanks.sent) + sum(treat.reciprocal.thanks.df$reciprocal.thanks.sent))*100, digits=3)

# Balance Table of Pre-treat activity

In [ ]:
# ============================================================================
# Supporting Information Tables
# Run AFTER the pre-dropped results cells in the notebook
# Requires: all.participants (with $dropped), all.results, all.results.pre.drop
# ============================================================================

library(tidyverse)
library(knitr)

# --- Smart formatters ---

# Number: 2 dp, but show 4 dp if rounds to 0.00
fmt <- function(x) {
  sapply(x, function(v) {
    if (is.na(v)) return("--")
    if (v == 0) return("0")
    if (abs(v) >= 0.005) return(sprintf("%.2f", v))
    if (abs(v) >= 0.00005) return(sprintf("%.4f", v))
    return(sprintf("%.2e", v))
  })
}

# p-value: smart thresholds
fmt_p <- function(p) {
  sapply(p, function(v) {
    if (is.na(v)) return("--")
    if (v < 0.0001) return("$<$0.0001")
    if (v < 0.001)  return(sprintf("%.4f", v))
    if (v < 0.01)   return(sprintf("%.3f", v))
    return(sprintf("%.2f", v))
  })
}

# Percentage for LaTeX
fmt_pct <- function(x) {
  sapply(x, function(v) {
    if (v == 0) return("0\\%")
    if (abs(v) < 0.005) return(sprintf("%.4f\\%%", v))
    sprintf("%.2f\\%%", v)
  })
}

# Cohen's d: standardized mean difference (pooled SD)
cohen_d <- function(x1, x2) {
  m1 <- mean(x1, na.rm=TRUE); m2 <- mean(x2, na.rm=TRUE)
  s1 <- sd(x1, na.rm=TRUE);   s2 <- sd(x2, na.rm=TRUE)
  s_pool <- sqrt((s1^2 + s2^2) / 2)
  if (s_pool == 0) return(0)
  (m2 - m1) / s_pool
}

# Cramér's V for multi-category chi-squared
cramer_v <- function(chi, n, k) {
  sqrt(chi$statistic / (n * (k - 1)))
}

# ============================================================================
# PANEL A: Previous Experience Categories by Treatment
# ============================================================================

cat("=================================================================\n")
cat("BALANCE TABLE: Treatment vs Control Pre-Treatment Characteristics\n")
cat("=================================================================\n\n")

cat("--- Panel A: Previous Experience Categories ---\n\n")

# Experience bin distribution by treatment arm
exp_by_treat <- participants %>%
  mutate(
    Experience = case_when(
      prev.experience.assignment.days == 0 ~ "Newcomer (0 days)",
      prev.experience.assignment.days == 1 ~ "1 day",
      prev.experience.assignment.days == 4 ~ "4 days",
      prev.experience.assignment.days == 30 ~ "30 days",
      prev.experience.assignment.days == 90 ~ "90 days",
      prev.experience.assignment.days == 180 ~ "180 days",
      prev.experience.assignment.days == 365 ~ "365 days",
      TRUE ~ paste0(prev.experience.assignment.days, " days")
    ),
    Treatment = ifelse(TREAT == 1, "Treatment", "Control")
  )

# Counts and proportions
exp_table <- exp_by_treat %>%
  group_by(Experience, Treatment) %>%
  summarise(N = n(), .groups = "drop") %>%
  pivot_wider(names_from = Treatment, values_from = N, values_fill = 0) %>%
  mutate(
    Control_Pct = round(Control / sum(Control) * 100, 1),
    Treatment_Pct = round(Treatment / sum(Treatment) * 100, 1)
  ) %>%
  arrange(match(Experience, c("Newcomer (0 days)", "1 day", "4 days", "30 days",
                               "90 days", "180 days", "365 days")))

cat("Experience Category     | Control (N)  | Control (%) | Treatment (N) | Treatment (%)\n")
cat("------------------------|--------------|-------------|---------------|---------------\n")
for (i in 1:nrow(exp_table)) {
  cat(sprintf("%-24s| %12d | %10.2f%% | %13d | %12.2f%%\n",
              exp_table$Experience[i],
              exp_table$Control[i], exp_table$Control_Pct[i],
              exp_table$Treatment[i], exp_table$Treatment_Pct[i]))
}

# Newcomer vs Experienced
newcomer_table <- exp_by_treat %>%
  mutate(Group = ifelse(newcomer, "Newcomer", "Experienced")) %>%
  group_by(Group, Treatment) %>%
  summarise(N = n(), .groups = "drop") %>%
  pivot_wider(names_from = Treatment, values_from = N, values_fill = 0)

cat("\nNewcomer/Experienced    | Control (N)  | Treatment (N)\n")
cat("------------------------|--------------|---------------\n")
for (i in 1:nrow(newcomer_table)) {
  cat(sprintf("%-24s| %12d | %13d\n",
              newcomer_table$Group[i],
              newcomer_table$Control[i],
              newcomer_table$Treatment[i]))
}

# Language distribution
lang_table <- exp_by_treat %>%
  group_by(lang, Treatment) %>%
  summarise(N = n(), .groups = "drop") %>%
  pivot_wider(names_from = Treatment, values_from = N, values_fill = 0) %>%
  mutate(
    Control_Pct = round(Control / sum(Control) * 100, 1),
    Treatment_Pct = round(Treatment / sum(Treatment) * 100, 1)
  )

cat("\nLanguage                | Control (N)  | Control (%) | Treatment (N) | Treatment (%)\n")
cat("------------------------|--------------|-------------|---------------|---------------\n")
for (i in 1:nrow(lang_table)) {
  cat(sprintf("%-24s| %12d | %10.2f%% | %13d | %12.2f%%\n",
              lang_table$lang[i],
              lang_table$Control[i], lang_table$Control_Pct[i],
              lang_table$Treatment[i], lang_table$Treatment_Pct[i]))
}

# ============================================================================
# PANEL B: Pre-Study Labor Hours by Treatment
# ============================================================================

cat("\n\n--- Panel B: Pre-Study Labor Hours ---\n\n")

labor_stats <- participants %>%
  mutate(Treatment = ifelse(TREAT == 1, "Treatment", "Control")) %>%
  group_by(Treatment) %>%
  summarise(
    N = n(),
    Mean_labor_hrs = mean(labor.hours.pre.treatment, na.rm = TRUE),
    SD_labor_hrs = sd(labor.hours.pre.treatment, na.rm = TRUE),
    Median_labor_hrs = median(labor.hours.pre.treatment, na.rm = TRUE),
    IQR_labor_hrs = IQR(labor.hours.pre.treatment, na.rm = TRUE),
    Mean_labor_hrs_per_day = mean(labor.hours.per.day.pre.treatment, na.rm = TRUE),
    SD_labor_hrs_per_day = sd(labor.hours.per.day.pre.treatment, na.rm = TRUE),
    Median_labor_hrs_per_day = median(labor.hours.per.day.pre.treatment, na.rm = TRUE),
    .groups = "drop"
  )

cat("                          | Control             | Treatment\n")
cat("--------------------------|---------------------|---------------------\n")
cat(sprintf("N                         | %19d | %d\n", labor_stats$N[1], labor_stats$N[2]))
cat(sprintf("Labor hours (total)       |                     |\n"))
cat(sprintf("  Mean (SD)               | %8.2f (%7.2f) | %8.2f (%7.2f)\n",
            labor_stats$Mean_labor_hrs[1], labor_stats$SD_labor_hrs[1],
            labor_stats$Mean_labor_hrs[2], labor_stats$SD_labor_hrs[2]))
cat(sprintf("  Median [IQR]            | %8s [%7s] | %8s [%7s]\n",
            fmt(labor_stats$Median_labor_hrs[1]), fmt(labor_stats$IQR_labor_hrs[1]),
            fmt(labor_stats$Median_labor_hrs[2]), fmt(labor_stats$IQR_labor_hrs[2])))
cat(sprintf("Labor hours per day       |                     |\n"))
cat(sprintf("  Mean (SD)               | %8.2f (%7.2f) | %8.2f (%7.2f)\n",
            labor_stats$Mean_labor_hrs_per_day[1], labor_stats$SD_labor_hrs_per_day[1],
            labor_stats$Mean_labor_hrs_per_day[2], labor_stats$SD_labor_hrs_per_day[2]))
cat(sprintf("  Median                  | %19s | %s\n",
            fmt(labor_stats$Median_labor_hrs_per_day[1]),
            fmt(labor_stats$Median_labor_hrs_per_day[2])))

# By newcomer status
cat("\nPre-study labor hours per day by experience:\n\n")
labor_by_exp <- participants %>%
  mutate(
    Treatment = ifelse(TREAT == 1, "Treatment", "Control"),
    Group = ifelse(newcomer, "Newcomer", "Experienced")
  ) %>%
  group_by(Group, Treatment) %>%
  summarise(
    N = n(),
    Mean = mean(labor.hours.per.day.pre.treatment, na.rm = TRUE),
    SD = sd(labor.hours.per.day.pre.treatment, na.rm = TRUE),
    Median = median(labor.hours.per.day.pre.treatment, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  pivot_wider(names_from = Treatment,
              values_from = c(N, Mean, SD, Median))

cat(sprintf("%-14s| Control: N=%5d, Mean=%s (SD=%s), Median=%s\n",
            "Newcomer",
            labor_by_exp$N_Control[labor_by_exp$Group=="Newcomer"],
            fmt(labor_by_exp$Mean_Control[labor_by_exp$Group=="Newcomer"]),
            fmt(labor_by_exp$SD_Control[labor_by_exp$Group=="Newcomer"]),
            fmt(labor_by_exp$Median_Control[labor_by_exp$Group=="Newcomer"])))
cat(sprintf("%-14s| Treat:   N=%5d, Mean=%s (SD=%s), Median=%s\n",
            "",
            labor_by_exp$N_Treatment[labor_by_exp$Group=="Newcomer"],
            fmt(labor_by_exp$Mean_Treatment[labor_by_exp$Group=="Newcomer"]),
            fmt(labor_by_exp$SD_Treatment[labor_by_exp$Group=="Newcomer"]),
            fmt(labor_by_exp$Median_Treatment[labor_by_exp$Group=="Newcomer"])))
cat(sprintf("%-14s| Control: N=%5d, Mean=%s (SD=%s), Median=%s\n",
            "Experienced",
            labor_by_exp$N_Control[labor_by_exp$Group=="Experienced"],
            fmt(labor_by_exp$Mean_Control[labor_by_exp$Group=="Experienced"]),
            fmt(labor_by_exp$SD_Control[labor_by_exp$Group=="Experienced"]),
            fmt(labor_by_exp$Median_Control[labor_by_exp$Group=="Experienced"])))
cat(sprintf("%-14s| Treat:   N=%5d, Mean=%s (SD=%s), Median=%s\n",
            "",
            labor_by_exp$N_Treatment[labor_by_exp$Group=="Experienced"],
            fmt(labor_by_exp$Mean_Treatment[labor_by_exp$Group=="Experienced"]),
            fmt(labor_by_exp$SD_Treatment[labor_by_exp$Group=="Experienced"]),
            fmt(labor_by_exp$Median_Treatment[labor_by_exp$Group=="Experienced"])))

# ============================================================================
# PANEL C: Thanks Sent and Received by Treatment
# ============================================================================

cat("\n\n--- Panel C: Pre-Study Thanks Sent & Received ---\n\n")

thanks_stats <- participants %>%
  mutate(Treatment = ifelse(TREAT == 1, "Treatment", "Control")) %>%
  group_by(Treatment) %>%
  summarise(
    N = n(),
    Mean_thanks_sent = mean(thanks.sent.pre.treatment, na.rm = TRUE),
    SD_thanks_sent = sd(thanks.sent.pre.treatment, na.rm = TRUE),
    Median_thanks_sent = median(thanks.sent.pre.treatment, na.rm = TRUE),
    Pct_any_thanks_sent = mean(thanks.sent.pre.treatment > 0, na.rm = TRUE) * 100,
    Mean_thanks_received = mean(number.thanks.received, na.rm = TRUE),
    SD_thanks_received = sd(number.thanks.received, na.rm = TRUE),
    Median_thanks_received = median(number.thanks.received, na.rm = TRUE),
    Pct_any_thanks_received = mean(number.thanks.received > 0, na.rm = TRUE) * 100,
    .groups = "drop"
  )

cat("                          | Control             | Treatment\n")
cat("--------------------------|---------------------|---------------------\n")
cat(sprintf("Thanks Sent (pre-study)  |                     |\n"))
cat(sprintf("  Mean (SD)               | %8s (%7s) | %8s (%7s)\n",
            fmt(thanks_stats$Mean_thanks_sent[1]), fmt(thanks_stats$SD_thanks_sent[1]),
            fmt(thanks_stats$Mean_thanks_sent[2]), fmt(thanks_stats$SD_thanks_sent[2])))
cat(sprintf("  Median                  | %19s | %s\n",
            fmt(thanks_stats$Median_thanks_sent[1]), fmt(thanks_stats$Median_thanks_sent[2])))
cat(sprintf("  %% with any thanks sent  | %18s | %s\n",
            fmt_pct(thanks_stats$Pct_any_thanks_sent[1]), fmt_pct(thanks_stats$Pct_any_thanks_sent[2])))
cat(sprintf("Thanks Received (pre-study)|                    |\n"))
cat(sprintf("  Mean (SD)               | %8s (%7s) | %8s (%7s)\n",
            fmt(thanks_stats$Mean_thanks_received[1]), fmt(thanks_stats$SD_thanks_received[1]),
            fmt(thanks_stats$Mean_thanks_received[2]), fmt(thanks_stats$SD_thanks_received[2])))
cat(sprintf("  Median                  | %19s | %s\n",
            fmt(thanks_stats$Median_thanks_received[1]), fmt(thanks_stats$Median_thanks_received[2])))
cat(sprintf("  %% with any received     | %18s | %s\n",
            fmt_pct(thanks_stats$Pct_any_thanks_received[1]), fmt_pct(thanks_stats$Pct_any_thanks_received[2])))

# By newcomer status
cat("\nThanks by experience group:\n")
thanks_by_exp <- participants %>%
  mutate(
    Treatment = ifelse(TREAT == 1, "Treatment", "Control"),
    Group = ifelse(newcomer, "Newcomer", "Experienced")
  ) %>%
  group_by(Group, Treatment) %>%
  summarise(
    N = n(),
    Mean_sent = mean(thanks.sent.pre.treatment, na.rm = TRUE),
    SD_sent = sd(thanks.sent.pre.treatment, na.rm = TRUE),
    Mean_received = mean(number.thanks.received, na.rm = TRUE),
    SD_received = sd(number.thanks.received, na.rm = TRUE),
    .groups = "drop"
  )

cat("\n  Group        | Treatment | N     | Thanks Sent Mean(SD)      | Thanks Received Mean(SD)\n")
cat("  -------------|-----------|-------|---------------------------|-------------------------\n")
for (i in 1:nrow(thanks_by_exp)) {
  cat(sprintf("  %-13s| %-9s | %5d | %8s (%7s)      | %8s (%7s)\n",
              thanks_by_exp$Group[i], thanks_by_exp$Treatment[i], thanks_by_exp$N[i],
              fmt(thanks_by_exp$Mean_sent[i]), fmt(thanks_by_exp$SD_sent[i]),
              fmt(thanks_by_exp$Mean_received[i]), fmt(thanks_by_exp$SD_received[i])))
}

# ============================================================================
# PANEL D: Balance Tests (t-tests / chi-squared)
# ============================================================================

cat("\n\n--- Panel D: Balance Tests (Treatment vs Control) ---\n\n")

# Continuous variables: Wilcoxon/t-tests
control <- subset(participants, TREAT == 0)
treatment <- subset(participants, TREAT == 1)

cat("Variable                          | Test        | Statistic   | p-value    | Std. Diff\n")
cat("----------------------------------|-------------|-------------|------------|----------\n")

n_total <- nrow(participants)

# Labor hours pre-treatment
t1 <- wilcox.test(control$labor.hours.pre.treatment,
                   treatment$labor.hours.pre.treatment)
d1 <- cohen_d(control$labor.hours.pre.treatment, treatment$labor.hours.pre.treatment)
cat(sprintf("Labor hours (pre-treatment)       | Wilcoxon    | %11.2f | %10s | %s\n",
            t1$statistic, fmt_p(t1$p.value), fmt(d1)))

# Labor hours per day pre-treatment
t2 <- wilcox.test(control$labor.hours.per.day.pre.treatment,
                   treatment$labor.hours.per.day.pre.treatment)
d2 <- cohen_d(control$labor.hours.per.day.pre.treatment, treatment$labor.hours.per.day.pre.treatment)
cat(sprintf("Labor hours/day (pre-treatment)   | Wilcoxon    | %11.2f | %10s | %s\n",
            t2$statistic, fmt_p(t2$p.value), fmt(d2)))

# Thanks sent (pre-study)
t3 <- wilcox.test(control$thanks.sent.pre.treatment,
                   treatment$thanks.sent.pre.treatment)
d3 <- cohen_d(control$thanks.sent.pre.treatment, treatment$thanks.sent.pre.treatment)
cat(sprintf("Thanks sent (pre-study)           | Wilcoxon    | %11.2f | %10s | %s\n",
            t3$statistic, fmt_p(t3$p.value), fmt(d3)))

# Thanks received (pre-study)
t4 <- wilcox.test(control$number.thanks.received,
                   treatment$number.thanks.received)
d4 <- cohen_d(control$number.thanks.received, treatment$number.thanks.received)
cat(sprintf("Thanks received (pre-study)       | Wilcoxon    | %11.2f | %10s | %s\n",
            t4$statistic, fmt_p(t4$p.value), fmt(d4)))

# Experience category (chi-squared)
chi1 <- chisq.test(table(participants$prev.experience.assignment.days, participants$TREAT))
v1 <- cramer_v(chi1, n_total, length(unique(participants$prev.experience.assignment.days)))
cat(sprintf("Experience category               | Chi-squared | %11.2f | %10s | %s (V)\n",
            chi1$statistic, fmt_p(chi1$p.value), fmt(v1)))

# Newcomer status
chi2 <- chisq.test(table(participants$newcomer, participants$TREAT))
d5 <- cohen_d(as.numeric(control$newcomer), as.numeric(treatment$newcomer))
cat(sprintf("Newcomer status                   | Chi-squared | %11.2f | %10s | %s\n",
            chi2$statistic, fmt_p(chi2$p.value), fmt(d5)))

# Language
chi3 <- chisq.test(table(participants$lang, participants$TREAT))
v3 <- cramer_v(chi3, n_total, length(unique(participants$lang)))
cat(sprintf("Language                          | Chi-squared | %11.2f | %10s | %s (V)\n",
            chi3$statistic, fmt_p(chi3$p.value), fmt(v3)))

cat("\n(Std. Diff = Cohen's d for continuous/binary, Cramér's V for multi-category.")
cat("\n |d| < 0.05 is considered negligible imbalance.)\n")

# ============================================================================
# COMBINED SUMMARY TABLE (kable output)
# ============================================================================

cat("\n\n=== COMPACT BALANCE TABLE ===\n\n")

summary_balance <- participants %>%
  mutate(Treatment = ifelse(TREAT == 1, "Treatment", "Control")) %>%
  group_by(Treatment) %>%
  summarise(
    N = n(),
    `Newcomer (%)` = sprintf("%.2f%%", mean(newcomer) * 100),
    `Experienced (%)` = sprintf("%.2f%%", mean(!newcomer) * 100),
    `Pre-study labor hrs/day: Mean (SD)` = sprintf("%.2f (%.2f)",
      mean(labor.hours.per.day.pre.treatment, na.rm=TRUE),
      sd(labor.hours.per.day.pre.treatment, na.rm=TRUE)),
    `Pre-study labor hrs/day: Median` = sprintf("%.2f",
      median(labor.hours.per.day.pre.treatment, na.rm=TRUE)),
    `Pre-study thanks sent: Mean (SD)` = sprintf("%.2f (%.2f)",
      mean(thanks.sent.pre.treatment, na.rm=TRUE), sd(thanks.sent.pre.treatment, na.rm=TRUE)),
    `Pre-study thanks received: Mean (SD)` = sprintf("%.2f (%.2f)",
      mean(number.thanks.received, na.rm=TRUE), sd(number.thanks.received, na.rm=TRUE)),
    `Arabic (%)` = sprintf("%.2f%%", mean(lang == "ar") * 100),
    `German (%)` = sprintf("%.2f%%", mean(lang == "de") * 100),
    `Persian (%)` = sprintf("%.2f%%", mean(lang == "fa") * 100),
    `Polish (%)` = sprintf("%.2f%%", mean(lang == "pl") * 100),
    .groups = "drop"
  ) %>%
  t() %>%
  as.data.frame()

colnames(summary_balance) <- summary_balance[1,]
summary_balance <- summary_balance[-1,]

print(kable(summary_balance, format = "simple", align = "rr"))

cat("\n=== END BALANCE TABLE ===\n")

# ============================================================================
# LATEX BALANCE TABLE
# ============================================================================

# Compute all values needed for the LaTeX table
ctrl <- subset(participants, TREAT == 0)
trt  <- subset(participants, TREAT == 1)

# Helper: format mean (sd)
fmt_mean_sd <- function(x, digits=2) {
  m <- mean(x, na.rm=TRUE)
  s <- sd(x, na.rm=TRUE)
  paste0(fmt(m), " (", fmt(s), ")")
}
fmt_n <- function(x) formatC(length(x), format="d", big.mark=",")

# Experience bins
exp_levels <- sort(unique(participants$prev.experience.assignment.days))
exp_labels <- ifelse(exp_levels == 0, "Newcomer (0 days)", paste0(exp_levels, " days"))

# Build rows
latex_lines <- c()
latex_lines <- c(latex_lines, "\\begin{table*}[t]")
latex_lines <- c(latex_lines, "\\centering")
latex_lines <- c(latex_lines, paste0("\\caption{Balance table comparing pre-treatment characteristics of treatment and control groups. ",
  "P-values are from Wilcoxon rank-sum tests (continuous variables) or chi-squared tests (categorical variables). ",
  "Std.~Diff.~is Cohen's $d$ for continuous and binary variables or Cram\\'er's $V$ for multi-category variables. ",
  "With $N > 15{,}000$, small p-values can arise from trivially small differences; all $|d| < 0.05$ confirm negligible imbalance.}"))
latex_lines <- c(latex_lines, "\\label{table:balance}")
latex_lines <- c(latex_lines, "\\begin{tabular}{lrrrll}")
latex_lines <- c(latex_lines, "\\toprule")
latex_lines <- c(latex_lines, " & \\textbf{Control} & \\textbf{Treatment} & \\textbf{Total} & \\textbf{p-value} & \\textbf{Std.~Diff.} \\\\")
latex_lines <- c(latex_lines, sprintf(" & (N = %s) & (N = %s) & (N = %s) & & \\\\",
  formatC(nrow(ctrl), big.mark=","),
  formatC(nrow(trt), big.mark=","),
  formatC(nrow(participants), big.mark=",")))
latex_lines <- c(latex_lines, "\\midrule")

# --- Experience categories ---
latex_lines <- c(latex_lines, "\\multicolumn{6}{l}{\\textit{Previous experience category}} \\\\")
for (i in seq_along(exp_levels)) {
  lvl <- exp_levels[i]
  n_ctrl <- sum(ctrl$prev.experience.assignment.days == lvl)
  n_trt  <- sum(trt$prev.experience.assignment.days == lvl)
  n_tot  <- n_ctrl + n_trt
  pct_ctrl <- sprintf("%.2f\\%%", n_ctrl / nrow(ctrl) * 100)
  pct_trt  <- sprintf("%.2f\\%%", n_trt / nrow(trt) * 100)
  pct_tot  <- sprintf("%.2f\\%%", n_tot / nrow(participants) * 100)
  latex_lines <- c(latex_lines,
    sprintf("\\quad %s & %s (%s) & %s (%s) & %s (%s) & & \\\\",
            exp_labels[i],
            formatC(n_ctrl, big.mark=","), pct_ctrl,
            formatC(n_trt, big.mark=","), pct_trt,
            formatC(n_tot, big.mark=","), pct_tot))
}
# Chi-squared test for experience
chi_exp <- chisq.test(table(participants$prev.experience.assignment.days, participants$TREAT))
v_exp <- cramer_v(chi_exp, nrow(participants), length(exp_levels))
latex_lines[length(latex_lines)] <- sub("& & \\\\$",
  sprintf("& %s & %s \\\\", fmt_p(chi_exp$p.value), fmt(v_exp)),
  latex_lines[length(latex_lines)])

# --- Newcomer vs Experienced ---
latex_lines <- c(latex_lines, "\\addlinespace")
latex_lines <- c(latex_lines, "\\multicolumn{6}{l}{\\textit{Newcomer status}} \\\\")
n_new_ctrl <- sum(ctrl$newcomer)
n_new_trt  <- sum(trt$newcomer)
n_exp_ctrl <- sum(!ctrl$newcomer)
n_exp_trt  <- sum(!trt$newcomer)
chi_new <- chisq.test(table(participants$newcomer, participants$TREAT))
d_new <- cohen_d(as.numeric(ctrl$newcomer), as.numeric(trt$newcomer))
latex_lines <- c(latex_lines,
  sprintf("\\quad Newcomer & %s (%.2f\\%%) & %s (%.2f\\%%) & %s (%.2f\\%%) & & \\\\",
          formatC(n_new_ctrl, big.mark=","), n_new_ctrl/nrow(ctrl)*100,
          formatC(n_new_trt, big.mark=","), n_new_trt/nrow(trt)*100,
          formatC(n_new_ctrl+n_new_trt, big.mark=","), (n_new_ctrl+n_new_trt)/nrow(participants)*100))
latex_lines <- c(latex_lines,
  sprintf("\\quad Experienced & %s (%.2f\\%%) & %s (%.2f\\%%) & %s (%.2f\\%%) & %s & %s \\\\",
          formatC(n_exp_ctrl, big.mark=","), n_exp_ctrl/nrow(ctrl)*100,
          formatC(n_exp_trt, big.mark=","), n_exp_trt/nrow(trt)*100,
          formatC(n_exp_ctrl+n_exp_trt, big.mark=","), (n_exp_ctrl+n_exp_trt)/nrow(participants)*100,
          fmt_p(chi_new$p.value), fmt(d_new)))

# --- Language ---
latex_lines <- c(latex_lines, "\\addlinespace")
latex_lines <- c(latex_lines, "\\multicolumn{6}{l}{\\textit{Language}} \\\\")
lang_names <- c(ar="Arabic", de="German", fa="Persian", pl="Polish")
chi_lang <- chisq.test(table(participants$lang, participants$TREAT))
for (l in names(lang_names)) {
  n_c <- sum(ctrl$lang == l)
  n_t <- sum(trt$lang == l)
  n_a <- n_c + n_t
  pval_str <- ""
  d_str <- ""
  if (l == tail(names(lang_names), 1)) {
    pval_str <- fmt_p(chi_lang$p.value)
    v_lang <- cramer_v(chi_lang, nrow(participants), length(unique(participants$lang)))
    d_str <- fmt(v_lang)
  }
  latex_lines <- c(latex_lines,
    sprintf("\\quad %s & %s (%.2f\\%%) & %s (%.2f\\%%) & %s (%.2f\\%%) & %s & %s \\\\",
            lang_names[l],
            formatC(n_c, big.mark=","), n_c/nrow(ctrl)*100,
            formatC(n_t, big.mark=","), n_t/nrow(trt)*100,
            formatC(n_a, big.mark=","), n_a/nrow(participants)*100,
            pval_str, d_str))
}

# --- Pre-study labor hours per day ---
latex_lines <- c(latex_lines, "\\addlinespace")
latex_lines <- c(latex_lines, "\\multicolumn{6}{l}{\\textit{Pre-study labor hours per day}} \\\\")
w_labor <- wilcox.test(ctrl$labor.hours.per.day.pre.treatment, trt$labor.hours.per.day.pre.treatment)
d_labor <- cohen_d(ctrl$labor.hours.per.day.pre.treatment, trt$labor.hours.per.day.pre.treatment)
latex_lines <- c(latex_lines,
  sprintf("\\quad Mean (SD) & %s & %s & %s & %s & %s \\\\",
          fmt_mean_sd(ctrl$labor.hours.per.day.pre.treatment, 2),
          fmt_mean_sd(trt$labor.hours.per.day.pre.treatment, 2),
          fmt_mean_sd(participants$labor.hours.per.day.pre.treatment, 2),
          fmt_p(w_labor$p.value), fmt(d_labor)))
latex_lines <- c(latex_lines,
  sprintf("\\quad Median & %s & %s & %s & & \\\\",
          fmt(median(ctrl$labor.hours.per.day.pre.treatment, na.rm=TRUE)),
          fmt(median(trt$labor.hours.per.day.pre.treatment, na.rm=TRUE)),
          fmt(median(participants$labor.hours.per.day.pre.treatment, na.rm=TRUE))))

# --- Pre-study labor hours (total) ---
latex_lines <- c(latex_lines, "\\addlinespace")
latex_lines <- c(latex_lines, "\\multicolumn{6}{l}{\\textit{Pre-study labor hours (total)}} \\\\")
w_labor_tot <- wilcox.test(ctrl$labor.hours.pre.treatment, trt$labor.hours.pre.treatment)
d_labor_tot <- cohen_d(ctrl$labor.hours.pre.treatment, trt$labor.hours.pre.treatment)
latex_lines <- c(latex_lines,
  sprintf("\\quad Mean (SD) & %s & %s & %s & %s & %s \\\\",
          fmt_mean_sd(ctrl$labor.hours.pre.treatment, 2),
          fmt_mean_sd(trt$labor.hours.pre.treatment, 2),
          fmt_mean_sd(participants$labor.hours.pre.treatment, 2),
          fmt_p(w_labor_tot$p.value), fmt(d_labor_tot)))
latex_lines <- c(latex_lines,
  sprintf("\\quad Median & %s & %s & %s & & \\\\",
          fmt(median(ctrl$labor.hours.pre.treatment, na.rm=TRUE)),
          fmt(median(trt$labor.hours.pre.treatment, na.rm=TRUE)),
          fmt(median(participants$labor.hours.pre.treatment, na.rm=TRUE))))

# --- Thanks sent (pre-study) ---
latex_lines <- c(latex_lines, "\\addlinespace")
latex_lines <- c(latex_lines, "\\multicolumn{6}{l}{\\textit{Thanks sent (pre-study)}} \\\\")
w_sent <- wilcox.test(ctrl$thanks.sent.pre.treatment, trt$thanks.sent.pre.treatment)
d_sent <- cohen_d(ctrl$thanks.sent.pre.treatment, trt$thanks.sent.pre.treatment)
latex_lines <- c(latex_lines,
  sprintf("\\quad Mean (SD) & %s & %s & %s & %s & %s \\\\",
          fmt_mean_sd(ctrl$thanks.sent.pre.treatment, 2),
          fmt_mean_sd(trt$thanks.sent.pre.treatment, 2),
          fmt_mean_sd(participants$thanks.sent.pre.treatment, 2),
          fmt_p(w_sent$p.value), fmt(d_sent)))
latex_lines <- c(latex_lines,
  sprintf("\\quad Median & %s & %s & %s & & \\\\",
          fmt(median(ctrl$thanks.sent.pre.treatment, na.rm=TRUE)),
          fmt(median(trt$thanks.sent.pre.treatment, na.rm=TRUE)),
          fmt(median(participants$thanks.sent.pre.treatment, na.rm=TRUE))))
latex_lines <- c(latex_lines,
  sprintf("\\quad \\%% with any sent & %s & %s & %s & & \\\\",
          fmt_pct(mean(ctrl$thanks.sent.pre.treatment > 0, na.rm=TRUE)*100),
          fmt_pct(mean(trt$thanks.sent.pre.treatment > 0, na.rm=TRUE)*100),
          fmt_pct(mean(participants$thanks.sent.pre.treatment > 0, na.rm=TRUE)*100)))

# --- Thanks received (pre-study) ---
latex_lines <- c(latex_lines, "\\addlinespace")
latex_lines <- c(latex_lines, "\\multicolumn{6}{l}{\\textit{Thanks received (pre-study)}} \\\\")
w_recv <- wilcox.test(ctrl$number.thanks.received, trt$number.thanks.received)
d_recv <- cohen_d(ctrl$number.thanks.received, trt$number.thanks.received)
latex_lines <- c(latex_lines,
  sprintf("\\quad Mean (SD) & %s & %s & %s & %s & %s \\\\",
          fmt_mean_sd(ctrl$number.thanks.received, 2),
          fmt_mean_sd(trt$number.thanks.received, 2),
          fmt_mean_sd(participants$number.thanks.received, 2),
          fmt_p(w_recv$p.value), fmt(d_recv)))
latex_lines <- c(latex_lines,
  sprintf("\\quad Median & %s & %s & %s & & \\\\",
          fmt(median(ctrl$number.thanks.received, na.rm=TRUE)),
          fmt(median(trt$number.thanks.received, na.rm=TRUE)),
          fmt(median(participants$number.thanks.received, na.rm=TRUE))))
latex_lines <- c(latex_lines,
  sprintf("\\quad \\%% with any received & %s & %s & %s & & \\\\",
          fmt_pct(mean(ctrl$number.thanks.received > 0, na.rm=TRUE)*100),
          fmt_pct(mean(trt$number.thanks.received > 0, na.rm=TRUE)*100),
          fmt_pct(mean(participants$number.thanks.received > 0, na.rm=TRUE)*100)))

latex_lines <- c(latex_lines, "\\bottomrule")
latex_lines <- c(latex_lines, "\\end{tabular}")
latex_lines <- c(latex_lines, "\\end{table*}")

# Write LaTeX file
latex_output_path <- file.path("tables", "balance_table.tex")
writeLines(latex_lines, latex_output_path)
cat(sprintf("\nLaTeX table written to: %s\n", latex_output_path))

# Also print to console
cat("\n=== LATEX OUTPUT ===\n\n")
cat(paste(latex_lines, collapse="\n"))
cat("\n")



# Pre-dropped results

In [ ]:
# Tag each observation as dropped or kept
dropped.blocks <- unique(c(
  subset(all.participants, thanks.not.received.user.deleted)$block,
  subset(all.participants, received.multiple.thanks)$block
))

all.participants$dropped <- all.participants$block %in% dropped.blocks
cat("Dropped:", sum(all.participants$dropped),
    "| Kept:", sum(!all.participants$dropped),
    "| Total:", nrow(all.participants), "\n")


# 1. Characteristics Table: Dropped vs. Kept

## 1a. Drop Reason Breakdown

cat("=== Blocks flagged for removal ===\n")
blocks.deleted  <- subset(all.participants, thanks.not.received.user.deleted)$block
blocks.multiple <- unique(subset(all.participants, received.multiple.thanks)$block)

cat("Blocks with deleted accounts:       ", length(unique(blocks.deleted)), "\n")
cat("Blocks with multiple thanks:        ", length(unique(blocks.multiple)), "\n")
cat("Overlap (in both):                  ",
    length(intersect(unique(blocks.deleted), unique(blocks.multiple))), "\n")
cat("Total unique blocks removed:        ", length(dropped.blocks), "\n")
cat("Participants in removed blocks:     ", sum(all.participants$dropped), "\n")

## 1b. Treatment Assignment Balance

In [ ]:
treat.tab <- all.participants %>%
  mutate(Status = ifelse(dropped, "Dropped", "Kept")) %>%
  group_by(Status, TREAT) %>%
  summarise(N = n(), .groups = "drop") %>%
  pivot_wider(names_from = TREAT, values_from = N, names_prefix = "TREAT_")

kable(treat.tab, caption = "Treatment assignment: Dropped vs. Kept")

## 1c. Language Distribution

In [ ]:
lang.tab <- all.participants %>%
  mutate(Status = ifelse(dropped, "Dropped", "Kept")) %>%
  group_by(Status, lang) %>%
  summarise(N = n(), .groups = "drop") %>%
  pivot_wider(names_from = lang, values_from = N, values_fill = 0)

kable(lang.tab, caption = "Language distribution: Dropped vs. Kept")

## 1d. Newcomer vs. Experienced

In [ ]:
newcomer.tab <- all.participants %>%
  mutate(Status = ifelse(dropped, "Dropped", "Kept"),
         Group  = ifelse(newcomer, "Newcomer", "Experienced")) %>%
  group_by(Status, Group) %>%
  summarise(N = n(), .groups = "drop") %>%
  pivot_wider(names_from = Group, values_from = N, values_fill = 0)

kable(newcomer.tab, caption = "Newcomer status: Dropped vs. Kept")

## 1e. Previous Experience Days

In [ ]:
exp.tab <- all.participants %>%
  mutate(Status = ifelse(dropped, "Dropped", "Kept")) %>%
  group_by(Status) %>%
  summarise(
    N     = n(),
    Mean  = mean(prev.experience.assignment.days, na.rm = TRUE),
    SD    = sd(prev.experience.assignment.days, na.rm = TRUE),
    Median = median(prev.experience.assignment.days, na.rm = TRUE),
    .groups = "drop"
  )

kable(exp.tab, digits = 2, caption = "Previous experience (days): Dropped vs. Kept")

## 1f. Pre-Treatment Outcomes

In [ ]:
outcome.tab <- all.participants %>%
  mutate(Status = ifelse(dropped, "Dropped", "Kept")) %>%
  group_by(Status) %>%
  summarise(
    N = n(),
    # Two-week retention
    retention.rate   = mean(two.week.retention, na.rm = TRUE),
    # Thanks sent
    thanks.sent.mean = mean(thanks.sent, na.rm = TRUE),
    thanks.sent.sd   = sd(thanks.sent, na.rm = TRUE),
    # Labor hours (pre-treatment)
    labor.hrs.pre.mean   = mean(labor.hours.pre.treatment, na.rm = TRUE),
    labor.hrs.pre.sd     = sd(labor.hours.pre.treatment, na.rm = TRUE),
    labor.hrs.pre.median = median(labor.hours.pre.treatment, na.rm = TRUE),
    # Labor hours per day (pre-treatment)
    labor.hrpd.pre.mean   = mean(labor.hours.per.day.pre.treatment, na.rm = TRUE),
    labor.hrpd.pre.sd     = sd(labor.hours.per.day.pre.treatment, na.rm = TRUE),
    labor.hrpd.pre.median = median(labor.hours.per.day.pre.treatment, na.rm = TRUE),
    # Post-treatment outcomes
    labor.hrs.post.mean   = mean(labor.hours.post.treatment, na.rm = TRUE),
    labor.hrpd.post.mean  = mean(labor.hours.per.day.post.treatment, na.rm = TRUE),
    .groups = "drop"
  )

kable(outcome.tab, digits = 4, caption = "Pre- and post-treatment outcomes: Dropped vs. Kept")

## 1g. Formal Balance Tests (Dropped vs. Kept)

In [ ]:
cat("--- t-tests: Dropped vs. Kept ---\n\n")

cat("Pre-treatment labor hours:\n")
print(t.test(labor.hours.pre.treatment ~ dropped, data = all.participants))

cat("\nPre-treatment labor hours per day:\n")
print(t.test(labor.hours.per.day.pre.treatment ~ dropped, data = all.participants))

cat("\nPrevious experience (days):\n")
print(t.test(prev.experience.assignment.days ~ dropped, data = all.participants))

cat("\nTwo-week retention:\n")
print(t.test(as.numeric(two.week.retention) ~ dropped, data = all.participants))

cat("\nThanks sent:\n")
print(t.test(thanks.sent ~ dropped, data = all.participants))

cat("\nChi-squared: Language × Dropped\n")
print(chisq.test(table(all.participants$lang, all.participants$dropped)))

cat("\nChi-squared: Newcomer × Dropped\n")
print(chisq.test(table(all.participants$newcomer, all.participants$dropped)))

cat("\nChi-squared: Treatment × Dropped\n")
print(chisq.test(table(all.participants$TREAT, all.participants$dropped)))

# 2. Pre-Drop Results (Full Sample, N = 15,558)

Re-run the primary ITT models on `all.participants` (before block removal).
Results are compared to the post-drop estimates from the main notebook.

## 2a. Prepare Pre-Drop Data

In [ ]:
# Center pre-treatment covariates on the full sample
all.participants$centered.labor.hours.pre.treatment <-
  scale(all.participants$labor.hours.pre.treatment, center = TRUE, scale = FALSE)
all.participants$centered.labor.hours.per.day.pre.treatment <-
  scale(all.participants$labor.hours.per.day.pre.treatment, center = TRUE, scale = FALSE)

cat("Full sample size:", nrow(all.participants), "\n")
cat("Blocks:", length(unique(all.participants$block)), "\n")

## 2b. ITT: Thanks Sent (Negative Binomial)

In [ ]:
thanks.model.pre <- glm.nb(thanks.sent ~ TREAT + lang + newcomer, data = all.participants)
cat("=== Thanks Sent (glm.nb) — Pre-Drop (N =", nrow(all.participants), ") ===\n")
summary(thanks.model.pre)
confint(thanks.model.pre)

## 2c. ITT: Two-Week Retention (Difference in Means)

In [ ]:
retention.pre <- difference_in_means(two.week.retention ~ TREAT,
                                      blocks = block, ci = TRUE,
                                      data = all.participants)
cat("=== Retention (difference_in_means) — Pre-Drop (N =", nrow(all.participants), ") ===\n")
summary(retention.pre)

## 2d. ITT: Labor Hours Per Day (Tweedie GLMM)

In [ ]:
# Baseline
cat("=== Labor Hours Per Day: Baseline — Pre-Drop ===\n")
lm.pre.baseline <- glmmTMB(
  labor.hours.per.day.post.treatment ~ TREAT + (1 | block),
  data = all.participants,
  family = tweedie(link = "log")
)
summary(lm.pre.baseline)

# With covariate
cat("\n=== Labor Hours Per Day: With Covariate — Pre-Drop ===\n")
lm.pre.cov <- glmmTMB(
  labor.hours.per.day.post.treatment ~ TREAT + centered.labor.hours.per.day.pre.treatment + (1 | block),
  data = all.participants,
  family = tweedie(link = "log")
)
summary(lm.pre.cov)

# With interaction (reported in df.estimates)
cat("\n=== Labor Hours Per Day: With Interaction — Pre-Drop ===\n")
lm.pre.int <- glmmTMB(
  labor.hours.per.day.post.treatment ~ TREAT * centered.labor.hours.per.day.pre.treatment + (1 | block),
  data = all.participants,
  family = tweedie(link = "log")
)
summary(lm.pre.int)

## 2e. ITT: Total Labor Hours (Tweedie GLMM)

In [ ]:
# Baseline
cat("=== Total Labor Hours: Baseline — Pre-Drop ===\n")
lm.total.pre.baseline <- glmmTMB(
  labor.hours.post.treatment ~ TREAT + (1 | block),
  data = all.participants,
  family = tweedie(link = "log")
)
summary(lm.total.pre.baseline)

# With covariate
cat("\n=== Total Labor Hours: With Covariate — Pre-Drop ===\n")
lm.total.pre.cov <- glmmTMB(
  labor.hours.post.treatment ~ TREAT + centered.labor.hours.pre.treatment + (1 | block),
  data = all.participants,
  family = tweedie(link = "log")
)
summary(lm.total.pre.cov)

# With interaction
cat("\n=== Total Labor Hours: With Interaction — Pre-Drop ===\n")
lm.total.pre.int <- glmmTMB(
  labor.hours.post.treatment ~ TREAT * centered.labor.hours.pre.treatment + (1 | block),
  data = all.participants,
  family = tweedie(link = "log")
)
summary(lm.total.pre.int)

## 2f. Aggregate ITT Results (Using df.estimates Function)

This uses the same `df.estimates()` function defined in the notebook to produce
directly comparable output.

In [ ]:
# Re-use the df.estimates function from the notebook (must already be defined)
# Run on the full pre-drop sample
all.results.pre.drop <- df.estimates(all.participants)

# Adjust for multiple comparisons (same Holm method as main analysis)
all.results.pre.drop.pvalues <- all.results.pre.drop[all.results.pre.drop$model != "manipulation check", ]$pvalue
all.results.pre.drop[all.results.pre.drop$model != "manipulation check", ]$pvalue <-
  p.adjust(all.results.pre.drop[all.results.pre.drop$model != "manipulation check", ]$pvalue, method = "holm")

cat("=== Aggregate ITT Results — Pre-Drop (N =", nrow(all.participants), ") ===\n")
print(all.results.pre.drop[, c("model", "estimator", "subgroup", "n.size",
                                 "Estimate", "Std..Error", "pvalue", "CI.Lower", "CI.Upper")])

## 2g. Side-by-Side Comparison

In [ ]:
# Post-drop results (already computed in notebook as `all.results`)
# Re-label for clarity
post <- all.results[, c("model", "Estimate", "Std..Error", "pvalue", "CI.Lower", "CI.Upper", "n.size")]
post$sample <- "Post-Drop (N=15274)"

pre <- all.results.pre.drop[, c("model", "Estimate", "Std..Error", "pvalue", "CI.Lower", "CI.Upper", "n.size")]
pre$sample <- "Pre-Drop (N=15558)"

comparison <- bind_rows(pre, post) %>%
  arrange(model, sample)

kable(comparison, digits = 6, caption = "Side-by-side: Pre-Drop vs. Post-Drop ITT Estimates (Holm-adjusted p-values)")

## Balance Table for Pre-dropped Results

In [ ]:
# ============================================================================
# Supporting Information Tables
# Run AFTER the pre-dropped results cells in the notebook
# Requires: all.participants (with $dropped), all.results, all.results.pre.drop
# ============================================================================

library(tidyverse)
library(knitr)

# --- Smart formatters ---

# Number: 2 dp, but show 4 dp if rounds to 0.00
fmt <- function(x) {
  sapply(x, function(v) {
    if (is.na(v)) return("--")
    if (v == 0) return("0")
    if (abs(v) >= 0.005) return(sprintf("%.2f", v))
    if (abs(v) >= 0.00005) return(sprintf("%.4f", v))
    return(sprintf("%.2e", v))
  })
}

# p-value: smart thresholds
fmt_p <- function(p) {
  sapply(p, function(v) {
    if (is.na(v)) return("--")
    if (v < 0.0001) return("$<$0.0001")
    if (v < 0.001)  return(sprintf("%.4f", v))
    if (v < 0.01)   return(sprintf("%.3f", v))
    return(sprintf("%.2f", v))
  })
}

# Percentage for LaTeX
fmt_pct <- function(x) {
  sapply(x, function(v) {
    if (v == 0) return("0\\%")
    if (abs(v) < 0.005) return(sprintf("%.4f\\%%", v))
    sprintf("%.2f\\%%", v)
  })
}

# Mean (SD) helper
ms <- function(x) paste0(fmt(mean(x, na.rm=TRUE)), " (", fmt(sd(x, na.rm=TRUE)), ")")

# ============================================================================
# TABLE 1: Balance of Dropped vs. Kept Observations
# ============================================================================

dropped <- subset(all.participants, dropped == TRUE)
kept    <- subset(all.participants, dropped == FALSE)

n_d <- nrow(dropped)
n_k <- nrow(kept)

L <- c()
L <- c(L, "\\begin{table}[ht]")
L <- c(L, "\\centering")
L <- c(L, "\\small")
L <- c(L, paste0("\\caption{Characteristics of 284 dropped participants (142 randomization blocks) ",
                  "compared with 15,274 retained participants. Blocks were removed if any member had ",
                  "a deleted account or received multiple thanks. P-values are from Welch t-tests ",
                  "(continuous) or chi-squared tests (categorical).}"))
L <- c(L, "\\label{table:dropped_balance}")
L <- c(L, "\\begin{tabular}{lrrl}")
L <- c(L, "\\toprule")
L <- c(L, paste0("\\textbf{Characteristic} & \\textbf{Dropped} & \\textbf{Kept} & \\textbf{p} \\\\"))
L <- c(L, paste0(" & (N = ", formatC(n_d, big.mark=","), ") & (N = ",
                  formatC(n_k, big.mark=","), ") & \\\\"))
L <- c(L, "\\midrule")

# Treatment assignment
L <- c(L, paste0("Treatment (\\%) & ",
  fmt_pct(mean(dropped$TREAT==1)*100), " & ",
  fmt_pct(mean(kept$TREAT==1)*100), " & ",
  fmt_p(chisq.test(table(all.participants$TREAT, all.participants$dropped))$p.value), " \\\\"))

# Newcomer
L <- c(L, paste0("Newcomer (\\%) & ",
  fmt_pct(mean(dropped$newcomer)*100), " & ",
  fmt_pct(mean(kept$newcomer)*100), " & ",
  fmt_p(chisq.test(table(all.participants$newcomer, all.participants$dropped))$p.value), " \\\\"))

# Language distribution
L <- c(L, "\\addlinespace")
L <- c(L, "\\multicolumn{4}{l}{\\textit{Language (\\%)}} \\\\")
chi_lang <- chisq.test(table(all.participants$lang, all.participants$dropped))
for (lg in c("ar","de","fa","pl")) {
  nm <- c(ar="\\quad Arabic", de="\\quad German", fa="\\quad Persian", pl="\\quad Polish")[lg]
  pstr <- ifelse(lg == "pl", fmt_p(chi_lang$p.value), "")
  L <- c(L, paste0(nm, " & ",
    fmt_pct(mean(dropped$lang==lg)*100), " & ",
    fmt_pct(mean(kept$lang==lg)*100), " & ", pstr, " \\\\"))
}

# Previous experience days
L <- c(L, "\\addlinespace")
t_exp <- t.test(prev.experience.assignment.days ~ dropped, data=all.participants)
L <- c(L, paste0("Prev.\\ experience (days), mean (SD) & ",
  ms(dropped$prev.experience.assignment.days), " & ",
  ms(kept$prev.experience.assignment.days), " & ",
  fmt_p(t_exp$p.value), " \\\\"))

# Pre-treatment labor hours per day
t_lhpd <- t.test(labor.hours.per.day.pre.treatment ~ dropped, data=all.participants)
L <- c(L, paste0("Pre-treat.\\ labor hrs/day, mean (SD) & ",
  ms(dropped$labor.hours.per.day.pre.treatment), " & ",
  ms(kept$labor.hours.per.day.pre.treatment), " & ",
  fmt_p(t_lhpd$p.value), " \\\\"))

# Pre-treatment labor hours total
t_lh <- t.test(labor.hours.pre.treatment ~ dropped, data=all.participants)
L <- c(L, paste0("Pre-treat.\\ labor hrs (total), mean (SD) & ",
  ms(dropped$labor.hours.pre.treatment), " & ",
  ms(kept$labor.hours.pre.treatment), " & ",
  fmt_p(t_lh$p.value), " \\\\"))

# Thanks sent (post-treatment outcome)
t_ts <- t.test(thanks.sent ~ dropped, data=all.participants)
L <- c(L, paste0("Thanks sent, mean (SD) & ",
  ms(dropped$thanks.sent), " & ",
  ms(kept$thanks.sent), " & ",
  fmt_p(t_ts$p.value), " \\\\"))

# Two-week retention
t_ret <- t.test(as.numeric(two.week.retention) ~ dropped, data=all.participants)
L <- c(L, paste0("Two-week retention rate & ",
  fmt(mean(dropped$two.week.retention, na.rm=TRUE)), " & ",
  fmt(mean(kept$two.week.retention, na.rm=TRUE)), " & ",
  fmt_p(t_ret$p.value), " \\\\"))

L <- c(L, "\\bottomrule")
L <- c(L, "\\end{tabular}")
L <- c(L, "\\end{table}")

cat("=== TABLE 1: DROPPED vs KEPT BALANCE ===\n\n")
cat(paste(L, collapse="\n"))
cat("\n\n")

# ============================================================================
# TABLE 2: Pre-Drop vs Post-Drop ITT Model Results
# ============================================================================

pre  <- all.results.pre.drop
post <- all.results

# Row builder
row_model <- function(model_name, display_name, estimator_name) {
  r_pre  <- pre[pre$model == model_name, ]
  r_post <- post[post$model == model_name, ]
  if (nrow(r_pre) == 0 || nrow(r_post) == 0) return(NULL)

  paste0(display_name, " & ", estimator_name, " & ",
    formatC(r_pre$n.size[1], big.mark=","), " & ",
    fmt(r_pre$Estimate[1]), " & ", fmt(r_pre$Std..Error[1]), " & ",
    fmt_p(r_pre$pvalue[1]), " & [", fmt(r_pre$CI.Lower[1]), ", ", fmt(r_pre$CI.Upper[1]), "] & ",
    formatC(r_post$n.size[1], big.mark=","), " & ",
    fmt(r_post$Estimate[1]), " & ", fmt(r_post$Std..Error[1]), " & ",
    fmt_p(r_post$pvalue[1]), " & [", fmt(r_post$CI.Lower[1]), ", ", fmt(r_post$CI.Upper[1]), "] \\\\")
}

M <- c()
M <- c(M, "\\begin{table*}[ht]")
M <- c(M, "\\centering")
M <- c(M, "\\small")
M <- c(M, paste0("\\caption{Intent-to-treat estimates before and after removing 284 participants ",
                  "(142 blocks with deleted accounts or multiple thanks). ",
                  "P-values adjusted for multiple comparisons (Holm method) ",
                  "except for the manipulation check.}"))
M <- c(M, "\\label{table:predrop_results}")
M <- c(M, "\\begin{tabular}{llrrrllrrrll}")
M <- c(M, "\\toprule")
M <- c(M, paste0(" & & \\multicolumn{5}{c}{\\textbf{Pre-Drop (N = 15,558)}}",
                  " & \\multicolumn{5}{c}{\\textbf{Post-Drop (N = 15,274)}} \\\\"))
M <- c(M, "\\cmidrule(lr){3-7} \\cmidrule(lr){8-12}")
M <- c(M, paste0("\\textbf{Outcome} & \\textbf{Estimator} & N & Est. & SE & p",
                  " & 95\\% CI & N & Est. & SE & p & 95\\% CI \\\\"))
M <- c(M, "\\midrule")

M <- c(M, row_model("thanks sent",         "Thanks sent",        "glm.nb"))
M <- c(M, row_model("retention",           "Two-week retention", "Diff.\\ in means"))
M <- c(M, row_model("labor hours per day", "Labor hrs/day",      "glmmTMB (Tweedie)"))
M <- c(M, "\\addlinespace")
M <- c(M, row_model("manipulation check",  "Manipulation check", "Diff.\\ in means"))

M <- c(M, "\\bottomrule")
M <- c(M, "\\end{tabular}")
M <- c(M, "\\end{table*}")

cat("=== TABLE 2: PRE-DROP vs POST-DROP MODEL RESULTS ===\n\n")
cat(paste(M, collapse="\n"))
cat("\n")


# Save Data for LaTeX file

In [ ]:
dfs<-Filter(function(x) is.data.frame(get(x)) , ls())
save(list=dfs, file="paper-data.RData", version = 2) # overleaf uses R 3.4 which requires version 2 of Rdata I believe

In [ ]:
Rscript -e 'load("data/paper-data.RData"); cat("labor.hours.pre.treatment present:", "labor.hours.pre.treatment" %in% names(participants), "\n")'

In [ ]:
colnames(participants)